# Tomato Growth Progression — TFT Training Pipeline
## AgriTwin-GH │ Dindigul Greenhouse │ Data Preparation & Cycle Augmentation

**Purpose:** Prepare a comprehensive training-ready dataset for Temporal Fusion Transformer (TFT) forecasting of tomato growth progression.

**Forecasting targets (built here, trained in Phase 2):**
- `target_stage_24h` / `target_stage_48h` — growth stage label at t+24 h and t+48 h
- `target_stage_index_24h` / `target_stage_index_48h` — numeric stage index
- `target_stage_progress_24h` / `target_stage_progress_48h` — stage progress %
- `target_hours_to_next_stage` — hours remaining until next stage transition at t

**Stage order (fixed):** `seedling → early_vegetative → flowering_initiation → flowering → unripe → ripe`

---

---
## Section A — Setup and Paths

In [2]:
# ── Section A1: Imports ─────────────────────────────────────────────────
import json
import warnings
from datetime import datetime, timedelta
from pathlib import Path

import matplotlib
matplotlib.use('Agg')   # non-interactive backend
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

warnings.filterwarnings('ignore')
np.random.seed(42)
pd.set_option('display.max_columns', 60)
pd.set_option('display.float_format', '{:.4f}'.format)
plt.style.use('seaborn-v0_8-whitegrid')

print('Libraries loaded.')
print(f'   pandas {pd.__version__}  |  numpy {np.__version__}')

Libraries loaded.
   pandas 2.3.3  |  numpy 2.4.2


In [3]:
# ── Section A2: Repository paths ──────────────────────────────────────────
REPO_ROOT     = Path('E:/AgriTwin-GH')
DATA_DIR      = REPO_ROOT / 'data' / 'processed' / 'Growth Progression'

HOURLY_CSV    = DATA_DIR / 'tomato_growth_progression_synthetic_hourly.csv'
CYCLE_SUM_CSV = DATA_DIR / 'tomato_growth_progression_cycle_summary.csv'
STAGE_SUM_CSV = DATA_DIR / 'tomato_growth_progression_stage_summary.csv'
METADATA_JSON = DATA_DIR / 'tomato_growth_progression_metadata.json'

for _f in [HOURLY_CSV, CYCLE_SUM_CSV, STAGE_SUM_CSV, METADATA_JSON]:
    if not _f.exists():
        raise FileNotFoundError(f'Required input missing: {_f}')
print('All required input files present.')

MODELS_ROOT   = REPO_ROOT / 'src' / 'agritwin_gh' / 'models'
RUN_TIMESTAMP = datetime.now().strftime('%Y%m%d_%H%M%S')
RUN_ID        = f'growth_progression_{RUN_TIMESTAMP}'
ARTIFACT_DIR  = MODELS_ROOT / 'artifacts' / RUN_ID
ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)

nb_path = REPO_ROOT / 'notebooks' / 'tomato_growth_progression_tft_pipeline.ipynb'
print(f'Notebook path   : {nb_path}')
print(f'Models root     : {MODELS_ROOT}')
print(f'Artifact dir    : {ARTIFACT_DIR}')
print(f'Run ID          : {RUN_ID}')

All required input files present.
Notebook path   : E:\AgriTwin-GH\notebooks\tomato_growth_progression_tft_pipeline.ipynb
Models root     : E:\AgriTwin-GH\src\agritwin_gh\models
Artifact dir    : E:\AgriTwin-GH\src\agritwin_gh\models\artifacts\growth_progression_20260308_202542
Run ID          : growth_progression_20260308_202542


In [4]:
# ── Section A3: Agronomic constants ──────────────────────────────────────────
STAGES = [
    'seedling', 'early_vegetative', 'flowering_initiation',
    'flowering', 'unripe', 'ripe',
]
STAGE_INDEX_MAP = {s: i for i, s in enumerate(STAGES)}

STAGE_DURATION_RANGES = {
    'seedling':              (12, 18),
    'early_vegetative':      (20, 28),
    'flowering_initiation':  ( 8, 13),
    'flowering':             (14, 20),
    'unripe':                (28, 38),
    'ripe':                  (10, 15),
}

GDD_TBASE = 10.0
GDD_TOPT  = 30.0

ENV_COLS = [
    'indoor_temp', 'indoor_humidity', 'indoor_air_velocity',
    'indoor_CO2', 'solarradiation', 'day_night_flag',
    'vpd', 'dew_point', 'leaf_wetness_proxy',
]

print('Agronomic configuration ready.')
print(f'   Stages   : {STAGES}')
print(f'   ENV_COLS : {ENV_COLS}')

Agronomic configuration ready.
   Stages   : ['seedling', 'early_vegetative', 'flowering_initiation', 'flowering', 'unripe', 'ripe']
   ENV_COLS : ['indoor_temp', 'indoor_humidity', 'indoor_air_velocity', 'indoor_CO2', 'solarradiation', 'day_night_flag', 'vpd', 'dew_point', 'leaf_wetness_proxy']


---
## Section B — Load Input Files

In [5]:
# ── Section B1: Load and standardise datasets ─────────────────────────────────
def load_hourly_dataset(path: Path) -> pd.DataFrame:
    '''Load the hourly CSV, enforce dtypes, and sort within cycles.'''
    df = pd.read_csv(path, parse_dates=['timestamp'])
    df['stage_index'] = df['stage_index'].astype(int)
    df['cycle_id']    = df['cycle_id'].astype(int)
    df = df.sort_values(['cycle_id', 'timestamp']).reset_index(drop=True)
    if 'is_stage_transition' in df.columns:
        df['is_stage_transition'] = (
            df['is_stage_transition']
            .map({'True': True, 'False': False, True: True, False: False})
            .astype(bool)
        )
    return df


hourly_df     = load_hourly_dataset(HOURLY_CSV)
cycle_summary = pd.read_csv(CYCLE_SUM_CSV)
stage_summary = pd.read_csv(STAGE_SUM_CSV)
metadata      = json.loads(METADATA_JSON.read_text())

ts_min   = hourly_df['timestamp'].min()
ts_max   = hourly_df['timestamp'].max()
cycle_ids = sorted(hourly_df['cycle_id'].unique().tolist())

print(f'hourly_df        : {hourly_df.shape}')
print(f'cycle_summary    : {cycle_summary.shape}')
print(f'stage_summary    : {stage_summary.shape}')
print(f'Date range       : {ts_min}  ->  {ts_max}')
print(f'Cycle IDs        : {cycle_ids}')
print('\nStage distribution (hourly rows):')
print(hourly_df['stage_name'].value_counts().reindex(STAGES).to_string())

hourly_df        : (10848, 37)
cycle_summary    : (4, 13)
stage_summary    : (24, 17)
Date range       : 2024-07-01 00:00:00  ->  2025-10-21 23:00:00
Cycle IDs        : [1, 2, 3, 4]

Stage distribution (hourly rows):
stage_name
seedling                1392
early_vegetative        2448
flowering_initiation    1032
flowering               1584
unripe                  3072
ripe                    1320


In [6]:
# ── Section B2: Schema overview and missing-value report ─────────────────────────
print(f'Column count: {hourly_df.shape[1]}')
print('Column schema (name : dtype):')
for col in hourly_df.columns:
    print(f'  {col:<42} {str(hourly_df[col].dtype)}')

miss    = hourly_df.isnull().sum()
missing = miss[miss > 0]
print('\nMissing values:')
print('  None.' if missing.empty else missing.to_string())

print('\nRows per cycle:')
print(
    hourly_df.groupby(['cycle_id', 'cycle_label'])['timestamp']
    .count().rename('hourly_rows').to_string()
)

Column count: 37
Column schema (name : dtype):
  timestamp                                  datetime64[ns]
  cycle_id                                   int64
  cycle_label                                object
  season_window                              object
  real_or_synthetic_flag                     object
  year                                       int64
  month                                      int64
  day_of_year                                int64
  week_of_year                               int64
  hour                                       int64
  season_label                               object
  days_from_cycle_start                      float64
  stage_name                                 object
  stage_index                                int64
  hours_in_current_stage                     float64
  days_in_current_stage                      float64
  stage_duration_hours                       int64
  stage_duration_days                        int64
  stage_progres

---
## Section C — Data Audit

In [7]:
# ── Section C1: Audit helper functions ───────────────────────────────────────────────

def _check_stage_order(df: pd.DataFrame) -> list:
    '''Detect any backward stage-index jump within a cycle.'''
    violations = []
    for cid, grp in df.groupby('cycle_id'):
        seq = grp.sort_values('timestamp')['stage_index'].values
        for j in range(1, len(seq)):
            if seq[j] < seq[j - 1]:
                violations.append({'cycle_id': int(cid),
                                   'jump': f'{seq[j-1]} -> {seq[j]}'})
                break
    return violations


def _check_hourly_gaps(df: pd.DataFrame) -> list:
    '''Detect cycles with inter-row time gaps > 1.5 hours.'''
    issues = []
    for cid, grp in df.groupby('cycle_id'):
        hrs = (
            grp.sort_values('timestamp')['timestamp']
            .diff().dt.total_seconds().div(3600).dropna()
        )
        big = hrs[hrs > 1.5]
        if not big.empty:
            issues.append({'cycle_id': int(cid),
                           'gap_count': int(len(big)),
                           'max_gap_hours': float(big.max())})
    return issues


def run_data_audit(df: pd.DataFrame, artifact_dir: Path) -> dict:
    '''Comprehensive quality audit. Saves supplementary CSVs alongside.'''
    report = {}
    report['total_rows']   = int(len(df))
    report['total_cycles'] = int(df['cycle_id'].nunique())
    report['duplicate_timestamp_cycle_pairs'] = int(
        df.duplicated(subset=['timestamp', 'cycle_id']).sum()
    )
    report['fully_duplicate_rows'] = int(df.duplicated().sum())
    report['invalid_stage_progress_rows'] = int(
        ((df['stage_progress_pct'] < 0) | (df['stage_progress_pct'] > 100.1)).sum()
    )
    report['negative_eta_rows'] = int(
        (df['estimated_hours_to_next_stage'] < 0).sum()
    )
    report['stage_order_violations'] = _check_stage_order(df)
    report['hourly_gap_issues']      = _check_hourly_gaps(df)

    stage_cnt = df['stage_name'].value_counts()
    report['stage_distribution_hours'] = stage_cnt.to_dict()
    report['stage_imbalance_ratio']    = float(stage_cnt.max() / stage_cnt.min())

    cont_env = [
        c for c in ENV_COLS
        if 'flag' not in c and 'proxy' not in c and c in df.columns
    ]
    outliers = {}
    for col in cont_env:
        q1, q3 = df[col].quantile([0.25, 0.75]).values
        iqr = q3 - q1
        outliers[col] = int(
            ((df[col] < q1 - 3 * iqr) | (df[col] > q3 + 3 * iqr)).sum()
        )
    report['outliers_iqr3_fence'] = outliers

    # Save supplementary CSVs
    df[cont_env].describe().round(4).to_csv(
        artifact_dir / 'audit_env_feature_stats.csv'
    )
    corr_cols = [c for c in cont_env + ['stage_index', 'total_cycle_progress_pct']
                 if c in df.columns]
    df[corr_cols].corr().round(4).to_csv(
        artifact_dir / 'audit_correlation_matrix.csv'
    )
    df.groupby('stage_name')[cont_env].mean().round(4).to_csv(
        artifact_dir / 'audit_per_stage_env_means.csv'
    )

    # Flatten to audit_report_summary.csv
    rows = []
    for k, v in report.items():
        if isinstance(v, list):
            rows.append({'check': k, 'key': 'count', 'value': str(len(v))})
            for item in v:
                rows.append({'check': k, 'key': 'item', 'value': str(item)})
        elif isinstance(v, dict):
            for kk, vv in v.items():
                rows.append({'check': k, 'key': str(kk), 'value': str(vv)})
        else:
            rows.append({'check': k, 'key': '', 'value': str(v)})
    pd.DataFrame(rows).to_csv(
        artifact_dir / 'audit_report_summary.csv', index=False
    )
    return report


print('Audit functions defined.')

Audit functions defined.


In [8]:
# ── Section C2: Execute audit and print results ──────────────────────────────────────────
audit_report = run_data_audit(hourly_df, ARTIFACT_DIR)

print('=== DATA AUDIT RESULTS ===')
_scalar_keys = [
    'total_rows', 'total_cycles',
    'duplicate_timestamp_cycle_pairs', 'fully_duplicate_rows',
    'invalid_stage_progress_rows', 'negative_eta_rows',
    'stage_imbalance_ratio',
]
for k in _scalar_keys:
    print(f'  {k:<46}: {audit_report[k]}')

_n_sv = len(audit_report['stage_order_violations'])
_n_gp = len(audit_report['hourly_gap_issues'])
print(f'  stage_order_violations      count : {_n_sv}')
print(f'  hourly_gap_issues           count : {_n_gp}')

print('\n  Stage distribution (hours):')
for s in STAGES:
    _n = audit_report['stage_distribution_hours'].get(s, 0)
    print(f'    {s:<32}: {_n:>6}')

print('\n  Outliers (IQR x 3 fence):')
for col, n in audit_report['outliers_iqr3_fence'].items():
    print(f'    {col:<36}: {n}')

print(f'\nAudit reports saved to: {ARTIFACT_DIR}')

=== DATA AUDIT RESULTS ===
  total_rows                                    : 10848
  total_cycles                                  : 4
  duplicate_timestamp_cycle_pairs               : 0
  fully_duplicate_rows                          : 0
  invalid_stage_progress_rows                   : 0
  negative_eta_rows                             : 0
  stage_imbalance_ratio                         : 2.9767441860465116
  stage_order_violations      count : 0
  hourly_gap_issues           count : 0

  Stage distribution (hours):
    seedling                        :   1392
    early_vegetative                :   2448
    flowering_initiation            :   1032
    flowering                       :   1584
    unripe                          :   3072
    ripe                            :   1320

  Outliers (IQR x 3 fence):
    indoor_temp                         : 0
    indoor_humidity                     : 0
    indoor_air_velocity                 : 0
    indoor_CO2                          : 0
  

---
## Section D — Cycle Expansion Strategy

In [9]:
# ── Section D1: Expansion configuration and augmentation plan ─────────────────────
#
# Strategy overview
# -----------------
# The 4 original cycles cover Kharif 2024, Rabi 2024, Summer 2025, Kharif 2025.
# Each supplies a real hourly environmental backbone (actual sensor data).
# New cycles are generated via four methods:
#   season_warp        - restamp prototype onto a new planting year + noise
#   duration_shift     - randomise stage durations within agronomic ranges + noise
#   parameter_jitter   - amplified Gaussian noise across all env features
#   cross_season_blend - linear blend of two prototype season env signals
# All methods are biologically constrained: fixed stage order is enforced,
# durations stay within agronomic ranges, and env values are physically clamped.
# -------------------------------------------------------------------------

EXPANSION_CONFIG = {
    'target_total_cycles':     16,
    'original_cycle_count':     4,
    'cycles_to_generate':      12,
    'rng_seed':               2026,
    'perturb_sigma': {
        'indoor_temp':          0.40,
        'indoor_humidity':      1.50,
        'indoor_air_velocity':  0.05,
        'indoor_CO2':          10.00,
        'solarradiation':       5.00,
        'vpd':                  0.05,
        'dew_point':            0.40,
        'day_night_flag':       0.00,
        'leaf_wetness_proxy':   0.00,
    },
}

ORIGINAL_META = {
    1: {'type': 'kharif', 'start': '2024-07-01'},
    2: {'type': 'rabi',   'start': '2024-11-10'},
    3: {'type': 'summer', 'start': '2025-03-05'},
    4: {'type': 'kharif', 'start': '2025-07-01'},
}

# (cycle_id, season_type, start_date, backbone_cycle_id, aug_method)
NEW_CYCLE_PLAN = [
    # Kharif-type (4 more)
    ( 5, 'kharif', '2026-07-01', 1, 'season_warp'),
    ( 6, 'kharif', '2027-07-05', 4, 'duration_shift'),
    ( 7, 'kharif', '2026-07-10', 4, 'parameter_jitter'),
    ( 8, 'kharif', '2028-07-01', 1, 'duration_shift'),
    # Rabi-type (4 more)
    ( 9, 'rabi',   '2026-11-05', 2, 'season_warp'),
    (10, 'rabi',   '2027-11-08', 2, 'duration_shift'),
    (11, 'rabi',   '2028-11-01', 2, 'parameter_jitter'),
    (12, 'rabi',   '2026-11-15', 2, 'duration_shift'),
    # Summer-type (4 more)
    (13, 'summer', '2026-03-10', 3, 'season_warp'),
    (14, 'summer', '2027-03-05', 3, 'duration_shift'),
    (15, 'summer', '2027-03-15', 3, 'parameter_jitter'),
    (16, 'summer', '2028-03-01', 3, 'cross_season_blend'),
]

METHOD_DESCRIPTIONS = {
    'season_warp'       : 'Restamp prototype env to new year + controlled noise',
    'duration_shift'    : 'Random stage durations within agronomic ranges + noise',
    'parameter_jitter'  : '2x-amplified Gaussian noise across all env features',
    'cross_season_blend': 'Linear blend of two prototype season envs (alpha=0.4) + noise',
}

print(f'{"ID":>3}  {"Season":7}  {"Start":12}  {"Backbone":>8}  Method')
print('-' * 60)
for _r in NEW_CYCLE_PLAN:
    _cid, _ct, _cs, _bb, _mth = _r
    print(f'{_cid:>3}  {_ct:7}  {_cs:12}  {_bb:>8}  {_mth}')

print(f'\n  Original cycles   : {EXPANSION_CONFIG["original_cycle_count"]}')
print(f'  Cycles to create  : {EXPANSION_CONFIG["cycles_to_generate"]}')
print(f'  Total target      : {EXPANSION_CONFIG["target_total_cycles"]}')
print(f'  Methods           : {sorted({r[4] for r in NEW_CYCLE_PLAN})}')

 ID  Season   Start         Backbone  Method
------------------------------------------------------------
  5  kharif   2026-07-01           1  season_warp
  6  kharif   2027-07-05           4  duration_shift
  7  kharif   2026-07-10           4  parameter_jitter
  8  kharif   2028-07-01           1  duration_shift
  9  rabi     2026-11-05           2  season_warp
 10  rabi     2027-11-08           2  duration_shift
 11  rabi     2028-11-01           2  parameter_jitter
 12  rabi     2026-11-15           2  duration_shift
 13  summer   2026-03-10           3  season_warp
 14  summer   2027-03-05           3  duration_shift
 15  summer   2027-03-15           3  parameter_jitter
 16  summer   2028-03-01           3  cross_season_blend

  Original cycles   : 4
  Cycles to create  : 12
  Total target      : 16
  Methods           : ['cross_season_blend', 'duration_shift', 'parameter_jitter', 'season_warp']


---
## Section E — Synthetic Cycle Augmentation

In [10]:
# ── Section E1: Augmentation helper functions ─────────────────────────────────────────

def _india_season_label(month: int) -> str:
    if month in [3, 4, 5]:    return 'summer'
    if month in [6, 7, 8, 9]: return 'southwest_monsoon'
    if month in [10, 11]:     return 'northeast_monsoon'
    return 'dry_winter'


def _season_window_label(cycle_type: str, ts: pd.Timestamp) -> str:
    spans = {'kharif': 'Jul-Oct', 'rabi': 'Nov-Feb', 'summer': 'Mar-Jun'}
    return f'{cycle_type.capitalize()} ({spans.get(cycle_type, "?")} {ts.year})'


def get_proto_stage_env(df: pd.DataFrame, cycle_id: int, stage_name: str) -> dict:
    '''Extract env arrays for one stage in one prototype cycle.'''
    mask = (df['cycle_id'] == cycle_id) & (df['stage_name'] == stage_name)
    sub  = df.loc[mask].sort_values('timestamp')
    if sub.empty:
        return {col: np.array([]) for col in ENV_COLS if col in df.columns}
    return {col: sub[col].values for col in ENV_COLS if col in sub.columns}


def warp_env_stage(proto_env: dict, target_hours: int,
                   rng: np.random.Generator, sigma: dict) -> dict:
    '''
    Linearly interpolate each env array from its original length to
    target_hours, then add Gaussian noise. Physical bounds are enforced.
    '''
    result = {}
    for col, vals in proto_env.items():
        n = len(vals)
        if n == 0:
            result[col] = np.zeros(target_hours)
            continue
        if n != target_hours:
            orig_x = np.linspace(0, 1, n)
            new_x  = np.linspace(0, 1, target_hours)
            warped = np.interp(new_x, orig_x, vals.astype(float))
        else:
            warped = vals.astype(float).copy()
        s = sigma.get(col, 0.0)
        if s > 0.0:
            warped = warped + rng.normal(0.0, s, target_hours)
        if   col == 'indoor_temp':          warped = np.clip(warped, 10.0, 45.0)
        elif col == 'indoor_humidity':      warped = np.clip(warped, 20.0, 100.0)
        elif col == 'solarradiation':       warped = np.clip(warped,  0.0, 1000.0)
        elif col == 'vpd':                  warped = np.clip(warped,  0.0,  5.0)
        elif col == 'indoor_CO2':           warped = np.clip(warped, 300.0, 1500.0)
        elif col == 'indoor_air_velocity':  warped = np.clip(warped,  0.0,  10.0)
        elif col == 'day_night_flag':       warped = np.clip(np.round(warped), 0, 1)
        elif col == 'leaf_wetness_proxy':   warped = np.clip(warped, 0.0, 1.0)
        result[col] = warped
    return result


def blend_env_stages(env_a: dict, env_b: dict, target_hours: int,
                     rng: np.random.Generator, alpha: float,
                     sigma: dict) -> dict:
    '''Blend: (1-alpha)*A + alpha*B + noise.'''
    wa = warp_env_stage(env_a, target_hours, rng, sigma)
    wb = warp_env_stage(env_b, target_hours, rng, sigma)
    return {
        col: (1.0 - alpha) * wa[col] + alpha * wb.get(col, wa[col])
        for col in wa
    }


def sample_stage_durations(rng: np.random.Generator) -> dict:
    '''Draw stage durations (days) uniformly within agronomic ranges.'''
    return {
        s: int(rng.integers(lo, hi + 1))
        for s, (lo, hi) in STAGE_DURATION_RANGES.items()
    }


def build_generated_cycle(
    cycle_id, cycle_type, cycle_start_str,
    backbone_id, method,
    hourly_df, stage_durations,
    rng, sigma, run_id,
    blend_id=None,
) -> pd.DataFrame:
    '''
    Build a fully-featured hourly DataFrame for one generated cycle.
    For each stage:
      1. Extract the stage from the backbone prototype cycle.
      2. Resample to the new duration via linear interpolation.
      3. Add method-specific Gaussian perturbation.
    All progression columns, rolling means, and GDD are recomputed.
    '''
    cycle_start = pd.Timestamp(cycle_start_str)
    total_hours = sum(d * 24 for d in stage_durations.values())
    eff_sigma   = {
        k: v * (2.0 if method == 'parameter_jitter' else 1.0)
        for k, v in sigma.items()
    }
    stage_blocks  = []
    hours_elapsed = 0
    cum_gdd       = 0.0

    for stage_idx, stage_name in enumerate(STAGES):
        stage_days  = stage_durations[stage_name]
        stage_hours = stage_days * 24

        proto = get_proto_stage_env(hourly_df, backbone_id, stage_name)
        if len(next(iter(proto.values()), [])) == 0:
            for fb in [int(c) for c in hourly_df['cycle_id'].unique() if c != backbone_id]:
                proto = get_proto_stage_env(hourly_df, fb, stage_name)
                if len(next(iter(proto.values()), [])) > 0:
                    break

        if method == 'cross_season_blend' and blend_id is not None:
            proto_b = get_proto_stage_env(hourly_df, blend_id, stage_name)
            if len(next(iter(proto_b.values()), [])) > 0:
                env = blend_env_stages(proto, proto_b, stage_hours, rng,
                                       alpha=0.4, sigma=eff_sigma)
            else:
                env = warp_env_stage(proto, stage_hours, rng, eff_sigma)
        else:
            env = warp_env_stage(proto, stage_hours, rng, eff_sigma)

        ts_range = pd.date_range(
            start=cycle_start + timedelta(hours=hours_elapsed),
            periods=stage_hours, freq='h',
        )
        ts_dt = pd.DatetimeIndex(ts_range)
        h_arr = np.arange(stage_hours, dtype=float)

        temp_arr = env['indoor_temp']
        hum_arr  = env['indoor_humidity']
        sol_arr  = np.maximum(0.0, env['solarradiation'])

        gdd_h       = np.maximum(0.0, np.minimum(temp_arr, GDD_TOPT) - GDD_TBASE) / 24.0
        cum_gdd_arr = cum_gdd + np.cumsum(gdd_h)
        cum_gdd     = float(cum_gdd_arr[-1])

        es_arr    = 0.6108 * np.exp(17.27 * temp_arr / (temp_arr + 237.3))
        vpd_proxy = np.maximum(0.0, es_arr * (1.0 - hum_arr / 100.0))
        light_flag = (sol_arr > 5.0).astype(int)
        iso_week   = ts_dt.isocalendar().week.astype(int).to_numpy()

        block = pd.DataFrame({
            'timestamp':                ts_range,
            'cycle_id':                 cycle_id,
            'cycle_label':              f'{cycle_type}_{cycle_start.year}',
            'season_window':            _season_window_label(cycle_type, cycle_start),
            'real_or_synthetic_flag':   'synthetic',
            'year':                     ts_dt.year.to_numpy(),
            'month':                    ts_dt.month.to_numpy(),
            'day_of_year':              ts_dt.day_of_year.to_numpy(),
            'week_of_year':             iso_week,
            'hour':                     ts_dt.hour.to_numpy(),
            'season_label':             [_india_season_label(m) for m in ts_dt.month],
            'days_from_cycle_start':    np.round((hours_elapsed + h_arr) / 24.0, 4),
            'stage_name':               stage_name,
            'stage_index':              stage_idx,
            'hours_in_current_stage':   h_arr,
            'days_in_current_stage':    np.round(h_arr / 24.0, 4),
            'stage_duration_hours':     stage_hours,
            'stage_duration_days':      stage_days,
            'stage_progress_pct':       np.round(h_arr / stage_hours * 100.0, 3),
            'total_cycle_progress_pct': np.round(
                (hours_elapsed + h_arr) / total_hours * 100.0, 3
            ),
            'estimated_days_to_next_stage':  np.round((stage_hours - h_arr) / 24.0, 4),
            'estimated_hours_to_next_stage': (stage_hours - h_arr).round(0),
            'is_stage_transition':      [
                (i == 0 and stage_idx > 0) for i in range(stage_hours)
            ],
            'indoor_temp':              np.round(temp_arr, 4),
            'indoor_humidity':          np.round(hum_arr, 4),
            'indoor_air_velocity':      np.round(env['indoor_air_velocity'], 4),
            'indoor_CO2':               np.round(env['indoor_CO2'], 4),
            'solarradiation':           np.round(sol_arr, 4),
            'day_night_flag':           env['day_night_flag'].astype(int),
            'vpd':                      np.round(env['vpd'], 4),
            'dew_point':                np.round(env['dew_point'], 4),
            'leaf_wetness_proxy':       np.round(
                env.get('leaf_wetness_proxy', np.zeros(stage_hours)), 4
            ),
            'temperature_rolling_mean_24h': np.nan,
            'humidity_rolling_mean_24h':    np.nan,
            'vpd_proxy':                np.round(vpd_proxy, 4),
            'light_period_flag':        light_flag,
            'cumulative_gdd_like_index': np.round(cum_gdd_arr, 4),
            'cycle_origin_type':        'generated',
            'prototype_cycle_id':       backbone_id,
            'augmentation_method':      method,
            'run_id':                   run_id,
        })
        stage_blocks.append(block)
        hours_elapsed += stage_hours

    cycle_df = pd.concat(stage_blocks, ignore_index=True)
    cycle_df['temperature_rolling_mean_24h'] = (
        cycle_df['indoor_temp'].rolling(24, min_periods=1).mean().round(4)
    )
    cycle_df['humidity_rolling_mean_24h'] = (
        cycle_df['indoor_humidity'].rolling(24, min_periods=1).mean().round(4)
    )
    return cycle_df


print('Augmentation helper functions defined.')

Augmentation helper functions defined.


In [11]:
# ── Section E2: Execute cycle generation ───────────────────────────────────────────────
rng   = np.random.default_rng(EXPANSION_CONFIG['rng_seed'])
sigma = EXPANSION_CONFIG['perturb_sigma']

generated_dfs  = []
gen_cycle_meta = []

for plan_row in NEW_CYCLE_PLAN:
    cid, ctype, cstart, backbone_id, method = plan_row
    stage_dur = sample_stage_durations(rng)

    blend_id = None
    if method == 'cross_season_blend':
        blend_id = 1 if backbone_id == 3 else 3

    total_d = sum(stage_dur.values())
    print(
        f'  Cycle {cid:>2} | {ctype:7} | {method:<20} '
        f'| backbone={backbone_id} | {total_d} days ... ',
        end='',
    )

    cyc_df = build_generated_cycle(
        cycle_id=cid, cycle_type=ctype, cycle_start_str=cstart,
        backbone_id=backbone_id, method=method,
        hourly_df=hourly_df, stage_durations=stage_dur,
        rng=rng, sigma=sigma, run_id=RUN_ID,
        blend_id=blend_id,
    )
    generated_dfs.append(cyc_df)
    print(f'{len(cyc_df):5d} rows')

    meta_row = {
        'cycle_id':             cid,
        'season_type':          ctype,
        'cycle_start':          cstart,
        'cycle_end':            str(cyc_df['timestamp'].max().date()),
        'total_days':           total_d,
        'backbone_cycle_id':    backbone_id,
        'augmentation_method':  method,
        'generated_cycle_flag': True,
        'hourly_rows':          len(cyc_df),
    }
    for s in STAGES:
        meta_row[f'days_{s}'] = stage_dur[s]
    for s in STAGES:
        _sub = cyc_df.loc[cyc_df['stage_name'] == s, 'indoor_temp']
        meta_row[f'mean_temp_{s}'] = round(float(_sub.mean()), 3)
    gen_cycle_meta.append(meta_row)

print(f'\nGenerated {len(generated_dfs)} new cycles.')

  Cycle  5 | kharif  | season_warp          | backbone=1 | 108 days ...  2592 rows
  Cycle  6 | kharif  | duration_shift       | backbone=4 | 111 days ...  2664 rows
  Cycle  7 | kharif  | parameter_jitter     | backbone=4 | 115 days ...  2760 rows
  Cycle  8 | kharif  | duration_shift       | backbone=1 | 120 days ...  2880 rows
  Cycle  9 | rabi    | season_warp          | backbone=2 | 124 days ...  2976 rows
  Cycle 10 | rabi    | duration_shift       | backbone=2 | 117 days ...  2808 rows
  Cycle 11 | rabi    | parameter_jitter     | backbone=2 | 110 days ...  2640 rows
  Cycle 12 | rabi    | duration_shift       | backbone=2 | 115 days ...  2760 rows
  Cycle 13 | summer  | season_warp          | backbone=3 | 101 days ...  2424 rows
  Cycle 14 | summer  | duration_shift       | backbone=3 | 121 days ...  2904 rows
  Cycle 15 | summer  | parameter_jitter     | backbone=3 | 110 days ...  2640 rows
  Cycle 16 | summer  | cross_season_blend   | backbone=3 | 109 days ...  2616 rows

Gen

In [12]:
# ── Section E3: Combine original + generated; validate ─────────────────────────────
hourly_df = hourly_df.copy()
hourly_df['cycle_origin_type']   = 'original'
hourly_df['prototype_cycle_id']  = hourly_df['cycle_id']
hourly_df['augmentation_method'] = 'original'
hourly_df['run_id']              = RUN_ID

combined_df = pd.concat([hourly_df] + generated_dfs, ignore_index=True)
combined_df = combined_df.sort_values(['cycle_id', 'timestamp']).reset_index(drop=True)

orig_rows = len(hourly_df)
gen_rows  = sum(len(d) for d in generated_dfs)
print(f'Combined dataset shape : {combined_df.shape}')
print(f'  Original rows        : {orig_rows:,}')
print(f'  Generated rows       : {gen_rows:,}')
print(f'  Total cycles         : {combined_df["cycle_id"].nunique()}')


def _validate_expanded(df: pd.DataFrame) -> bool:
    ok = True
    for cid, grp in df.groupby('cycle_id'):
        seq = grp.sort_values('timestamp')['stage_index'].values
        if any(seq[i] < seq[i - 1] for i in range(1, len(seq))):
            print(f'  FAIL: backward stage jump in cycle {cid}')
            ok = False
        present = set(grp['stage_name'].unique())
        if present != set(STAGES):
            print(f'  FAIL: missing stages in cycle {cid}: {set(STAGES) - present}')
            ok = False
    if (df['stage_progress_pct'] < 0).any():
        print('  FAIL: negative stage_progress_pct found')
        ok = False
    if ok:
        print('Validation passed: stage order and completeness OK for all cycles.')
    return ok


_validate_expanded(combined_df)

print('\nRows per cycle (showing origin type):')
print(
    combined_df.groupby(['cycle_id', 'cycle_origin_type'])['timestamp']
    .count().rename('rows').to_string()
)

Combined dataset shape : (43512, 41)
  Original rows        : 10,848
  Generated rows       : 32,664
  Total cycles         : 16
Validation passed: stage order and completeness OK for all cycles.

Rows per cycle (showing origin type):
cycle_id  cycle_origin_type
1         original             2712
2         original             2616
3         original             2808
4         original             2712
5         generated            2592
6         generated            2664
7         generated            2760
8         generated            2880
9         generated            2976
10        generated            2808
11        generated            2640
12        generated            2760
13        generated            2424
14        generated            2904
15        generated            2640
16        generated            2616


---
## Section F — TFT Dataset Construction

In [13]:
# ── Section F1: Add TFT forecast target columns ───────────────────────────────────
#
# For each row t in a cycle, shift within-cycle (no cross-cycle leakage):
#   t+24h and t+48h values of: stage_name, stage_index, stage_progress_pct
# The last 24 (or 48) rows of each cycle will have NaN targets.
# These NaN rows are excluded during model training (masked in Phase 2).
# -------------------------------------------------------------------------

def add_forecast_targets(df: pd.DataFrame) -> pd.DataFrame:
    '''
    Append TFT target columns via within-cycle negative shifts.
    No information leaks across cycle boundaries.
    '''
    df = df.copy().sort_values(['cycle_id', 'timestamp']).reset_index(drop=True)
    for n, suffix in [(24, '24h'), (48, '48h')]:
        grp = df.groupby('cycle_id', sort=False)
        df[f'target_stage_{suffix}']          = grp['stage_name'].shift(-n)
        df[f'target_stage_index_{suffix}']    = grp['stage_index'].shift(-n)
        df[f'target_stage_progress_{suffix}'] = grp['stage_progress_pct'].shift(-n)
    df['target_hours_to_next_stage'] = df['estimated_hours_to_next_stage']
    return df


print('Building TFT forecast target columns ...')
tft_df = add_forecast_targets(combined_df)

TARGET_COLS = [
    'target_stage_24h', 'target_stage_48h',
    'target_stage_index_24h', 'target_stage_index_48h',
    'target_stage_progress_24h', 'target_stage_progress_48h',
    'target_hours_to_next_stage',
]

print(f'Final TFT dataset shape : {tft_df.shape}')
print('\nTarget column coverage (non-null rows):')
for col in TARGET_COLS:
    nn  = tft_df[col].notna().sum()
    pct = nn / len(tft_df) * 100
    print(f'  {col:<42}: {nn:7,}  ({pct:.1f} %)')

Building TFT forecast target columns ...
Final TFT dataset shape : (43512, 48)

Target column coverage (non-null rows):
  target_stage_24h                          :  43,128  (99.1 %)
  target_stage_48h                          :  42,744  (98.2 %)
  target_stage_index_24h                    :  43,128  (99.1 %)
  target_stage_index_48h                    :  42,744  (98.2 %)
  target_stage_progress_24h                 :  43,128  (99.1 %)
  target_stage_progress_48h                 :  42,744  (98.2 %)
  target_hours_to_next_stage                :  43,512  (100.0 %)


In [14]:
# ── Section F2: Feature catalog and target definition ─────────────────────────────

FEATURE_CATALOG = {
    'description':             'TFT feature groups for tomato growth progression forecasting',
    'project':                 'AgriTwin-GH',
    'run_id':                  RUN_ID,
    'entity_id':               'cycle_id',
    'time_index':              'timestamp',
    'group_ids':               ['cycle_id'],
    'max_encoder_length':      168,
    'max_prediction_length':    48,
    'static_categoricals':     ['cycle_label', 'season_window', 'cycle_origin_type'],
    'static_reals':            ['prototype_cycle_id'],
    'time_varying_known_categoricals': ['season_label'],
    'time_varying_known_reals': [
        'hour', 'day_of_year', 'week_of_year', 'month', 'year',
        'day_night_flag', 'light_period_flag',
        'stage_duration_hours', 'stage_duration_days',
        'estimated_hours_to_next_stage', 'estimated_days_to_next_stage',
    ],
    'time_varying_unknown_reals': [
        'indoor_temp', 'indoor_humidity', 'indoor_air_velocity',
        'indoor_CO2', 'solarradiation', 'vpd', 'dew_point',
        'leaf_wetness_proxy', 'temperature_rolling_mean_24h',
        'humidity_rolling_mean_24h', 'vpd_proxy',
        'cumulative_gdd_like_index', 'stage_index',
        'stage_progress_pct', 'total_cycle_progress_pct',
    ],
    'targets': {
        'primary':   [
            'target_stage_index_24h', 'target_stage_index_48h',
            'target_stage_progress_24h', 'target_stage_progress_48h',
            'target_hours_to_next_stage',
        ],
        'auxiliary': ['target_stage_24h', 'target_stage_48h'],
    },
    'notes': {
        'max_encoder_length'    : '168 h (7 days) captures multi-stage context + rolling means',
        'max_prediction_length' : '48 h matches the longest forecast horizon',
        'known_at_forecast_time': 'Calendar + scheduled agronomic features available at inference',
        'unknown_observed'      : 'Real-time sensor readings; not available at future steps',
    },
}

TARGET_DEFINITION = {
    'forecasting_goal': 'Predict tomato growth stage and stage progress at t+24h and t+48h',
    'time_resolution':  'Hourly',
    'stage_index_map':  STAGE_INDEX_MAP,
    'primary_targets':  [
        {'name': 'target_stage_index_24h',
         'type': 'ordinal regression / 6-class classification (0-5)',
         'source_col': 'stage_index', 'shift_hours': 24,
         'range': [0, 5],
         'description': 'Growth stage numeric index 24 h ahead'},
        {'name': 'target_stage_index_48h',
         'type': 'ordinal regression / 6-class classification (0-5)',
         'source_col': 'stage_index', 'shift_hours': 48,
         'range': [0, 5],
         'description': 'Growth stage numeric index 48 h ahead'},
        {'name': 'target_stage_progress_24h',
         'type': 'regression (0-100)', 'source_col': 'stage_progress_pct',
         'shift_hours': 24, 'range': [0.0, 100.0],
         'description': 'Stage progress percentage 24 h ahead'},
        {'name': 'target_stage_progress_48h',
         'type': 'regression (0-100)', 'source_col': 'stage_progress_pct',
         'shift_hours': 48, 'range': [0.0, 100.0],
         'description': 'Stage progress percentage 48 h ahead'},
        {'name': 'target_hours_to_next_stage',
         'type': 'regression (>= 0)', 'source_col': 'estimated_hours_to_next_stage',
         'shift_hours': 0,
         'description': 'Hours until next stage transition at current time t'},
    ],
    'auxiliary_targets': [
        {'name': 'target_stage_24h', 'type': 'multi-class string label',
         'source_col': 'stage_name', 'shift_hours': 24},
        {'name': 'target_stage_48h', 'type': 'multi-class string label',
         'source_col': 'stage_name', 'shift_hours': 48},
    ],
}

print('Feature catalog and target definition created.')
_n_kn  = len(FEATURE_CATALOG['time_varying_known_reals'])
_n_unk = len(FEATURE_CATALOG['time_varying_unknown_reals'])
print(f'  Static categoricals         : {FEATURE_CATALOG["static_categoricals"]}')
print(f'  Time-varying known (real)   : {_n_kn} features')
print(f'  Time-varying unknown (real) : {_n_unk} features')
print(f'  Primary targets             : {FEATURE_CATALOG["targets"]["primary"]}')

Feature catalog and target definition created.
  Static categoricals         : ['cycle_label', 'season_window', 'cycle_origin_type']
  Time-varying known (real)   : 11 features
  Time-varying unknown (real) : 15 features
  Primary targets             : ['target_stage_index_24h', 'target_stage_index_48h', 'target_stage_progress_24h', 'target_stage_progress_48h', 'target_hours_to_next_stage']


---
## Section G — Save Intermediate Artifacts

In [15]:
# ── Section G1: Build expansion summary and dataset profile ───────────────────────
orig_meta_rows = []
for cyc in metadata.get('cycles', []):
    cid    = cyc['cycle_id']
    o_type = ORIGINAL_META[cid]['type']
    row = {
        'cycle_id':             cid,
        'season_type':          o_type,
        'cycle_start':          cyc['cycle_start'],
        'cycle_end':            cyc['cycle_end'],
        'total_days':           cyc['total_days'],
        'backbone_cycle_id':    cid,
        'augmentation_method':  'original',
        'generated_cycle_flag': False,
        'hourly_rows':          int((hourly_df['cycle_id'] == cid).sum()),
    }
    for s in STAGES:
        row[f'days_{s}'] = cyc['stage_durations_days'][s]
    for s in STAGES:
        _sub = stage_summary.loc[
            (stage_summary['cycle_id'] == cid) & (stage_summary['stage_name'] == s),
            'mean_indoor_temp'
        ]
        row[f'mean_temp_{s}'] = round(float(_sub.mean()), 3) if not _sub.empty else float('nan')
    orig_meta_rows.append(row)

cycle_expansion_df = pd.DataFrame(orig_meta_rows + gen_cycle_meta)

trainable_rows = int(tft_df['target_stage_index_48h'].notna().sum())
dataset_profile = {
    'run_id':                    RUN_ID,
    'created_on':                datetime.now().isoformat(),
    'total_rows':                int(len(tft_df)),
    'trainable_rows_48h_target': trainable_rows,
    'total_cycles':              int(tft_df['cycle_id'].nunique()),
    'original_cycles':           int(hourly_df['cycle_id'].nunique()),
    'generated_cycles':          len(generated_dfs),
    'date_range': {
        'start': str(tft_df['timestamp'].min()),
        'end':   str(tft_df['timestamp'].max()),
    },
    'column_count': int(len(tft_df.columns)),
    'columns':      list(tft_df.columns),
    'target_cols':  TARGET_COLS,
    'stage_distribution': {
        s: int((tft_df['stage_name'] == s).sum()) for s in STAGES
    },
    'cycle_origin_distribution': (
        tft_df.groupby('cycle_origin_type')['cycle_id'].nunique().to_dict()
    ),
    'rows_with_24h_target': int(tft_df['target_stage_index_24h'].notna().sum()),
    'rows_with_48h_target': trainable_rows,
}

# Compact preview: 10 evenly-spaced rows from each cycle
preview_frames = []
for cid, grp in tft_df.groupby('cycle_id'):
    idx = np.linspace(0, len(grp) - 1, min(10, len(grp)), dtype=int)
    preview_frames.append(grp.iloc[idx])
preview_df = pd.concat(preview_frames, ignore_index=True)

print(f'Cycle expansion summary : {cycle_expansion_df.shape}')
print(f'Dataset profile keys    : {list(dataset_profile.keys())}')

Cycle expansion summary : (16, 21)
Dataset profile keys    : ['run_id', 'created_on', 'total_rows', 'trainable_rows_48h_target', 'total_cycles', 'original_cycles', 'generated_cycles', 'date_range', 'column_count', 'columns', 'target_cols', 'stage_distribution', 'cycle_origin_distribution', 'rows_with_24h_target', 'rows_with_48h_target']


In [16]:
# ── Section G2: Save all datasets and metadata files ───────────────────────────────
saved_files = []

def _save_csv(df, name):
    p = ARTIFACT_DIR / name
    df.to_csv(p, index=False)
    saved_files.append(name)
    print(f'  saved  {name:<55} ({len(df):,} rows)')

def _save_parquet(df, name):
    p = ARTIFACT_DIR / name
    df.to_parquet(p, index=False)
    saved_files.append(name)
    print(f'  saved  {name:<55} (parquet)')

def _save_json(obj, name):
    p = ARTIFACT_DIR / name
    p.write_text(json.dumps(obj, indent=2, default=str))
    saved_files.append(name)
    print(f'  saved  {name}')


print('Saving intermediate artifacts ...\n')
_save_csv(tft_df,             'expanded_hourly_growth_dataset.csv')
_save_parquet(tft_df,         'expanded_hourly_growth_dataset.parquet')
_save_csv(cycle_expansion_df, 'cycle_expansion_summary.csv')
_save_csv(preview_df,         'preview_sample.csv')
_save_json(FEATURE_CATALOG,   'feature_catalog.json')
_save_json(TARGET_DEFINITION, 'target_definition.json')
_save_json(dataset_profile,   'dataset_profile.json')

for _fname in [
    'audit_report_summary.csv', 'audit_env_feature_stats.csv',
    'audit_correlation_matrix.csv', 'audit_per_stage_env_means.csv',
]:
    if (ARTIFACT_DIR / _fname).exists():
        saved_files.append(_fname)
        print(f'  registered (from Section C) {_fname}')

print(f'\n{len(saved_files)} artifacts saved to:\n  {ARTIFACT_DIR}')

Saving intermediate artifacts ...

  saved  expanded_hourly_growth_dataset.csv                      (43,512 rows)
  saved  expanded_hourly_growth_dataset.parquet                  (parquet)
  saved  cycle_expansion_summary.csv                             (16 rows)
  saved  preview_sample.csv                                      (160 rows)
  saved  feature_catalog.json
  saved  target_definition.json
  saved  dataset_profile.json
  registered (from Section C) audit_report_summary.csv
  registered (from Section C) audit_env_feature_stats.csv
  registered (from Section C) audit_correlation_matrix.csv
  registered (from Section C) audit_per_stage_env_means.csv

11 artifacts saved to:
  E:\AgriTwin-GH\src\agritwin_gh\models\artifacts\growth_progression_20260308_202542


In [17]:
# ── Section G3: Generate and save diagnostic plots ────────────────────────────────

plot_files = []

def _save_fig(fig, name):
    p = ARTIFACT_DIR / name
    fig.savefig(p, dpi=120, bbox_inches='tight')
    plt.close(fig)
    plot_files.append(name)
    print(f'  plot   {name}')


# Plot 1: Stage distribution -- original vs generated
orig_cnt = (
    tft_df.loc[tft_df['cycle_origin_type'] == 'original', 'stage_name']
    .value_counts().reindex(STAGES).fillna(0)
)
gen_cnt = (
    tft_df.loc[tft_df['cycle_origin_type'] == 'generated', 'stage_name']
    .value_counts().reindex(STAGES).fillna(0)
)
_x, _w = np.arange(len(STAGES)), 0.4
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
axes[0].bar(_x - _w/2, orig_cnt.values, _w, label='Original',  color='steelblue',  alpha=0.85)
axes[0].bar(_x + _w/2, gen_cnt.values,  _w, label='Generated', color='darkorange', alpha=0.85)
axes[0].set_xticks(_x)
axes[0].set_xticklabels(STAGES, rotation=20, ha='right', fontsize=8)
axes[0].set_ylabel('Total hourly rows')
axes[0].set_title('Stage Distribution -- Original vs Generated')
axes[0].legend()
_sch = tft_df.groupby(['cycle_id', 'stage_name'])['timestamp'].count().rename('hours').reset_index()
for _i, _sn in enumerate(STAGES):
    _sub = _sch.loc[_sch['stage_name'] == _sn, 'hours']
    axes[1].scatter([_i] * len(_sub), _sub.values, alpha=0.5, s=22, color='teal')
axes[1].set_xticks(range(len(STAGES)))
axes[1].set_xticklabels(STAGES, rotation=20, ha='right', fontsize=8)
axes[1].set_ylabel('Hours per stage (per cycle)')
axes[1].set_title('Per-Cycle Stage Duration Spread')
plt.tight_layout()
_save_fig(fig, 'stage_distribution.png')

# Plot 2: Cycle duration distribution
cycle_days = (
    tft_df.groupby(['cycle_id', 'cycle_origin_type'])
    .apply(lambda g: (g['timestamp'].max() - g['timestamp'].min()).days + 1)
    .reset_index(name='total_days')
)
_colors = cycle_days['cycle_origin_type'].map(
    {'original': 'steelblue', 'generated': 'darkorange'}
)
fig, ax = plt.subplots(figsize=(13, 4))
ax.bar(cycle_days['cycle_id'].astype(str), cycle_days['total_days'], color=_colors, alpha=0.85)
ax.axhline(cycle_days['total_days'].mean(), color='gray', linestyle='--', linewidth=1, label='Mean')
ax.set_xlabel('Cycle ID')
ax.set_ylabel('Total days')
ax.set_title('Cycle Duration Distribution  (blue=original, orange=generated)')
ax.legend()
plt.tight_layout()
_save_fig(fig, 'cycle_duration_distribution.png')

# Plot 3: Per-stage duration heatmap
_sdcols = [f'days_{s}' for s in STAGES]
_heat   = cycle_expansion_df.set_index('cycle_id')[_sdcols].copy()
_heat.columns = STAGES
fig, ax = plt.subplots(figsize=(14, 6))
_im = ax.imshow(_heat.values.T, aspect='auto', cmap='YlGn')
ax.set_xticks(range(len(cycle_expansion_df)))
ax.set_xticklabels(cycle_expansion_df['cycle_id'].astype(str))
ax.set_yticks(range(len(STAGES)))
ax.set_yticklabels(STAGES, fontsize=9)
ax.set_xlabel('Cycle ID')
ax.set_title('Stage Duration (days) per Cycle -- Heatmap')
plt.colorbar(_im, ax=ax, label='Days')
for _ci in range(len(cycle_expansion_df)):
    for _si in range(len(STAGES)):
        ax.text(_ci, _si, str(int(_heat.iloc[_ci, _si])),
                ha='center', va='center', fontsize=7)
plt.tight_layout()
_save_fig(fig, 'per_stage_durations.png')

# Plot 4: Environmental feature distributions
_env_plot = ['indoor_temp', 'indoor_humidity', 'indoor_CO2',
             'solarradiation', 'vpd', 'dew_point']
fig, axes = plt.subplots(2, 3, figsize=(14, 7))
axes = axes.flatten()
for _i, _col in enumerate(_env_plot):
    for _org, _grp in tft_df.groupby('cycle_origin_type'):
        axes[_i].hist(_grp[_col].dropna(), bins=50, alpha=0.5,
                      label=_org.capitalize(), density=True)
    axes[_i].set_title(_col, fontsize=9)
    axes[_i].tick_params(labelsize=7)
    axes[_i].legend(fontsize=7)
plt.suptitle('Env Feature Distributions -- Original vs Generated', fontsize=11, y=1.01)
plt.tight_layout()
_save_fig(fig, 'env_feature_distributions.png')

# Plot 5: Original vs generated cycle comparison
_pairs = [(1, 5), (2, 9), (3, 13)]
fig, axes = plt.subplots(len(_pairs), 2, figsize=(16, 4 * len(_pairs)))
for _ri, (_oid, _gid) in enumerate(_pairs):
    for _cj, _feat in enumerate(['indoor_temp', 'indoor_humidity']):
        _ax = axes[_ri, _cj]
        for _c, _clr, _lbl in [
            (_oid, 'steelblue',  f'Cycle {_oid} (original)'),
            (_gid, 'darkorange', f'Cycle {_gid} (generated)'),
        ]:
            _v = tft_df.loc[tft_df['cycle_id'] == _c, _feat].values
            _ax.plot(np.arange(len(_v)), _v, color=_clr,
                     alpha=0.6, linewidth=0.6, label=_lbl)
        _ax.set_title(f'{_feat}  |  cycles {_oid} vs {_gid}', fontsize=9)
        _ax.set_xlabel('Hour offset', fontsize=8)
        _ax.tick_params(labelsize=7)
        _ax.legend(fontsize=7)
plt.suptitle('Original vs Generated -- Environmental Profiles', fontsize=11, y=1.01)
plt.tight_layout()
_save_fig(fig, 'original_vs_generated_comparison.png')

saved_files.extend(plot_files)
print(f'\n{len(plot_files)} plots saved.')

  plot   stage_distribution.png
  plot   cycle_duration_distribution.png
  plot   per_stage_durations.png
  plot   env_feature_distributions.png
  plot   original_vs_generated_comparison.png

5 plots saved.


In [18]:
# ── Section G4: End-of-Phase 1 summary ────────────────────────────────────────────────
_sep = '=' * 72
print(_sep)
print('  AgriTwin-GH | TFT Pipeline -- Phase 1 Complete')
print(_sep)
print()
print(f'  Run ID                : {RUN_ID}')
print(f'  Artifact directory    : {ARTIFACT_DIR}')
print()
_n_orig = int(hourly_df['cycle_id'].nunique())
_n_gen  = len(generated_dfs)
_n_tot  = int(tft_df['cycle_id'].nunique())
print(f'  Final dataset shape   : {tft_df.shape}')
print(f'  Original cycles       : {_n_orig}')
print(f'  Generated cycles      : {_n_gen}')
print(f'  Total cycles          : {_n_tot}')
print()
print(f'  Trainable rows (48h target present) : {trainable_rows:,}')
print()
print('  Target columns created:')
for t in TARGET_COLS:
    _nn = tft_df[t].notna().sum()
    print(f'    {t:<44}: {_nn:,} non-null')
print()
print('  All artifacts saved:')
for _f in saved_files:
    print(f'    * {_f}')
print()
print('  Ready for Phase 2: TFT model training.')
print(_sep)

  AgriTwin-GH | TFT Pipeline -- Phase 1 Complete

  Run ID                : growth_progression_20260308_202542
  Artifact directory    : E:\AgriTwin-GH\src\agritwin_gh\models\artifacts\growth_progression_20260308_202542

  Final dataset shape   : (43512, 48)
  Original cycles       : 4
  Generated cycles      : 12
  Total cycles          : 16

  Trainable rows (48h target present) : 42,744

  Target columns created:
    target_stage_24h                            : 43,128 non-null
    target_stage_48h                            : 42,744 non-null
    target_stage_index_24h                      : 43,128 non-null
    target_stage_index_48h                      : 42,744 non-null
    target_stage_progress_24h                   : 43,128 non-null
    target_stage_progress_48h                   : 42,744 non-null
    target_hours_to_next_stage                  : 43,512 non-null

  All artifacts saved:
    * expanded_hourly_growth_dataset.csv
    * expanded_hourly_growth_dataset.parquet
    * cy

---
## Section H — Load Prepared Dataset

Load the Phase 1 artifact (expanded hourly dataset with 7 TFT targets), verify integrity, and register the **run ID** so all Phase 2 artifacts land in the correct folder.

In [19]:
# ── Section H1: Imports for Phase 2 ──────────────────────────────────────────────────
import os, json, pickle, warnings
from pathlib import Path
from datetime import datetime

import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import RobustScaler, LabelEncoder

os.environ['TOKENIZERS_PARALLELISM'] = 'false'
warnings.filterwarnings('ignore')

import torch
import pytorch_lightning as pl
from pytorch_forecasting import TimeSeriesDataSet
from pytorch_forecasting.data import GroupNormalizer, NaNLabelEncoder

print('Phase 2 imports OK')
print(f'  torch              : {torch.__version__}')
print(f'  pytorch_lightning   : {pl.__version__}')
print(f'  CUDA available      : {torch.cuda.is_available()}')
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'  Device              : {DEVICE}')

Phase 2 imports OK
  torch              : 2.10.0+cpu
  pytorch_lightning   : 2.6.1
  CUDA available      : False
  Device              : cpu


In [20]:
# ── Section H2: Resolve artifact directory from Phase 1 ──────────────────────────
REPO_ROOT    = Path('E:/AgriTwin-GH')
MODELS_ROOT  = REPO_ROOT / 'src' / 'agritwin_gh' / 'models'
ARTIFACT_BASE = MODELS_ROOT / 'artifacts'

# Pick the most recent growth_progression artifact folder produced in Phase 1
_dirs = sorted(
    [d for d in ARTIFACT_BASE.iterdir()
     if d.is_dir() and d.name.startswith('growth_progression_')],
    key=lambda p: p.stat().st_mtime,
)
if not _dirs:
    raise RuntimeError('No Phase 1 artifact directory found. Run Phase 1 first.')
ARTIFACT_DIR = _dirs[-1]
RUN_ID       = ARTIFACT_DIR.name          # e.g. growth_progression_20260308_121104

DATASET_CSV  = ARTIFACT_DIR / 'expanded_hourly_growth_dataset.csv'
if not DATASET_CSV.exists():
    raise FileNotFoundError(f'Dataset not found: {DATASET_CSV}')

print(f'Run ID          : {RUN_ID}')
print(f'Artifact dir    : {ARTIFACT_DIR}')
print(f'Dataset CSV     : {DATASET_CSV.stat().st_size/1e6:.1f} MB')

Run ID          : growth_progression_20260308_202542
Artifact dir    : E:\AgriTwin-GH\src\agritwin_gh\models\artifacts\growth_progression_20260308_202542
Dataset CSV     : 16.7 MB


In [21]:
# ── Section H3: Load and verify the expanded dataset ──────────────────────────────
STAGES = [
    'seedling', 'early_vegetative', 'flowering_initiation',
    'flowering', 'unripe', 'ripe',
]

df = pd.read_csv(DATASET_CSV, parse_dates=['timestamp'])
df['cycle_id']   = df['cycle_id'].astype(int)
df['stage_index'] = df['stage_index'].astype(int)
df = df.sort_values(['cycle_id', 'timestamp']).reset_index(drop=True)

TARGET_COLS = [
    'target_stage_index_24h', 'target_stage_index_48h',
    'target_stage_progress_24h', 'target_stage_progress_48h',
    'target_hours_to_next_stage',
]

print(f'Loaded shape        : {df.shape}')
print(f'Total cycles        : {df["cycle_id"].nunique()}')
print(f'Timestamp range     : {df["timestamp"].min()}  ->  {df["timestamp"].max()}')

print('\nStage distribution (hourly rows):')
for s in STAGES:
    n = (df['stage_name'] == s).sum()
    print(f'  {s:<28}: {n:>7,}')

print('\nTarget column non-null coverage:')
for c in TARGET_COLS:
    nn  = df[c].notna().sum()
    pct = nn / len(df) * 100
    print(f'  {c:<42}: {nn:>7,}  ({pct:.1f}%)')

# Timestamp continuity (no multi-hour gaps within any cycle)
_gap_violations = []
for cid, grp in df.groupby('cycle_id'):
    diffs = grp['timestamp'].sort_values().diff().dt.total_seconds().div(3600).dropna()
    bad   = diffs[diffs > 1.5]
    if not bad.empty:
        _gap_violations.append({'cycle_id': int(cid), 'gaps': int(len(bad)), 'max_h': float(bad.max())})
if _gap_violations:
    print('\nWARN: timestamp gaps >1.5h found:', _gap_violations)
else:
    print('\nTimestamp continuity: OK (no gaps >1.5h)')

print('\nCycle summary:')
print(
    df.groupby(['cycle_id', 'cycle_origin_type'])['timestamp']
    .count().rename('rows').to_string()
)

Loaded shape        : (43512, 48)
Total cycles        : 16
Timestamp range     : 2024-07-01 00:00:00  ->  2029-02-18 23:00:00

Stage distribution (hourly rows):
  seedling                    :   5,544
  early_vegetative            :   9,504
  flowering_initiation        :   4,176
  flowering                   :   6,840
  unripe                      :  12,624
  ripe                        :   4,824

Target column non-null coverage:
  target_stage_index_24h                    :  43,128  (99.1%)
  target_stage_index_48h                    :  42,744  (98.2%)
  target_stage_progress_24h                 :  43,128  (99.1%)
  target_stage_progress_48h                 :  42,744  (98.2%)
  target_hours_to_next_stage                :  43,512  (100.0%)

Timestamp continuity: OK (no gaps >1.5h)

Cycle summary:
cycle_id  cycle_origin_type
1         original             2712
2         original             2616
3         original             2808
4         original             2712
5         generated

---
## Section I — Feature Role Assignment

Explicitly assign every column to a TFT feature role. Saves `feature_roles.json` to the artifact directory.

In [22]:
# ── Section I1: Define feature roles ─────────────────────────────────────────────────────
#
# Leakage prevention decisions
# ----------------------------
# * stage_name / target_stage_* (string) are excluded from input features;
#   string label targets are used only for evaluation, not as model inputs.
# * stage_index used as OBSERVED (not known-future) because we know the
#   current stage but NOT future stages at inference time.
# * estimated_hours_to_next_stage is an OBSERVED input -- it is the same
#   as target_hours_to_next_stage (current-step value, not a future value).
# * day_night_flag / light_period_flag are calendar-derivable -> KNOWN.
# -------------------------------------------------------------------------

FEATURE_ROLES = {
    # --- identifiers (not features) ---
    'group_id'       : 'cycle_id',
    'time_idx'       : 'time_idx',          # integer index added in Section J

    # --- static categorical (constant within a cycle) ---
    'static_categoricals': [
        'cycle_origin_type',    # 'original' or 'generated'
        'season_label',         # 'southwest_monsoon' / 'northeast_monsoon' / 'summer'
    ],

    # --- static real (constant within a cycle) ---
    'static_reals': [],         # none needed

    # --- time-varying KNOWN (available at future inference time) ---
    'time_varying_known_categoricals': [],   # no hour-varying categoricals
    'time_varying_known_reals': [
        'hour',
        'month',
        'week_of_year',
        'day_of_year',
        'day_night_flag',
        'light_period_flag',
    ],

    # --- time-varying OBSERVED (known only up to present, not future) ---
    'time_varying_unknown_reals': [
        'indoor_temp',
        'indoor_humidity',
        'indoor_air_velocity',
        'indoor_CO2',
        'solarradiation',
        'vpd',
        'dew_point',
        'leaf_wetness_proxy',
        'temperature_rolling_mean_24h',
        'humidity_rolling_mean_24h',
        'cumulative_gdd_like_index',
        'stage_index',           # current stage (ordinal, treated as real)
        'stage_progress_pct',    # current stage progress %
    ],

    # --- regression targets ---
    'targets_regression': [
        'target_stage_index_24h',
        'target_stage_index_48h',
        'target_stage_progress_24h',
        'target_stage_progress_48h',
        'target_hours_to_next_stage',
    ],

    # --- horizon config ---
    'encoder_length'    : 72,
    'prediction_horizon': 48,

    # --- leakage notes ---
    'leakage_notes': [
        'stage_index (observed) contains only current-step stage, not future.',
        'target_stage_index_* are built via within-cycle shift in Phase 1 (no cross-cycle leakage).',
        'NaN target rows (last 48h of each cycle) are excluded before training.',
        'Rolling means computed within original cycle boundaries; no look-ahead.',
    ],
}

(ARTIFACT_DIR / 'feature_roles.json').write_text(
    json.dumps(FEATURE_ROLES, indent=2), encoding='utf-8'
)
print('Feature roles saved to feature_roles.json')
print('\nSummary:')
for role, val in FEATURE_ROLES.items():
    if isinstance(val, list):
        print(f'  {role:<40}: {len(val)} items -> {val}')
    else:
        print(f'  {role:<40}: {val}')

Feature roles saved to feature_roles.json

Summary:
  group_id                                : cycle_id
  time_idx                                : time_idx
  static_categoricals                     : 2 items -> ['cycle_origin_type', 'season_label']
  static_reals                            : 0 items -> []
  time_varying_known_categoricals         : 0 items -> []
  time_varying_known_reals                : 6 items -> ['hour', 'month', 'week_of_year', 'day_of_year', 'day_night_flag', 'light_period_flag']
  time_varying_unknown_reals              : 13 items -> ['indoor_temp', 'indoor_humidity', 'indoor_air_velocity', 'indoor_CO2', 'solarradiation', 'vpd', 'dew_point', 'leaf_wetness_proxy', 'temperature_rolling_mean_24h', 'humidity_rolling_mean_24h', 'cumulative_gdd_like_index', 'stage_index', 'stage_progress_pct']
  targets_regression                      : 5 items -> ['target_stage_index_24h', 'target_stage_index_48h', 'target_stage_progress_24h', 'target_stage_progress_48h', 'target_h

---
## Section J — Sequence Construction

Add a monotonic `time_idx` integer per cycle (required by `TimeSeriesDataSet`), encode categorical columns, drop NaN-target rows, and persist the sequence-ready DataFrame.

In [23]:
# ── Section J1: Prepare sequence-ready DataFrame ─────────────────────────────────────
seq_df = df.copy()

# 1. Add global integer time index (required by TimeSeriesDataSet)
#    It must be consecutive within each group after sorted by timestamp.
seq_df = seq_df.sort_values(['cycle_id', 'timestamp']).reset_index(drop=True)
seq_df['time_idx'] = seq_df.groupby('cycle_id').cumcount().astype(int)

# 2. For TimeSeriesDataSet we also need a global monotonic time index
#    that is used for min_encoder_length arithmetic.
#    Use timestamp-derived integer (hours from an epoch).
_epoch = seq_df['timestamp'].min()
seq_df['abs_time_idx'] = (
    (seq_df['timestamp'] - _epoch).dt.total_seconds() // 3600
).astype(int)

# 3. Encode categorical columns as strings (TimeSeriesDataSet handles encoding)
for cat_col in (FEATURE_ROLES['static_categoricals']
                + FEATURE_ROLES['time_varying_known_categoricals']):
    if cat_col in seq_df.columns:
        seq_df[cat_col] = seq_df[cat_col].astype(str)

# 4. Ensure integer-typed known reals are float
for col in FEATURE_ROLES['time_varying_known_reals']:
    seq_df[col] = seq_df[col].astype(float)

# 5. Drop rows where any regression target is NaN
#    (last 48 h of each cycle has no 48h-ahead label)
_before = len(seq_df)
seq_df = seq_df.dropna(subset=TARGET_COLS).reset_index(drop=True)
_after  = len(seq_df)
print(f'Rows dropped (NaN targets): {_before - _after:,}')
print(f'Sequence-ready rows        : {_after:,}')

# 6. Ensure target columns are float32
for t in TARGET_COLS:
    seq_df[t] = seq_df[t].astype('float32')

ENC_LEN  = FEATURE_ROLES['encoder_length']       # 72
PRED_LEN = FEATURE_ROLES['prediction_horizon']   # 48

# 7. Verify each cycle is at least encoder + prediction long
_cycle_lens = seq_df.groupby('cycle_id')['time_idx'].count()
_min_required = ENC_LEN + PRED_LEN
_short = _cycle_lens[_cycle_lens < _min_required]
if not _short.empty:
    print(f'WARN: {len(_short)} cycles shorter than {_min_required}h (will be skipped by DataSet): {list(_short.index)}')

print(f'\nEncoder length     : {ENC_LEN} h')
print(f'Prediction horizon : {PRED_LEN} h')
print(f'Min sequence length: {_min_required} h')
print(f'Usable cycles      : {int((~_cycle_lens.index.isin(_short.index)).sum())} / {len(_cycle_lens)}')

Rows dropped (NaN targets): 768
Sequence-ready rows        : 42,744

Encoder length     : 72 h
Prediction horizon : 48 h
Min sequence length: 120 h
Usable cycles      : 16 / 16


In [24]:
# ── Section J2: Save sequence-ready DataFrame ─────────────────────────────────────────
_seq_out = ARTIFACT_DIR / 'sequence_ready_dataset.csv'
seq_df.to_csv(_seq_out, index=False)
print(f'Sequence-ready dataset saved: {_seq_out.name}  ({_seq_out.stat().st_size/1e6:.1f} MB)')
print(f'Shape: {seq_df.shape}')
print(f'Columns ({len(seq_df.columns)}): {list(seq_df.columns)}')

Sequence-ready dataset saved: sequence_ready_dataset.csv  (17.3 MB)
Shape: (42744, 50)
Columns (50): ['timestamp', 'cycle_id', 'cycle_label', 'season_window', 'real_or_synthetic_flag', 'year', 'month', 'day_of_year', 'week_of_year', 'hour', 'season_label', 'days_from_cycle_start', 'stage_name', 'stage_index', 'hours_in_current_stage', 'days_in_current_stage', 'stage_duration_hours', 'stage_duration_days', 'stage_progress_pct', 'total_cycle_progress_pct', 'estimated_days_to_next_stage', 'estimated_hours_to_next_stage', 'is_stage_transition', 'indoor_temp', 'indoor_humidity', 'indoor_air_velocity', 'indoor_CO2', 'solarradiation', 'day_night_flag', 'vpd', 'dew_point', 'leaf_wetness_proxy', 'temperature_rolling_mean_24h', 'humidity_rolling_mean_24h', 'vpd_proxy', 'light_period_flag', 'cumulative_gdd_like_index', 'cycle_origin_type', 'prototype_cycle_id', 'augmentation_method', 'run_id', 'target_stage_24h', 'target_stage_index_24h', 'target_stage_progress_24h', 'target_stage_48h', 'target_s

---
## Section K — Dataset Splitting

Split cycles into train / validation / test sets. **Splits are always at cycle boundaries** — never inside a cycle. The split is stratified by season type so each set contains a representative mix of kharif / rabi / summer cycles.

In [25]:
# ── Section K1: Cycle-aware stratified split ─────────────────────────────────────────
import math

# Build a per-cycle metadata table
cycle_meta = (
    seq_df.groupby('cycle_id')
    .agg(
        rows          = ('time_idx', 'count'),
        season_type   = ('cycle_label', 'first'),
        origin        = ('cycle_origin_type', 'first'),
        start_ts      = ('timestamp', 'min'),
    )
    .reset_index()
)
# Extract season from cycle_label (e.g. 'kharif_2024' -> 'kharif')
cycle_meta['season'] = cycle_meta['season_type'].str.split('_').str[0]

all_cycle_ids = sorted(cycle_meta['cycle_id'].tolist())
n_cycles      = len(all_cycle_ids)
print(f'Total usable cycles: {n_cycles}')

# Strategy: 70/15/15 by cycle count, stratified by season type
# With 16 cycles (4 per season x 3 seasons + 4 extra kharif):
#   kharif: 8 cycles  -> train 6, val 1, test 1
#   rabi  : 4 cycles  -> train 3, val 1, test 1  (rabi has 5 total: 1 orig + 4 gen)
#   summer: 4 cycles  -> train 3, val 1, test 1  (summer: 1 orig + 4 gen)

rng_split = np.random.default_rng(7)

train_ids, val_ids, test_ids = [], [], []

for season, grp in cycle_meta.groupby('season'):
    cids = grp.sort_values('start_ts')['cycle_id'].tolist()
    n    = len(cids)
    # Shuffle within season so generated cycles are spread across splits
    rng_split.shuffle(cids)
    n_test = max(1, math.floor(n * 0.15))
    n_val  = max(1, math.floor(n * 0.15))
    n_train = n - n_test - n_val
    if n_train < 1:
        n_train = 1
        if n_val > 1: n_val -= 1
        elif n_test > 1: n_test -= 1
    train_ids.extend(cids[:n_train])
    val_ids.extend(cids[n_train:n_train + n_val])
    test_ids.extend(cids[n_train + n_val:])
    print(f'  {season:8}: {n:>2} cycles -> train={n_train}  val={n_val}  test={n_test}')

# Sanity
_all_assigned = set(train_ids) | set(val_ids) | set(test_ids)
assert _all_assigned == set(all_cycle_ids), 'Some cycles unassigned!'
assert not (set(train_ids) & set(val_ids)), 'Train/val overlap!'
assert not (set(train_ids) & set(test_ids)), 'Train/test overlap!'

print(f'\nFinal split:')
print(f'  Train : {sorted(train_ids)}')
print(f'  Val   : {sorted(val_ids)}')
print(f'  Test  : {sorted(test_ids)}')

Total usable cycles: 16
  kharif  :  6 cycles -> train=4  val=1  test=1
  rabi    :  5 cycles -> train=3  val=1  test=1
  summer  :  5 cycles -> train=3  val=1  test=1

Final split:
  Train : [1, 2, 3, 5, 6, 8, 9, 11, 13, 14]
  Val   : [4, 10, 15]
  Test  : [7, 12, 16]


In [26]:
# ── Section K2: Apply split labels and save split summary ─────────────────────────
def _label_split(cid):
    if cid in train_ids: return 'train'
    if cid in val_ids:   return 'val'
    return 'test'

seq_df['split'] = seq_df['cycle_id'].map(_label_split)

train_df = seq_df[seq_df['split'] == 'train'].copy()
val_df   = seq_df[seq_df['split'] == 'val'].copy()
test_df  = seq_df[seq_df['split'] == 'test'].copy()

split_summary = {
    'run_id'        : RUN_ID,
    'split_strategy': 'cycle-stratified 70/15/15 by season type',
    'train': {
        'cycles': sorted(train_ids),
        'rows'  : int(len(train_df)),
        'seasons': train_df['season_label'].unique().tolist(),
    },
    'val': {
        'cycles': sorted(val_ids),
        'rows'  : int(len(val_df)),
    },
    'test': {
        'cycles': sorted(test_ids),
        'rows'  : int(len(test_df)),
    },
    'encoder_length'    : ENC_LEN,
    'prediction_horizon': PRED_LEN,
    'leakage_check'     : 'Splits applied at cycle boundaries only.',
}

(ARTIFACT_DIR / 'split_summary.json').write_text(
    json.dumps(split_summary, indent=2), encoding='utf-8'
)

print('Split summary saved.')
print(f'  Train : {len(train_ids):>2} cycles   {len(train_df):>7,} rows')
print(f'  Val   : {len(val_ids):>2} cycles   {len(val_df):>7,} rows')
print(f'  Test  : {len(test_ids):>2} cycles   {len(test_df):>7,} rows')

Split summary saved.
  Train : 10 cycles    26,736 rows
  Val   :  3 cycles     8,016 rows
  Test  :  3 cycles     7,992 rows


---
## Section L — Dataset Normalization

Fit scalers on the **training set only**, transform train / val / test, and persist the fitted scalers for inference.

In [27]:
# ── Section L1: Fit RobustScaler on training data ──────────────────────────────────
#
# RobustScaler (IQR-based) handles the skewed env distributions well.
# Scalers are fit ONLY on train_df to prevent leakage into val/test.
# Categorical columns and target columns are NOT scaled here;
# TimeSeriesDataSet handles target normalisation internally via
# GroupNormalizer / EncoderNormalizer when the model is built.
# -------------------------------------------------------------------------

ALL_REAL_FEATURES = (
    FEATURE_ROLES['time_varying_known_reals']
    + FEATURE_ROLES['time_varying_unknown_reals']
)

scaler = RobustScaler()
scaler.fit(train_df[ALL_REAL_FEATURES])

# Apply transform to all three splits
for _split_df in [train_df, val_df, test_df]:
    _split_df[ALL_REAL_FEATURES] = scaler.transform(
        _split_df[ALL_REAL_FEATURES]
    ).astype('float32')

# Apply to the full seq_df so normalised version is consistent
seq_df[ALL_REAL_FEATURES] = scaler.transform(
    seq_df[ALL_REAL_FEATURES]
).astype('float32')

# Persist scaler
_scaler_path = ARTIFACT_DIR / 'robust_scaler.pkl'
with open(_scaler_path, 'wb') as _f:
    pickle.dump(scaler, _f)

print(f'RobustScaler fitted on {len(train_df):,} training rows.')
print(f'Scaled features : {len(ALL_REAL_FEATURES)}')
print(f'Scaler saved    : {_scaler_path.name}')

# Sanity-check: train scaled stats should be near 0 median, ~1 IQR
print('\nPost-scaling train stats (first 5 features):')
print(
    train_df[ALL_REAL_FEATURES[:5]].describe().loc[['25%', '50%', '75%']].round(3).to_string()
)

RobustScaler fitted on 26,736 training rows.
Scaled features : 19
Scaler saved    : robust_scaler.pkl

Post-scaling train stats (first 5 features):
       hour   month  week_of_year  day_of_year  day_night_flag
25% -0.5000 -0.6000       -0.5830      -0.5760         -1.0000
50%  0.0000  0.0000        0.0000       0.0000          0.0000
75%  0.5000  0.4000        0.4170       0.4240          0.0000


In [28]:
# ── Section L2: Build TimeSeriesDataSet objects ────────────────────────────────────
#
# We use a single regression target (stage_index_24h) as the primary target
# for TimeSeriesDataSet.  Additional targets are carried as extra columns and
# extracted during the custom training loop (Phase 3).
#
# NaNLabelEncoder(add_nan=True) is used for all categoricals so that
# val/test cycles (unseen group IDs) are encoded without ValueError.
# This is the pytorch-forecasting recommended pattern for hold-out groups.
#
# EncoderNormalizer (transformation=None) replaces the previous
# GroupNormalizer(transformation='softplus') for the following reasons:
#   • softplus is nonlinear — it compresses the [0,5] stage-index range
#     unevenly, making it hard for the model to distinguish intermediate
#     stages (3-Flowering, 4-Unripe) from stage 5 (Ripe).
#   • EncoderNormalizer normalises linearly by the encoder window's own
#     target statistics (mean/std) at batch time — fully adaptive and
#     has no unseen-group issue at inference with cycle_id="999".
#   • Linear normalisation preserves ordinal spacing between stage labels,
#     so the model correctly penalises a 2-stage jump more than a 1-stage jump.
# -------------------------------------------------------------------------

from pytorch_forecasting.data.encoders import NaNLabelEncoder, EncoderNormalizer

_primary_target = 'target_stage_index_24h'

# Pre-build encoder dict: allow unknown categories in val/test groups
_cat_encoders = {
    col: NaNLabelEncoder(add_nan=True)
    for col in FEATURE_ROLES['static_categoricals'] + ['cycle_id']
}

training_dataset = TimeSeriesDataSet(
    train_df,
    time_idx                       = 'time_idx',
    target                         = _primary_target,
    group_ids                      = ['cycle_id'],
    min_encoder_length             = ENC_LEN // 2,
    max_encoder_length             = ENC_LEN,
    min_prediction_length          = 1,
    max_prediction_length          = PRED_LEN,
    static_categoricals            = FEATURE_ROLES['static_categoricals'],
    static_reals                   = FEATURE_ROLES['static_reals'],
    time_varying_known_categoricals= [],
    time_varying_known_reals       = FEATURE_ROLES['time_varying_known_reals'],
    time_varying_unknown_reals     = (
        FEATURE_ROLES['time_varying_unknown_reals']
        + [_primary_target]       # target as unknown real (standard for TFT)
    ),
    target_normalizer              = EncoderNormalizer(
        transformation=None,      # linear — preserves ordinal stage-index spacing
    ),
    categorical_encoders           = _cat_encoders,
    add_relative_time_idx          = True,
    add_target_scales              = True,
    add_encoder_length             = True,
    allow_missing_timesteps        = False,
)

validation_dataset = TimeSeriesDataSet.from_dataset(
    training_dataset, val_df, predict=False, stop_randomization=True
)
test_dataset = TimeSeriesDataSet.from_dataset(
    training_dataset, test_df, predict=True, stop_randomization=True
)

print(f'Training   TimeSeriesDataSet : {len(training_dataset):>6,} samples')
print(f'Validation TimeSeriesDataSet : {len(validation_dataset):>6,} samples')
print(f'Test       TimeSeriesDataSet : {len(test_dataset):>6,} samples')

# Persist the dataset objects so Phase 3 can reload without re-processing
_ds_path = ARTIFACT_DIR / 'tft_training_dataset.pkl'
with open(_ds_path, 'wb') as _f:
    pickle.dump({'training': training_dataset,
                 'validation': validation_dataset,
                 'test': test_dataset,
                 'train_df': train_df,
                 'val_df': val_df,
                 'test_df': test_df}, _f)
print(f'\nDataset objects persisted: {_ds_path.name}  ({_ds_path.stat().st_size/1e6:.1f} MB)')
print('\nNOTE: target_normalizer changed to EncoderNormalizer(transformation=None)')
print('      This fixes softplus compression that caused Flowering/Unripe → Ripe jump in v1.')


Training   TimeSeriesDataSet : 27,206 samples
Validation TimeSeriesDataSet :  8,157 samples
Test       TimeSeriesDataSet :      3 samples

Dataset objects persisted: tft_training_dataset.pkl  (18.2 MB)

NOTE: target_normalizer changed to EncoderNormalizer(transformation=None)
      This fixes softplus compression that caused Flowering/Unripe → Ripe jump in v1.


---
## Section M — Dataset Validation

Verify sequence shapes, feature counts, target integrity, and stage-transition coverage. Saves `sequence_dataset_preview.csv` and `sequence_metadata.json`.

In [29]:
# ── Section M1: Sequence shape and feature validation ─────────────────────────────
# to_dataloader() applies pytorch-forecasting's custom collate fn which
# splits the full sequence into encoder_cat/encoder_cont/decoder_cat/decoder_cont.
_loader = training_dataset.to_dataloader(train=False, batch_size=4, num_workers=0)
(_x, _y) = next(iter(_loader))

print('=== TimeSeriesDataSet batch shapes ================================')
print(f'  encoder_cat  : {_x["encoder_cat"].shape}   # (B, enc_len, n_cat)')
print(f'  encoder_cont : {_x["encoder_cont"].shape}  # (B, enc_len, n_cont)')
print(f'  decoder_cat  : {_x["decoder_cat"].shape}   # (B, pred_len, n_cat)')
print(f'  decoder_cont : {_x["decoder_cont"].shape}  # (B, pred_len, n_cont)')
print(f'  target y[0]  : {_y[0].shape}               # (B, pred_len)')

_n_enc_cont = _x['encoder_cont'].shape[-1]
_n_dec_cont = _x['decoder_cont'].shape[-1]
_n_enc_cat  = _x['encoder_cat'].shape[-1]

print('\n=== Feature count summary ==================================')
print(f'  Encoder categorical features   : {_n_enc_cat}')
print(f'  Encoder continuous features    : {_n_enc_cont}')
print(f'  Decoder continuous features    : {_n_dec_cont} (known + relative_time_idx extras)')

=== TimeSeriesDataSet batch shapes ================================
  encoder_cat  : torch.Size([4, 72, 2])   # (B, enc_len, n_cat)
  encoder_cont : torch.Size([4, 72, 24])  # (B, enc_len, n_cont)
  decoder_cat  : torch.Size([4, 48, 2])   # (B, pred_len, n_cat)
  decoder_cont : torch.Size([4, 48, 24])  # (B, pred_len, n_cont)
  target y[0]  : torch.Size([4, 48])               # (B, pred_len)

=== Feature count summary ==================================
  Encoder categorical features   : 2
  Encoder continuous features    : 24
  Decoder continuous features    : 24 (known + relative_time_idx extras)


In [30]:
# ── Section M2: Target integrity and stage transition coverage ─────────────────────
print('=== Target column integrity ================================')
for t in TARGET_COLS:
    nan_n  = seq_df[t].isna().sum()
    neg_n  = (seq_df[t] < 0).sum()
    mn, mx = seq_df[t].min(), seq_df[t].max()
    print(f'  {t:<42} NaN={nan_n}  neg={neg_n}  [{mn:.3f}, {mx:.3f}]')

# Stage transition coverage per split
print('\n=== Stage transition coverage per split ====================')
for _name, _sdf in [('train', train_df), ('val', val_df), ('test', test_df)]:
    _trans = _sdf['is_stage_transition'].sum() if 'is_stage_transition' in _sdf.columns else 'N/A'
    _stages = sorted(_sdf['stage_name'].unique().tolist())
    print(f'  {_name:5}: transitions={_trans}   stages present={_stages}')

# Stage index range check
print('\n=== Stage index range in targets ===========================')
for t in ['target_stage_index_24h', 'target_stage_index_48h']:
    vals = seq_df[t].dropna().astype(int)
    print(f'  {t}: unique={sorted(vals.unique().tolist())}')

# Progress range check
for t in ['target_stage_progress_24h', 'target_stage_progress_48h']:
    mn, mx = seq_df[t].dropna().min(), seq_df[t].dropna().max()
    print(f'  {t}: [{mn:.1f}, {mx:.1f}]')

=== Target column integrity ================================
  target_stage_index_24h                     NaN=0  neg=0  [0.000, 5.000]
  target_stage_index_48h                     NaN=0  neg=0  [0.000, 5.000]
  target_stage_progress_24h                  NaN=0  neg=0  [0.000, 99.890]
  target_stage_progress_48h                  NaN=0  neg=0  [0.000, 99.890]
  target_hours_to_next_stage                 NaN=0  neg=0  [1.000, 912.000]

=== Stage transition coverage per split ====================
  train: transitions=50   stages present=['early_vegetative', 'flowering', 'flowering_initiation', 'ripe', 'seedling', 'unripe']
  val  : transitions=15   stages present=['early_vegetative', 'flowering', 'flowering_initiation', 'ripe', 'seedling', 'unripe']
  test : transitions=15   stages present=['early_vegetative', 'flowering', 'flowering_initiation', 'ripe', 'seedling', 'unripe']

=== Stage index range in targets ===========================
  target_stage_index_24h: unique=[0, 1, 2, 3, 4, 5]
  

In [31]:
# ── Section M3: Save preview and metadata ───────────────────────────────────────────────
# Preview: 5 rows from each cycle
_prev_frames = []
for cid, grp in seq_df.groupby('cycle_id'):
    idx = np.linspace(0, len(grp) - 1, min(5, len(grp)), dtype=int)
    _prev_frames.append(grp.iloc[idx])
preview_df = pd.concat(_prev_frames, ignore_index=True)
_preview_path = ARTIFACT_DIR / 'sequence_dataset_preview.csv'
preview_df.to_csv(_preview_path, index=False)
print(f'Preview saved : {_preview_path.name}  ({len(preview_df)} rows)')

# Sequence metadata
seq_meta = {
    'run_id'             : RUN_ID,
    'created_on'         : datetime.now().isoformat(),
    'sequence_ready_rows': int(len(seq_df)),
    'encoder_length'     : ENC_LEN,
    'prediction_horizon' : PRED_LEN,
    'primary_target'     : 'target_stage_index_24h',
    'all_targets'        : TARGET_COLS,
    'train_cycles'       : sorted(train_ids),
    'val_cycles'         : sorted(val_ids),
    'test_cycles'        : sorted(test_ids),
    'train_samples'      : int(len(training_dataset)),
    'val_samples'        : int(len(validation_dataset)),
    'test_samples'       : int(len(test_dataset)),
    'scaled_features'    : ALL_REAL_FEATURES,
    'scaler_type'        : 'RobustScaler',
    'scaler_file'        : 'robust_scaler.pkl',
    'dataset_file'       : 'tft_training_dataset.pkl',
    'feature_counts': {
        'static_categoricals'            : len(FEATURE_ROLES['static_categoricals']),
        'time_varying_known_categoricals' : len(FEATURE_ROLES['time_varying_known_categoricals']),
        'time_varying_known_reals'        : len(FEATURE_ROLES['time_varying_known_reals']),
        'time_varying_unknown_reals'      : len(FEATURE_ROLES['time_varying_unknown_reals']),
    },
    'batch_shapes': {
        'encoder_cat' : list(_x['encoder_cat'].shape),
        'encoder_cont': list(_x['encoder_cont'].shape),
        'decoder_cat' : list(_x['decoder_cat'].shape),
        'decoder_cont': list(_x['decoder_cont'].shape),
        'target'      : list(_y[0].shape),
    },
    'ready_for_phase3': True,
}
_meta_path = ARTIFACT_DIR / 'sequence_metadata.json'
_meta_path.write_text(json.dumps(seq_meta, indent=2), encoding='utf-8')
print(f'Metadata saved: {_meta_path.name}')

print('\n' + '=' * 68)
print('  Phase 2 (H-M): Dataset preparation COMPLETE')
print('=' * 68)
print(f'  Run ID           : {RUN_ID}')
print(f'  Artifact dir     : {ARTIFACT_DIR}')
print(f'  Sequence rows    : {len(seq_df):,}')
print(f'  Train samples    : {len(training_dataset):,}')
print(f'  Val samples      : {len(validation_dataset):,}')
print(f'  Test samples     : {len(test_dataset):,}')
print(f'  Next step        : Phase 3 — TFT model training (Sections K-N)')
print('=' * 68)

Preview saved : sequence_dataset_preview.csv  (80 rows)
Metadata saved: sequence_metadata.json

  Phase 2 (H-M): Dataset preparation COMPLETE
  Run ID           : growth_progression_20260308_202542
  Artifact dir     : E:\AgriTwin-GH\src\agritwin_gh\models\artifacts\growth_progression_20260308_202542
  Sequence rows    : 42,744
  Train samples    : 27,206
  Val samples      : 8,157
  Test samples     : 3
  Next step        : Phase 3 — TFT model training (Sections K-N)


---
## Section N — Training Configuration

Define all hyperparameters for the TFT training run, persist them as
`training_config.json`, and reload the `TimeSeriesDataSet` objects produced
by Phase 2.  
Key choices:
- `hidden_size` / `attention_head_size` kept compact (32 / 4) for CPU training
- QuantileLoss with 7 quantiles gives calibrated prediction intervals
- `encoder_length=72 h`, `prediction_length=48 h` (matches Phase 2 sequences)


In [32]:
# ── Section N1: Phase 3 imports ───────────────────────────────────────────────
import json, pickle, shutil, warnings
from datetime import datetime
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import lightning.pytorch as pl                              # use lightning.pytorch
from lightning.pytorch.callbacks import (                  # TFT is lightning.pytorch
    EarlyStopping, LearningRateMonitor, ModelCheckpoint
)
from pytorch_forecasting import TemporalFusionTransformer
from pytorch_forecasting.metrics import QuantileLoss
from sklearn.metrics import (
    accuracy_score, balanced_accuracy_score,
    confusion_matrix, explained_variance_score,
    f1_score, mean_absolute_error,
    mean_absolute_percentage_error, mean_squared_error,
    median_absolute_error, precision_score,
    r2_score, recall_score,
)

warnings.filterwarnings("ignore")


In [33]:
# ── Section N2: Resolve artifact directory and define training config ─────────
REPO_ROOT      = Path().resolve().parent
ARTIFACT_BASE  = REPO_ROOT / "src" / "agritwin_gh" / "models" / "artifacts"
MODEL_OUT_DIR  = REPO_ROOT / "src" / "agritwin_gh" / "models"
MODEL_OUT_DIR.mkdir(parents=True, exist_ok=True)

ARTIFACT_DIR = sorted(
    [d for d in ARTIFACT_BASE.iterdir()
     if d.is_dir() and d.name.startswith("growth_progression_")],
    key=lambda p: p.stat().st_mtime,
)[-1]
RUN_ID = ARTIFACT_DIR.name
print(f"Artifact dir : {ARTIFACT_DIR.name}")

# Hyperparameters (v2 — improved for ordinal stage accuracy)
# Changes vs v1:
#   hidden_size            : 32  → 64   (double capacity to separate 6 stages)
#   hidden_continuous_size : 16  → 32   (scaled with hidden_size)
#   lstm_layers            : 1   → 2    (deeper temporal reasoning)
#   dropout                : 0.1 → 0.15 (slightly more regularisation for larger model)
#   max_epochs             : 50  → 100  (larger model needs more iterations)
#   early_stop_patience    : 5   → 10   (don't stop before the model converges)
#   reduce_on_plateau_patience: 3 → 5   (give LR scheduler more patience)
#   gradient_clip_val      : 0.1 → 0.5  (allow larger gradient updates for faster convergence)
#   learning_rate          : 3e-4→ 1e-3 (higher initial LR; scheduler will reduce it)
TRAINING_CONFIG = {
    "run_id"                    : RUN_ID,
    "learning_rate"             : 1e-3,
    "batch_size"                : 64,
    "max_epochs"                : 100,
    "dropout"                   : 0.15,
    "attention_head_size"       : 4,
    "hidden_size"               : 64,
    "hidden_continuous_size"    : 32,
    "lstm_layers"               : 2,
    "encoder_length"            : 72,
    "prediction_length"         : 48,
    "gradient_clip_val"         : 0.5,
    "early_stop_patience"       : 10,
    "reduce_on_plateau_patience": 5,
    "quantiles"                 : [0.02, 0.1, 0.25, 0.5, 0.75, 0.9, 0.98],
    "seed"                      : 42,
    "accelerator"               : "cpu",
    "num_workers"               : 0,
}
MEDIAN_IDX = len(TRAINING_CONFIG["quantiles"]) // 2   # index 3 → quantile 0.5

_cfg_path = ARTIFACT_DIR / "training_config.json"
_cfg_path.write_text(json.dumps(TRAINING_CONFIG, indent=2))
print(f"Training config saved : {_cfg_path.name}")

for _k, _v in TRAINING_CONFIG.items():
    if _k not in ("run_id", "accelerator", "num_workers"):
        print(f"  {_k:<30}: {_v}")


Artifact dir : growth_progression_20260308_202542
Training config saved : training_config.json
  learning_rate                 : 0.001
  batch_size                    : 64
  max_epochs                    : 100
  dropout                       : 0.15
  attention_head_size           : 4
  hidden_size                   : 64
  hidden_continuous_size        : 32
  lstm_layers                   : 2
  encoder_length                : 72
  prediction_length             : 48
  gradient_clip_val             : 0.5
  early_stop_patience           : 10
  reduce_on_plateau_patience    : 5
  quantiles                     : [0.02, 0.1, 0.25, 0.5, 0.75, 0.9, 0.98]
  seed                          : 42


In [34]:
# ── Section N3: Load Phase 2 dataset objects ──────────────────────────────────
print("Loading Phase 2 dataset objects…")
_pkl = ARTIFACT_DIR / "tft_training_dataset.pkl"
with open(_pkl, "rb") as _fh:
    _saved = pickle.load(_fh)

training_dataset   = _saved["training"]
validation_dataset = _saved["validation"]
test_dataset       = _saved["test"]
train_df           = _saved["train_df"]
val_df             = _saved["val_df"]
test_df            = _saved["test_df"]

_BS = TRAINING_CONFIG["batch_size"]
_NW = TRAINING_CONFIG["num_workers"]
train_dl = training_dataset.to_dataloader(  train=True,  batch_size=_BS,    num_workers=_NW)
val_dl   = validation_dataset.to_dataloader(train=False, batch_size=_BS * 2, num_workers=_NW)
test_dl  = test_dataset.to_dataloader(      train=False, batch_size=_BS * 2, num_workers=_NW)

print(f"  Train batches : {len(train_dl):,}  ({len(training_dataset):,} samples)")
print(f"  Val   batches : {len(val_dl):,}  ({len(validation_dataset):,} samples)")
print(f"  Test  batches : {len(test_dl):,}  ({len(test_dataset):,} samples)")

# Pre-build fast O(1) lookup for ground-truth target values from val / test DFs.
# Targets were NOT RobustScaler-scaled (only features were), so values are in
# the original [0, 5] integer stage-index space.
_val_gt_lookup  = val_df.set_index(["cycle_id", "time_idx"])["target_stage_index_24h"]
_test_gt_lookup = test_df.set_index(["cycle_id", "time_idx"])["target_stage_index_24h"]
print("Ground-truth lookup tables built.")


Loading Phase 2 dataset objects…
  Train batches : 425  (27,206 samples)
  Val   batches : 64  (8,157 samples)
  Test  batches : 1  (3 samples)
Ground-truth lookup tables built.


---
## Section O — TFT Model Architecture (v2)

Instantiate `TemporalFusionTransformer` directly from the `training_dataset`
so that all categorical embedding sizes, variable cardinalities and encoder
configurations are inferred automatically.

**v2 design changes** (fixes for Flowering→Ripe and Unripe→Ripe stage skipping):
- **`hidden_size` 32 → 64**: Doubles model capacity; the wider hidden dimension provides more representational room to separate all 6 ordinal stages instead of collapsing intermediate stages.
- **`lstm_layers` 1 → 2**: Two LSTM layers enable the encoder to build richer temporal features (short-term diurnal patterns AND long-term growth trends).
- **`hidden_continuous_size` 16 → 32**: Scaled with hidden_size to avoid an embedding bottleneck.
- **`OrdinalQuantileLoss`** replaces plain `QuantileLoss`: adds a quadratic penalty for `|median_pred − truth| > 1 stage`, discouraging multi-stage jumps while keeping the 7-quantile prediction intervals.
- **`EncoderNormalizer(transformation=None)`** (from Section L2): linear normalisation preserves ordinal spacing — the model can now properly learn Flowering (3) vs Unripe (4) vs Ripe (5) boundaries.


In [35]:
# ── Section O1: Build Temporal Fusion Transformer ─────────────────────────────
#
# OrdinalQuantileLoss — prevents multi-stage jumps (e.g. Flowering→Ripe)
# -----------------------------------------------------------------------
# Standard QuantileLoss penalises all residuals linearly, so predicting
# stage 5 instead of 3 (2-stage jump) costs only 2× predicting stage 4
# instead of 3 (1-stage jump).  In practice the model exploits this by
# collapsing to safe predictions (Early_Veg or Ripe).
#
# OrdinalQuantileLoss adds a quadratic penalty that activates whenever
# |median_prediction − truth| > 1 stage, making multi-stage jumps
# disproportionately costly and forcing the model to distinguish
# Flowering (3) / Unripe (4) from Ripe (5).
#
# ordinal_weight=0.4 was chosen so the auxiliary penalty is ~40% of the
# base QuantileLoss — meaningful but not overwhelming.
#
import torch

class OrdinalQuantileLoss(QuantileLoss):
    """QuantileLoss augmented with an ordinal skip penalty."""

    def __init__(self, quantiles, ordinal_weight: float = 0.4, **kwargs):
        super().__init__(quantiles=quantiles, **kwargs)
        self.ordinal_weight = ordinal_weight

    def loss(self, y_pred: torch.Tensor, y_actual: torch.Tensor) -> torch.Tensor:
        q_loss  = super().loss(y_pred, y_actual)           # standard quantile loss
        med_idx = len(self.quantiles) // 2
        q_med   = y_pred[..., med_idx]                    # median quantile (B, T)
        gap     = (q_med - y_actual).abs()
        # quadratic penalty kicks in only when error > 1 stage
        skip    = torch.clamp(gap - 1.0, min=0.0) ** 2
        return q_loss + self.ordinal_weight * skip.mean()

# ── TFT model (v2: 2× capacity + ordinal loss) ─────────────────────────────
# Key architecture changes vs v1:
#   hidden_size 32→64, lstm_layers 1→2, hidden_continuous_size 16→32
#   OrdinalQuantileLoss replaces plain QuantileLoss
pl.seed_everything(TRAINING_CONFIG["seed"], workers=True)

tft = TemporalFusionTransformer.from_dataset(
    training_dataset,
    learning_rate           = TRAINING_CONFIG["learning_rate"],
    hidden_size             = TRAINING_CONFIG["hidden_size"],
    lstm_layers             = TRAINING_CONFIG["lstm_layers"],
    attention_head_size     = TRAINING_CONFIG["attention_head_size"],
    dropout                 = TRAINING_CONFIG["dropout"],
    hidden_continuous_size  = TRAINING_CONFIG["hidden_continuous_size"],
    loss                    = OrdinalQuantileLoss(
        quantiles=TRAINING_CONFIG["quantiles"], ordinal_weight=0.4,
    ),
    reduce_on_plateau_patience = TRAINING_CONFIG["reduce_on_plateau_patience"],
    log_interval            = 10,
    log_val_interval        = 1,
)

_n_total   = sum(p.numel() for p in tft.parameters())
_n_train   = sum(p.numel() for p in tft.parameters() if p.requires_grad)
_n_heads   = TRAINING_CONFIG["hidden_size"] // TRAINING_CONFIG["attention_head_size"]

print("Temporal Fusion Transformer built (v2 — larger + ordinal loss)")
print(f"  Total parameters     : {_n_total:,}")
print(f"  Trainable parameters : {_n_train:,}")
print(f"  Attention heads      : {_n_heads}  (hidden_size {TRAINING_CONFIG['hidden_size']} ÷ head_size {TRAINING_CONFIG['attention_head_size']})")
print(f"  Output transformer   : {type(tft.output_transformer).__name__}")
print(f"  Loss                 : OrdinalQuantileLoss  (ordinal_weight=0.4, {len(TRAINING_CONFIG['quantiles'])} quantiles)")


Seed set to 42


Temporal Fusion Transformer built (v2 — larger + ordinal loss)
  Total parameters     : 502,125
  Trainable parameters : 502,125
  Attention heads      : 16  (hidden_size 64 ÷ head_size 4)
  Output transformer   : EncoderNormalizer
  Loss                 : OrdinalQuantileLoss  (ordinal_weight=0.4, 7 quantiles)


In [36]:
# ── Section O2: Save architecture summary ─────────────────────────────────────
_arch_lines = [
    "Model            : TemporalFusionTransformer (v2)",
    f"hidden_size      : {TRAINING_CONFIG['hidden_size']}",
    f"lstm_layers      : {TRAINING_CONFIG['lstm_layers']}",
    f"attention_head_size : {TRAINING_CONFIG['attention_head_size']}",
    f"attention_heads  : {_n_heads}",
    f"hidden_continuous_size : {TRAINING_CONFIG['hidden_continuous_size']}",
    f"dropout          : {TRAINING_CONFIG['dropout']}",
    f"loss             : OrdinalQuantileLoss(ordinal_weight=0.4) {TRAINING_CONFIG['quantiles']}",
    f"target_normalizer: EncoderNormalizer(transformation=None)",
    f"encoder_length   : {TRAINING_CONFIG['encoder_length']}",
    f"prediction_length: {TRAINING_CONFIG['prediction_length']}",
    f"total_parameters : {_n_total:,}",
    f"trainable_params : {_n_train:,}",
]
(ARTIFACT_DIR / "model_architecture.txt").write_text("\n".join(_arch_lines))

print("Architecture summary (v2):")
for ln in _arch_lines:
    print(f"  {ln}")


Architecture summary (v2):
  Model            : TemporalFusionTransformer (v2)
  hidden_size      : 64
  lstm_layers      : 2
  attention_head_size : 4
  attention_heads  : 16
  hidden_continuous_size : 32
  dropout          : 0.15
  loss             : OrdinalQuantileLoss(ordinal_weight=0.4) [0.02, 0.1, 0.25, 0.5, 0.75, 0.9, 0.98]
  target_normalizer: EncoderNormalizer(transformation=None)
  encoder_length   : 72
  prediction_length: 48
  total_parameters : 502,125
  trainable_params : 502,125


---
## Section P — Training Loop

Train the TFT with:
- **EarlyStopping** on `val_loss` (patience = 5 epochs)
- **ModelCheckpoint** — saves the single best epoch by val_loss
- **LearningRateMonitor** — logs LR each epoch
- **gradient clipping** at 0.1 (standard for TFT attention)

Training progress is logged by the default `TensorBoardLogger`.  
After `fit()` the best checkpoint is reloaded as `tft_best` for evaluation.


In [37]:
# ── Section P0: pandas / pytorch-forecasting compatibility patches ─────────────
#
# Patch 1 — pandas 3.x
#   pytorch-forecasting 1.x calls df.loc with 1-element tuple keys.
#   pandas 3.0 added strict=True to zip() in _multi_take(), raising ValueError
#   when len(tup) < len(_AXIS_ORDERS).  Pad tup with slice(None) to fix it.
#
# Patch 2 — pytorch-forecasting 1.x + lightning 2.6
#   TemporalFusionTransformer.on_epoch_end() tries to log attention weights
#   (interpretation) at the END of TRAINING epochs when log_interval > 0.
#   In lightning 2.6, the model is in eval mode (self.training=False) when
#   on_train_epoch_end fires, so the condition `not self.training` is True and
#   log_interpretation() is called with training-step outputs that do NOT
#   contain an 'interpretation' key → KeyError.
#   Fix: guard log_interpretation with a key-presence check so training outputs
#   that lack 'interpretation' are silently skipped.

import pandas as _pd_patch

# ── Patch 1: pandas _multi_take ──────────────────────────────────────────────
if int(_pd_patch.__version__.split(".")[0]) >= 3:
    from pandas.core.indexing import _LocIndexer as _LIp

    def _patched_multi_take(self, tup):
        ao = self.obj._AXIS_ORDERS
        if len(tup) < len(ao):
            tup = tup + (slice(None),) * (len(ao) - len(tup))
        d = {
            axis: self._get_listlike_indexer(key, axis)
            for key, axis in zip(tup, ao)
        }
        return self.obj._reindex_with_indexers(d, allow_dups=True)

    _LIp._multi_take = _patched_multi_take
    print(f"Patch 1  pandas {_pd_patch.__version__}: _multi_take fixed ✓")
else:
    print(f"Patch 1  pandas {_pd_patch.__version__}: no patch needed")

# ── Patch 2: TFT on_epoch_end interpretation KeyError ────────────────────────
try:
    from pytorch_forecasting.models.temporal_fusion_transformer._tft import (
        TemporalFusionTransformer as _TFT,
    )
    _orig_on_epoch_end = _TFT.on_epoch_end.__wrapped__ if hasattr(
        _TFT.on_epoch_end, "__wrapped__"
    ) else _TFT.on_epoch_end

    def _patched_on_epoch_end(self, outputs):
        # Skip interpretation logging when outputs lack the 'interpretation'
        # key (training step outputs in lightning 2.6+).
        if (
            self.log_interval > 0
            and not self.training
            and outputs
            and "interpretation" not in outputs[0]
        ):
            return
        _orig_on_epoch_end(self, outputs)

    _TFT.on_epoch_end = _patched_on_epoch_end
    print("Patch 2  TFT on_epoch_end: interpretation KeyError guard added ✓")
except Exception as _e:
    print(f"Patch 2  TFT on_epoch_end: could not patch ({_e})")


Patch 1  pandas 2.3.3: no patch needed
Patch 2  TFT on_epoch_end: interpretation KeyError guard added ✓


In [38]:
# ── Section P1: Configure callbacks and trainer ───────────────────────────────
_CKPT_DIR = ARTIFACT_DIR / "checkpoints"
_CKPT_DIR.mkdir(exist_ok=True)

# ── Live epoch-progress callback ──────────────────────────────────────────────
class EpochProgressCallback(pl.Callback):
    """Prints a training table row after each epoch — visible in Jupyter."""

    def on_train_start(self, trainer, pl_module):
        hdr = f"{'Epoch':>7} │ {'train_loss':>12} │ {'val_loss':>12} │ {'LR':>10}"
        sep = "─" * 7 + "─┼─" + "─" * 12 + "─┼─" + "─" * 12 + "─┼─" + "─" * 10
        print(hdr)
        print(sep)

    def on_validation_epoch_end(self, trainer, pl_module):
        if trainer.sanity_checking:
            return
        ep  = trainer.current_epoch + 1
        mx  = trainer.max_epochs
        m   = trainer.callback_metrics
        tr  = float(m.get("train_loss_epoch", m.get("train_loss", float("nan"))))
        vl  = float(m.get("val_loss",  float("nan")))
        lr_key = next((k for k in m if k.startswith("lr")), None)
        lr  = float(m[lr_key]) if lr_key else float("nan")
        print(f"  {ep:>4}/{mx:<4} │ {tr:>12.6f} │ {vl:>12.6f} │ {lr:>10.2e}",
              flush=True)

    def on_train_end(self, trainer, pl_module):
        print("─" * 55)
        print("Training complete.")

_callbacks = [
    EarlyStopping(
        monitor  = "val_loss",
        patience = TRAINING_CONFIG["early_stop_patience"],
        mode     = "min",
        verbose  = False,          # silenced — EpochProgressCallback handles output
    ),
    ModelCheckpoint(
        dirpath    = str(_CKPT_DIR),
        filename   = "tft-{epoch:02d}-{val_loss:.4f}",
        monitor    = "val_loss",
        save_top_k = 1,
        mode       = "min",
    ),
    LearningRateMonitor(logging_interval="epoch"),
    EpochProgressCallback(),
]

trainer = pl.Trainer(
    max_epochs             = TRAINING_CONFIG["max_epochs"],
    accelerator            = TRAINING_CONFIG["accelerator"],
    devices                = 1,
    gradient_clip_val      = TRAINING_CONFIG["gradient_clip_val"],
    callbacks              = _callbacks,
    enable_progress_bar    = True,
    enable_model_summary   = True,
    log_every_n_steps      = 10,
)

print(f"Trainer ready  — max_epochs={TRAINING_CONFIG['max_epochs']},  "
      f"early_stop_patience={TRAINING_CONFIG['early_stop_patience']},  "
      f"gradient_clip={TRAINING_CONFIG['gradient_clip_val']}")
print("Starting training…  (use lightning.pytorch Trainer for compatibility)")


GPU available: False, used: False
TPU available: False, using: 0 TPU cores
💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.


Trainer ready  — max_epochs=100,  early_stop_patience=10,  gradient_clip=0.5
Starting training…  (use lightning.pytorch Trainer for compatibility)


In [39]:
# ── Section P2: Run training, load best checkpoint ────────────────────────────
print(f"Train batches : {len(train_dl):>4}  "
      f"({len(train_dl.dataset):,} samples)")
print(f"Val   batches : {len(val_dl):>4}  "
      f"({len(val_dl.dataset):,} samples)")
print(f"Batch size    : {train_dl.batch_size}")
print()

_t0 = datetime.now()
trainer.fit(tft, train_dataloaders=train_dl, val_dataloaders=val_dl)
_elapsed = (datetime.now() - _t0).total_seconds()

print(f"\nTraining finished in {_elapsed / 60:.1f} min  "
      f"({trainer.current_epoch + 1} epochs run)")

_best_ckpt     = trainer.checkpoint_callback.best_model_path
_best_val_loss = float(trainer.checkpoint_callback.best_model_score)
print(f"Best checkpoint : {Path(_best_ckpt).name}")
print(f"Best val loss   : {_best_val_loss:.6f}")

# Reload best weights for evaluation
tft_best = TemporalFusionTransformer.load_from_checkpoint(_best_ckpt)
tft_best.eval()
print("Best model loaded.")

# Persist training history from TensorBoard CSVs
_log_dir  = Path(trainer.logger.log_dir)
_hist_csv = _log_dir / "metrics.csv"
if _hist_csv.exists():
    _history_df = pd.read_csv(_hist_csv)
else:
    _history_df = pd.DataFrame([
        {"metric": k, "value": float(v)}
        for k, v in trainer.logged_metrics.items()
    ])
_hist_path = ARTIFACT_DIR / "training_history.csv"
_history_df.to_csv(_hist_path, index=False)
print(f"Training history saved : {_hist_path.name}  ({len(_history_df)} rows)")


Train batches :  425  (27,206 samples)
Val   batches :   64  (8,157 samples)
Batch size    : 64



┏━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━┳━━━━━━━┳━━━━━━━┓
┃    ┃ Name                               ┃ Type                            ┃ Params ┃ Mode  ┃ FLOPs ┃
┡━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━╇━━━━━━━╇━━━━━━━┩
│ 0  │ loss                               │ OrdinalQuantileLoss             │      0 │ train │     0 │
│ 1  │ logging_metrics                    │ ModuleList                      │      0 │ train │     0 │
│ 2  │ input_embeddings                   │ MultiEmbedding                  │     33 │ train │     0 │
│ 3  │ prescalers                         │ ModuleDict                      │  1.5 K │ train │     0 │
│ 4  │ static_variable_selection          │ VariableSelectionNetwork        │ 21.2 K │ train │     0 │
│ 5  │ encoder_variable_selection         │ VariableSelectionNetwork        │  158 K │ train │     0 │
│ 6  │ decoder_variable_selection         │ VariableSelectionNetwork        │ 49.3 K │ train │     0 │
│ 7  │ static_context_variable_selection  │ GatedResidualNetwork            │ 16.8 K │ train │     0 │
│ 8  │ static_context_initial_hidden_lstm │ GatedResidualNetwork            │ 16.8 K │ train │     0 │
│ 9  │ static_context_initial_cell_lstm   │ GatedResidualNetwork            │ 16.8 K │ train │     0 │
│ 10 │ static_context_enrichment          │ GatedResidualNetwork            │ 16.8 K │ train │     0 │
│ 11 │ lstm_encoder                       │ LSTM                            │ 66.6 K │ train │     0 │
│ 12 │ lstm_decoder                       │ LSTM                            │ 66.6 K │ train │     0 │
│ 13 │ post_lstm_gate_encoder             │ GatedLinearUnit                 │  8.3 K │ train │     0 │
│ 14 │ post_lstm_add_norm_encoder         │ AddNorm                         │    128 │ train │     0 │
│ 15 │ static_enrichment                  │ GatedResidualNetwork            │ 20.9 K │ train │     0 │
│ 16 │ multihead_attn                     │ InterpretableMultiHeadAttention │ 10.4 K │ train │     0 │
│ 17 │ post_attn_gate_norm                │ GateAddNorm                     │  8.4 K │ train │     0 │
│ 18 │ pos_wise_ff                        │ GatedResidualNetwork            │ 16.8 K │ train │     0 │
│ 19 │ pre_output_gate_norm               │ GateAddNorm                     │  8.4 K │ train │     0 │
│ 20 │ output_layer                       │ Linear                          │    455 │ train │     0 │
└────┴────────────────────────────────────┴─────────────────────────────────┴────────┴───────┴───────┘

Trainable params: 502 K                                                                                            
Non-trainable params: 0                                                                                            
Total params: 502 K                                                                                                
Total estimated model params size (MB): 2                                                                          
Modules in train mode: 629                                                                                         
Modules in eval mode: 0                                                                                            
Total FLOPs: 0

Epoch │   train_loss │     val_loss │         LR

────────┼──────────────┼──────────────┼───────────

...

91/100  │     0.101180 │     0.100407 │   3.13e-05

92/100  │     0.101189 │     0.100409 │   3.13e-05

───────────────────────────────────────────────────────

Training complete.


Training finished in 2383.3 min  (93 epochs run)
Best checkpoint : tft-epoch=81-val_loss=0.1004.ckpt
Best val loss   : 0.100402
Best model loaded.
Training history saved : training_history.csv  (7 rows)


---
## Section Q — Validation Metrics

Generate predictions on validation and test sets, then compute:

**Classification** (stage index rounded to int):
accuracy, balanced accuracy, precision (macro), recall (macro),
F1 (macro + weighted), ordinal distance, confusion matrix

**Regression** (continuous stage-index predictions):
MAE, RMSE, R², MAPE, median absolute error, explained variance

**Slicing**:
- All 48 decoder steps (aggregate)
- 24 h horizon — decoder step 23 (t + 24 from encoder end)
- 48 h horizon — decoder step 47 (t + 48 from encoder end)
- Per-stage breakdown
- Per-cycle breakdown


In [61]:
# ── Section Q1: Collect predictions and ground truth ─────────────────────────
#
# Prediction de-normalisation:
#   model(x) → out.prediction  shape (B, pred_len, n_quantiles)  [normalised]
#   model.transform_output(out.prediction, x["target_scale"])     [original scale]
#
# Ground truth:
#   x["decoder_time_idx"]  shape (B, pred_len)  gives the per-step time_idx.
#   Combined with x["groups"][:, 0] (cycle_id) this maps to val_df / test_df
#   whose target columns are in original [0, 5] stage-index scale.
#
STAGE_NAMES   = ["Germination", "Seedling", "Vegetative",
                 "Flowering",   "Fruiting",  "Harvest Ready"]
STAGE_INDICES = list(range(6))
H24 = 23   # decoder step index for 24 h ahead  (0-based, hourly data)
H48 = 47   # decoder step index for 48 h ahead

def _collect_preds(model, dataloader, gt_lookup):
    """
    Iterate *dataloader*, run inference, collect median predictions
    (de-normalised to original scale) and ground truth from *gt_lookup*.

    Returns
    -------
    preds  : ndarray (n_seqs, 48) — median quantile, original scale
    truths : ndarray (n_seqs, 48) — from df lookup, original scale (NaN at edges)
    cids   : ndarray (n_seqs,)    — cycle_id
    """
    model.eval()
    all_p, all_t, all_c = [], [], []

    with torch.no_grad():
        for x, _y in dataloader:
            out      = model(x)                                   # forward pass
            pred_dn  = model.transform_output(                    # de-normalise
                out.prediction, x["target_scale"]
            )                                                     # (B, 48, 7)
            pred_med = pred_dn[..., MEDIAN_IDX].cpu().numpy()    # (B, 48)

            cids_b  = x["groups"][:, 0].cpu().numpy().astype(int)      # (B,)
            tidxs_b = x["decoder_time_idx"].cpu().numpy().astype(int)  # (B, 48)
            B = cids_b.shape[0]

            gt_b = np.full((B, 48), np.nan, dtype=np.float32)
            for b in range(B):
                for k in range(48):
                    try:
                        gt_b[b, k] = gt_lookup.loc[(cids_b[b], tidxs_b[b, k])]
                    except KeyError:
                        pass   # boundary row not in split — leave NaN

            all_p.append(pred_med)
            all_t.append(gt_b)
            all_c.append(cids_b)

    return (np.concatenate(all_p, 0),
            np.concatenate(all_t, 0),
            np.concatenate(all_c, 0))

print("Collecting validation predictions…")
val_preds,  val_truths,  val_cids  = _collect_preds(tft_best, val_dl,  _val_gt_lookup)
print("Collecting test predictions…")
test_preds, test_truths, test_cids = _collect_preds(tft_best, test_dl, _test_gt_lookup)

print(f"\nVal  predictions : {val_preds.shape}    "
      f"GT non-NaN : {np.isfinite(val_truths).sum():,}/{val_truths.size}")
print(f"Test predictions : {test_preds.shape}     "
      f"GT non-NaN : {np.isfinite(test_truths).sum():,}/{test_truths.size}")
print(f"Val  pred range  : [{val_preds.min():.3f}, {val_preds.max():.3f}]")



Val  predictions : (8157, 48)    GT non-NaN : 130,128/391536
Test predictions : (3, 48)     GT non-NaN : 48/144
Val  pred range  : [0.000, 7.067]


In [62]:
# ── Section Q2: Classification and regression metric helpers ──────────────────
def _to_cls(arr: np.ndarray) -> np.ndarray:
    """Clip + round continuous stage-index predictions to integer [0, 5]."""
    return np.clip(np.round(arr), 0, 5).astype(int)

def _cls_metrics(y_true_f, y_pred_f, prefix=""):
    """Classification metrics on flat finite elements."""
    flat_t = y_true_f.ravel()
    flat_p = y_pred_f.ravel()
    fin    = np.isfinite(flat_t) & np.isfinite(flat_p)
    y_t, y_p = _to_cls(flat_t[fin]), _to_cls(flat_p[fin])
    labs       = sorted(set(y_t.tolist()) | set(y_p.tolist()))
    return {
        f"{prefix}accuracy"         : float(accuracy_score(y_t, y_p)),
        f"{prefix}balanced_accuracy": float(balanced_accuracy_score(y_t, y_p)),
        f"{prefix}precision_macro"  : float(precision_score(y_t, y_p, average="macro",    zero_division=0, labels=labs)),
        f"{prefix}recall_macro"     : float(recall_score(y_t,    y_p, average="macro",    zero_division=0, labels=labs)),
        f"{prefix}f1_macro"         : float(f1_score(y_t,        y_p, average="macro",    zero_division=0, labels=labs)),
        f"{prefix}f1_weighted"      : float(f1_score(y_t,        y_p, average="weighted", zero_division=0)),
        f"{prefix}ordinal_distance" : float(np.mean(np.abs(y_t.astype(float) - y_p.astype(float)))),
    }

def _reg_metrics(y_true_f, y_pred_f, prefix=""):
    """Regression metrics on flat finite elements."""
    y_t, y_p = y_true_f.ravel(), y_pred_f.ravel()
    fin = np.isfinite(y_t) & np.isfinite(y_p)
    y_t, y_p = y_t[fin], y_p[fin]
    return {
        f"{prefix}mae"              : float(mean_absolute_error(y_t, y_p)),
        f"{prefix}rmse"             : float(np.sqrt(mean_squared_error(y_t, y_p))),
        f"{prefix}r2"               : float(r2_score(y_t, y_p)),
        f"{prefix}mape"             : float(mean_absolute_percentage_error(y_t + 1e-8, y_p + 1e-8)),
        f"{prefix}median_abs_error" : float(median_absolute_error(y_t, y_p)),
        f"{prefix}explained_variance": float(explained_variance_score(y_t, y_p)),
    }
print("Metric helpers defined.")


Metric helpers defined.


In [63]:
# ── Section Q3: Compute aggregate, horizon, per-stage, per-cycle metrics ──────

# --- Aggregate (all decoder steps) ---
vm_all_cls = _cls_metrics(val_truths, val_preds, "val_all_")
vm_all_reg = _reg_metrics(val_truths, val_preds, "val_all_")

# --- 24 h horizon (decoder step H24=23) ---
vm_24h_cls = _cls_metrics(val_truths[:, H24:H24+1], val_preds[:, H24:H24+1], "val_24h_")
vm_24h_reg = _reg_metrics(val_truths[:, H24:H24+1], val_preds[:, H24:H24+1], "val_24h_")

# --- 48 h horizon (decoder step H48=47) ---
vm_48h_cls = _cls_metrics(val_truths[:, H48:H48+1], val_preds[:, H48:H48+1], "val_48h_")
vm_48h_reg = _reg_metrics(val_truths[:, H48:H48+1], val_preds[:, H48:H48+1], "val_48h_")

# --- Test set ---
tm_all_cls = _cls_metrics(test_truths, test_preds, "test_all_")
tm_all_reg = _reg_metrics(test_truths, test_preds, "test_all_")

# --- Confusion matrices ---
cm_24h = confusion_matrix(
    _to_cls(val_truths[:, H24]), _to_cls(val_preds[:, H24]), labels=STAGE_INDICES)
cm_48h = confusion_matrix(
    _to_cls(val_truths[:, H48]), _to_cls(val_preds[:, H48]), labels=STAGE_INDICES)
cm_24h_df = pd.DataFrame(cm_24h, index=STAGE_NAMES, columns=STAGE_NAMES)
cm_48h_df = pd.DataFrame(cm_48h, index=STAGE_NAMES, columns=STAGE_NAMES)

# --- Per-stage metrics (val set) ---
_vt_flat = val_truths.ravel()
_vp_flat = val_preds.ravel()
_fin_m   = np.isfinite(_vt_flat)
_vt_cls  = _to_cls(_vt_flat[_fin_m])
_vp_cls  = _to_cls(_vp_flat[_fin_m])
_vt_raw  = _vt_flat[_fin_m]
_vp_raw  = _vp_flat[_fin_m]

per_stage_records = []
for _si, _sn in zip(STAGE_INDICES, STAGE_NAMES):
    _m = _vt_cls == _si
    if _m.sum() < 5:
        continue
    per_stage_records.append({
        "stage_index": _si, "stage_name": _sn,
        "n_samples"  : int(_m.sum()),
        "precision"  : round(precision_score(_vt_cls[_m], _vp_cls[_m], labels=[_si], average="micro", zero_division=0), 4),
        "recall"     : round(recall_score(   _vt_cls[_m], _vp_cls[_m], labels=[_si], average="micro", zero_division=0), 4),
        "f1"         : round(f1_score(       _vt_cls[_m], _vp_cls[_m], labels=[_si], average="micro", zero_division=0), 4),
        "mae"        : round(mean_absolute_error(_vt_raw[_m], _vp_raw[_m]), 4),
    })
per_stage_df = pd.DataFrame(per_stage_records)

# --- Per-cycle metrics (val set) ---
per_cycle_records = []
for _cid in np.unique(val_cids):
    _m = val_cids == _cid
    if _m.sum() < 3:
        continue
    _ct, _cp = val_truths[_m], val_preds[_m]
    _mf = np.isfinite(_ct.ravel())
    if _mf.sum() == 0:
        continue
    per_cycle_records.append({
        "cycle_id"        : int(_cid),
        "n_sequences"     : int(_m.sum()),
        "mae"             : round(mean_absolute_error(_ct.ravel()[_mf], _cp.ravel()[_mf]), 4),
        "rmse"            : round(float(np.sqrt(mean_squared_error(_ct.ravel()[_mf], _cp.ravel()[_mf]))), 4),
        "accuracy"        : round(accuracy_score(_to_cls(_ct.ravel()[_mf]), _to_cls(_cp.ravel()[_mf])), 4),
        "ordinal_distance": round(float(np.mean(np.abs(_ct.ravel()[_mf] - _cp.ravel()[_mf]))), 4),
    })
per_cycle_df = pd.DataFrame(per_cycle_records)

# --- Horizon step-by-step metrics ---
horizon_records = []
for _step in range(0, 48, 4):
    _m_t = val_truths[:, _step]
    _m_p = val_preds[:, _step]
    _fin  = np.isfinite(_m_t)
    if _fin.sum() == 0:
        continue
    horizon_records.append({
        "decoder_step" : _step,
        "horizon_hours": _step + 1,
        **_cls_metrics(_m_t[_fin][:, None], _m_p[_fin][:, None]),
        **_reg_metrics(_m_t[_fin][:, None], _m_p[_fin][:, None]),
    })
horizon_df = pd.DataFrame(horizon_records)

print(f"Metrics computed\n")
print(f"  [Val — all horizons]  acc={vm_all_cls['val_all_accuracy']:.4f}  "
      f"f1={vm_all_cls['val_all_f1_macro']:.4f}  "
      f"mae={vm_all_reg['val_all_mae']:.4f}  "
      f"r2={vm_all_reg['val_all_r2']:.4f}")
print(f"  [Val — 24 h horizon]  acc={vm_24h_cls['val_24h_accuracy']:.4f}  "
      f"mae={vm_24h_reg['val_24h_mae']:.4f}")
print(f"  [Val — 48 h horizon]  acc={vm_48h_cls['val_48h_accuracy']:.4f}  "
      f"mae={vm_48h_reg['val_48h_mae']:.4f}")

print(f"\nPer-stage breakdown (val):")
print(per_stage_df[["stage_name", "n_samples", "f1", "mae"]].to_string(index=False))
print(f"\nPer-cycle breakdown (val):")
print(per_cycle_df.to_string(index=False))


Metrics computed

  [Val — all horizons]  acc=0.8372  f1=0.8291  mae=0.2333  r2=0.8407
  [Val — 24 h horizon]  acc=0.8410  mae=0.2295
  [Val — 48 h horizon]  acc=0.8322  mae=0.2384

Per-stage breakdown (val):
   stage_name  n_samples     f1    mae
  Germination      15528 1.0000 0.0000
     Seedling      29952 0.9292 0.1443
   Vegetative      13824 0.8799 0.2099
    Flowering      23040 0.8094 0.4830
     Fruiting      36864 0.8969 0.3257
Harvest Ready      10920 1.0000 0.0000

Per-cycle breakdown (val):
 cycle_id  n_sequences    mae   rmse  accuracy  ordinal_distance
       10         2711 0.2333 0.6306    0.8372            0.2333


In [64]:
# ── Section Q4: Save all metric artefacts ─────────────────────────────────────
def _round_floats(d):
    return {k: round(v, 6) if isinstance(v, float) else v for k, v in d.items()}

validation_metrics = {
    "run_id"         : RUN_ID,
    "best_val_loss"  : _best_val_loss,
    "training_epochs": trainer.current_epoch + 1,
    **vm_all_cls, **vm_all_reg,
    **vm_24h_cls, **vm_24h_reg,
    **vm_48h_cls, **vm_48h_reg,
}
test_metrics = {
    "run_id"       : RUN_ID,
    "n_test_seqs"  : int(test_preds.shape[0]),
    **tm_all_cls, **tm_all_reg,
}

(ARTIFACT_DIR / "validation_metrics.json").write_text(
    json.dumps(_round_floats(validation_metrics), indent=2))
(ARTIFACT_DIR / "test_metrics.json").write_text(
    json.dumps(_round_floats(test_metrics), indent=2))

cm_24h_df.to_csv(ARTIFACT_DIR / "confusion_matrix_24h.csv")
cm_48h_df.to_csv(ARTIFACT_DIR / "confusion_matrix_48h.csv")

_reg_rows = [
    {"split": "val_all",  **{k.replace("val_all_",  ""): v for k, v in vm_all_reg.items()}},
    {"split": "val_24h",  **{k.replace("val_24h_",  ""): v for k, v in vm_24h_reg.items()}},
    {"split": "val_48h",  **{k.replace("val_48h_",  ""): v for k, v in vm_48h_reg.items()}},
    {"split": "test_all", **{k.replace("test_all_", ""): v for k, v in tm_all_reg.items()}},
]
pd.DataFrame(_reg_rows).to_csv(ARTIFACT_DIR / "regression_metrics.csv", index=False)
per_stage_df.to_csv(ARTIFACT_DIR / "per_stage_metrics.csv",  index=False)
per_cycle_df.to_csv(ARTIFACT_DIR / "per_cycle_metrics.csv",  index=False)
horizon_df.to_csv(  ARTIFACT_DIR / "horizon_metrics.csv",    index=False)

print("Metric artefacts saved:")
for _name in ["validation_metrics.json", "test_metrics.json",
              "confusion_matrix_24h.csv", "confusion_matrix_48h.csv",
              "regression_metrics.csv",   "per_stage_metrics.csv",
              "per_cycle_metrics.csv",    "horizon_metrics.csv"]:
    _p = ARTIFACT_DIR / _name
    print(f"  {_name:<35}  {_p.stat().st_size / 1e3:6.1f} KB")


Metric artefacts saved:
  validation_metrics.json                 1.5 KB
  test_metrics.json                       0.5 KB
  confusion_matrix_24h.csv                0.2 KB
  confusion_matrix_48h.csv                0.2 KB
  regression_metrics.csv                  0.5 KB
  per_stage_metrics.csv                   0.3 KB
  per_cycle_metrics.csv                   0.1 KB
  horizon_metrics.csv                     3.3 KB


---
## Section R — Model Saving

Save the trained model and print a final metrics summary.

Saved files:
- `src/agritwin_gh/models/growth_progression_<run_id>.pt` — state dict for inference
- `src/agritwin_gh/models/growth_progression_<run_id>.ckpt` — full Lightning checkpoint (weights + hparams, for fine-tuning or continued training)
- `growth_progression_<run_id>/training_history.csv` — per-epoch metrics log


In [65]:
# ── Section R1: Save model state dict and Lightning checkpoint ────────────────
_model_pt   = MODEL_OUT_DIR / f"growth_progression_{RUN_ID}.pt"
_model_ckpt = MODEL_OUT_DIR / f"growth_progression_{RUN_ID}.ckpt"

torch.save(tft_best.state_dict(), _model_pt)
print(f"State dict saved  : {_model_pt.name}  ({_model_pt.stat().st_size / 1e6:.1f} MB)")

if _best_ckpt and Path(_best_ckpt).exists():
    shutil.copy2(_best_ckpt, _model_ckpt)
    print(f"Checkpoint saved  : {_model_ckpt.name}  ({_model_ckpt.stat().st_size / 1e6:.1f} MB)")

# training_history.csv was already written in Section P2
_hist = ARTIFACT_DIR / "training_history.csv"
print(f"Training history  : {_hist.name}  ({_hist.stat().st_size / 1e3:.1f} KB)")


State dict saved  : growth_progression_growth_progression_20260308_202542.pt  (2.3 MB)
Checkpoint saved  : growth_progression_growth_progression_20260308_202542.ckpt  (6.7 MB)
Training history  : training_history.csv  (0.2 KB)


In [66]:
# ── Section R2: Final metrics summary ─────────────────────────────────────────
_sep = "=" * 65
print(_sep)
print("  PHASE 3 — BEST METRICS SUMMARY")
print(_sep)
print(f"  Run ID                    : {RUN_ID}")
print(f"  Best val loss             : {_best_val_loss:.6f}")
print(f"  Training epochs           : {trainer.current_epoch + 1}")
print()
print("  Validation — all horizons:")
print(f"    Accuracy                : {vm_all_cls['val_all_accuracy']:.4f}")
print(f"    Balanced Accuracy       : {vm_all_cls['val_all_balanced_accuracy']:.4f}")
print(f"    Precision (macro)       : {vm_all_cls['val_all_precision_macro']:.4f}")
print(f"    Recall    (macro)       : {vm_all_cls['val_all_recall_macro']:.4f}")
print(f"    F1        (macro)       : {vm_all_cls['val_all_f1_macro']:.4f}")
print(f"    F1        (weighted)    : {vm_all_cls['val_all_f1_weighted']:.4f}")
print(f"    Ordinal distance        : {vm_all_cls['val_all_ordinal_distance']:.4f}")
print(f"    MAE                     : {vm_all_reg['val_all_mae']:.4f}")
print(f"    RMSE                    : {vm_all_reg['val_all_rmse']:.4f}")
print(f"    R²                      : {vm_all_reg['val_all_r2']:.4f}")
print(f"    MAPE                    : {vm_all_reg['val_all_mape']:.4f}")
print(f"    Median Abs Error        : {vm_all_reg['val_all_median_abs_error']:.4f}")
print(f"    Explained Variance      : {vm_all_reg['val_all_explained_variance']:.4f}")
print()
print(f"  Validation — 24 h horizon (decoder step {H24}):")
print(f"    Accuracy                : {vm_24h_cls['val_24h_accuracy']:.4f}")
print(f"    F1 (macro)              : {vm_24h_cls['val_24h_f1_macro']:.4f}")
print(f"    Ordinal distance        : {vm_24h_cls['val_24h_ordinal_distance']:.4f}")
print(f"    MAE                     : {vm_24h_reg['val_24h_mae']:.4f}")
print(f"    RMSE                    : {vm_24h_reg['val_24h_rmse']:.4f}")
print()
print(f"  Validation — 48 h horizon (decoder step {H48}):")
print(f"    Accuracy                : {vm_48h_cls['val_48h_accuracy']:.4f}")
print(f"    F1 (macro)              : {vm_48h_cls['val_48h_f1_macro']:.4f}")
print(f"    Ordinal distance        : {vm_48h_cls['val_48h_ordinal_distance']:.4f}")
print(f"    MAE                     : {vm_48h_reg['val_48h_mae']:.4f}")
print(f"    RMSE                    : {vm_48h_reg['val_48h_rmse']:.4f}")
print()
print(f"  Test — all horizons (3 cycles, n={test_preds.shape[0]} sequences):")
print(f"    Accuracy                : {tm_all_cls['test_all_accuracy']:.4f}")
print(f"    F1 (macro)              : {tm_all_cls['test_all_f1_macro']:.4f}")
print(f"    Ordinal distance        : {tm_all_cls['test_all_ordinal_distance']:.4f}")
print(f"    MAE                     : {tm_all_reg['test_all_mae']:.4f}")
print()
print(f"  Confusion matrix — 24 h horizon:")
print(cm_24h_df.to_string())
print()
print(f"  Confusion matrix — 48 h horizon:")
print(cm_48h_df.to_string())
print()
print(f"  Horizon step profile (sample):")
print(horizon_df[["horizon_hours", "accuracy", "mae", "r2"]].to_string(index=False))
print()
print(_sep)
print(f"  Model  : src/agritwin_gh/models/growth_progression_{RUN_ID}.pt")
print(f"  Ckpt   : src/agritwin_gh/models/growth_progression_{RUN_ID}.ckpt")
print(f"  Metrics: {ARTIFACT_DIR.name}/")
print(_sep)


  PHASE 3 — BEST METRICS SUMMARY
  Run ID                    : growth_progression_20260308_202542
  Best val loss             : 0.100402
  Training epochs           : 93

  Validation — all horizons:
    Accuracy                : 0.8372
    Balanced Accuracy       : 0.8577
    Precision (macro)       : 0.8385
    Recall    (macro)       : 0.8577
    F1        (macro)       : 0.8291
    F1        (weighted)    : 0.8417
    Ordinal distance        : 0.1856
    MAE                     : 0.2333
    RMSE                    : 0.6306
    R²                      : 0.8407
    MAPE                    : 0.0959
    Median Abs Error        : 0.0000
    Explained Variance      : 0.8490

  Validation — 24 h horizon (decoder step 23):
    Accuracy                : 0.8410
    F1 (macro)              : 0.8346
    Ordinal distance        : 0.1815
    MAE                     : 0.2295
    RMSE                    : 0.6271

  Validation — 48 h horizon (decoder step 47):
    Accuracy                : 0.8322
 

---
## Section S — Load Model for Inference

Load the trained TFT model, Phase 2 scaler, and feature configuration from the
run artifacts. Three loading paths are attempted in priority order:

1. **Lightning checkpoint** `growth_progression_<run_id>.ckpt` in `models/`
2. **Artifact checkpoint** `checkpoints/*.ckpt` written by `trainer.fit`
3. **State-dict** `.pt` — reconstructs architecture from `training_dataset`

> Run Sections N–R first so that model files exist.

In [67]:
# ── Section S1: Imports and artifact configuration ────────────────
import json, pickle, warnings
from pathlib import Path

import numpy as np
import pandas as pd
import torch
from pytorch_forecasting import TemporalFusionTransformer, TimeSeriesDataSet
from pytorch_forecasting.metrics import QuantileLoss

warnings.filterwarnings('ignore')

REPO_ROOT     = Path().resolve().parent
ARTIFACT_BASE = REPO_ROOT / 'src' / 'agritwin_gh' / 'models' / 'artifacts'
MODEL_OUT_DIR = REPO_ROOT / 'src' / 'agritwin_gh' / 'models'

# Resolve most-recent artifact run directory
ARTIFACT_DIR = sorted(
    [d for d in ARTIFACT_BASE.iterdir()
     if d.is_dir() and d.name.startswith('growth_progression_')],
    key=lambda p: p.stat().st_mtime,
)[-1]
RUN_ID = ARTIFACT_DIR.name
print(f'Artifact dir      : {RUN_ID}')

# Feature / target configuration
feature_roles   = json.loads((ARTIFACT_DIR / 'feature_roles.json').read_text())
seq_meta        = json.loads((ARTIFACT_DIR / 'sequence_metadata.json').read_text())
target_def      = json.loads((ARTIFACT_DIR / 'target_definition.json').read_text())
training_config = json.loads((ARTIFACT_DIR / 'training_config.json').read_text())

# RobustScaler fitted in Phase 2
with open(ARTIFACT_DIR / 'robust_scaler.pkl', 'rb') as _fh:
    scaler = pickle.load(_fh)
scaled_features = seq_meta['scaled_features']
print(f'Scaler            : {type(scaler).__name__}  ({len(scaled_features)} features)')

# TimeSeriesDataSet objects (required for from_dataset inference path)
print('Loading TimeSeriesDataSet objects…')
with open(ARTIFACT_DIR / 'tft_training_dataset.pkl', 'rb') as _fh:
    _saved = pickle.load(_fh)
training_dataset   = _saved['training']
validation_dataset = _saved['validation']
test_dataset       = _saved['test']
train_df           = _saved['train_df']
val_df             = _saved['val_df']
test_df            = _saved['test_df']
print('TimeSeriesDataSets loaded.')

# Shared constants
QUANTILES  = training_config['quantiles']          # [0.02, 0.1, 0.25, 0.5, 0.75, 0.9, 0.98]
MEDIAN_IDX = len(QUANTILES) // 2                   # index 3  -> q=0.5
H24 = 23                                           # decoder step for 24 h horizon
H48 = 47                                           # decoder step for 48 h horizon
ENC_LEN  = feature_roles['encoder_length']         # 72
PRED_LEN = feature_roles['prediction_horizon']     # 48

# Stage label mapping (preserves training ordinal order 0-5)
_stage_map  = target_def['stage_index_map']        # {name: idx}
STAGE_NAMES = [''] * 6
for _nm, _idx in _stage_map.items():
    STAGE_NAMES[_idx] = _nm
_NICE = {
    'seedling'             : 'Seedling',
    'early_vegetative'     : 'Early Vegetative',
    'flowering_initiation' : 'Flowering Init.',
    'flowering'            : 'Flowering',
    'unripe'               : 'Unripe',
    'ripe'                 : 'Ripe',
}
STAGE_LABELS = [_NICE.get(s, s.title()) for s in STAGE_NAMES]

print(f'Stage labels      : {STAGE_LABELS}')
print(f'Quantiles         : {QUANTILES}  (median idx={MEDIAN_IDX})')
print(f'Encoder / decoder : {ENC_LEN} / {PRED_LEN} steps  (H24=step {H24}, H48=step {H48})')


Artifact dir      : growth_progression_20260308_202542
Scaler            : RobustScaler  (19 features)
Loading TimeSeriesDataSet objects…
TimeSeriesDataSets loaded.
Stage labels      : ['Seedling', 'Early Vegetative', 'Flowering Init.', 'Flowering', 'Unripe', 'Ripe']
Quantiles         : [0.02, 0.1, 0.25, 0.5, 0.75, 0.9, 0.98]  (median idx=3)
Encoder / decoder : 72 / 48 steps  (H24=step 23, H48=step 47)


In [68]:
# ── Section S2: Load TFT model ───────────────────────────────
#
# Loading priority:
#   1. MODEL_OUT_DIR / growth_progression_<run_id>.ckpt  (Section R1 output)
#   2. ARTIFACT_DIR  / checkpoints/*.ckpt                (trainer callback)
#   3. MODEL_OUT_DIR / growth_progression_<run_id>.pt    (state-dict fallback)
#
_ckpt_main = MODEL_OUT_DIR / f'growth_progression_{RUN_ID}.ckpt'
_ckpt_arts = (sorted((ARTIFACT_DIR / 'checkpoints').glob('*.ckpt'))
              if (ARTIFACT_DIR / 'checkpoints').exists() else [])
_pt_main   = MODEL_OUT_DIR / f'growth_progression_{RUN_ID}.pt'

if _ckpt_main.exists():
    tft_infer = TemporalFusionTransformer.load_from_checkpoint(str(_ckpt_main))
    _src = f'Lightning checkpoint  : {_ckpt_main.name}'
elif _ckpt_arts:
    _best_art = max(_ckpt_arts, key=lambda p: p.stat().st_mtime)
    tft_infer = TemporalFusionTransformer.load_from_checkpoint(str(_best_art))
    _src = f'Artifact checkpoint   : {_best_art.name}'
elif _pt_main.exists():
    cfg = training_config
    tft_infer = TemporalFusionTransformer.from_dataset(
        training_dataset,
        learning_rate          = cfg['learning_rate'],
        hidden_size            = cfg['hidden_size'],
        lstm_layers            = cfg['lstm_layers'],
        attention_head_size    = cfg['attention_head_size'],
        dropout                = cfg['dropout'],
        hidden_continuous_size = cfg['hidden_continuous_size'],
        loss                   = QuantileLoss(quantiles=cfg['quantiles']),
    )
    tft_infer.load_state_dict(
        torch.load(_pt_main, map_location='cpu', weights_only=True))
    _src = f'State dict            : {_pt_main.name}'
else:
    raise FileNotFoundError(
        'No trained model found.\n'
        f'Expected one of:\n'
        f'  {_ckpt_main}\n'
        f'  {ARTIFACT_DIR / "checkpoints"}/*.ckpt\n'
        f'  {_pt_main}\n'
        'Run Sections N–R (trainer.fit) first.'
    )

tft_infer.eval()
_n_params = sum(p.numel() for p in tft_infer.parameters())
print('TFT model loaded')
print(f'  Source     : {_src}')
print(f'  Parameters : {_n_params:,}')
print(f'  Device     : {next(tft_infer.parameters()).device}')


TFT model loaded
  Source     : Lightning checkpoint  : growth_progression_growth_progression_20260308_202542.ckpt
  Parameters : 502,125
  Device     : cpu


---
## Section T — Inference Function

`predict_growth_stage(sequence_df)` accepts **72 rows** of observed data and
returns a structured prediction dict:

| Key | Type | Description |
|---|---|---|
| `stage_24h` / `stage_48h` | str | Predicted stage name at t+24 h / t+48 h |
| `stage_index_24h` / `stage_index_48h` | int 0–5 | Ordinal stage class |
| `stage_cont_24h` / `stage_cont_48h` | float | Continuous TFT prediction |
| `progress_24h` / `progress_48h` | float % | Within-stage completion (fractional part × 100) |
| `hours_to_next_stage` | float | Rough estimate of hours to next transition |
| `class_probs_24h` / `class_probs_48h` | ndarray (6,) | Soft stage probabilities |
| `quantile_preds_24h` / `quantile_preds_48h` | ndarray (7,) | Raw de-normalised quantiles |

**Pipeline**:
1. Extrapolate 48-step decoder window (known calendar / light features)
2. Apply Phase 2 `RobustScaler` to numerical columns
3. Build `TimeSeriesDataSet` via `from_dataset(predict=True)`
4. Run TFT forward pass → de-normalise with `model.transform_output()`
5. Derive stage class, progress, and Gaussian soft probabilities


In [69]:
# ── Section T1: DataFrame preprocessing helpers ─────────────────────
_KNOWN_REALS = feature_roles['time_varying_known_reals']
_UNK_REALS   = feature_roles['time_varying_unknown_reals']  # includes target col
_STATIC_CATS = feature_roles['static_categoricals']
_TGT_COL     = training_dataset.target                      # 'target_stage_index_24h'


def build_inference_df(sequence_df: pd.DataFrame) -> pd.DataFrame:
    # Build a TimeSeriesDataSet-compatible DataFrame for predict=True mode.
    #
    # sequence_df : >= ENC_LEN rows in original (unscaled) units.
    # Required columns: time_idx, cycle_id, cycle_origin_type, season_label,
    #   all time_varying_known_reals and time_varying_unknown_reals.
    #
    # Returns DataFrame of ENC_LEN + PRED_LEN rows.
    # Decoder rows carry forward the last encoder values for unknown reals;
    # the target column is set to 0 (ignored in predict mode).
    assert len(sequence_df) >= ENC_LEN, (
        f'Need >= {ENC_LEN} encoder rows, got {len(sequence_df)}')

    enc_df   = sequence_df.iloc[-ENC_LEN:].copy().reset_index(drop=True)
    last_row = enc_df.iloc[-1]
    last_t   = int(last_row['time_idx'])

    dec_rows = []
    for k in range(1, PRED_LEN + 1):
        row = {}
        # Static fields
        row['cycle_id']          = last_row['cycle_id']
        row['cycle_origin_type'] = last_row['cycle_origin_type']
        row['season_label']      = last_row['season_label']
        row['time_idx']          = last_t + k

        # Known reals: extrapolate cyclical calendar and light features
        fut_hour = int((last_row['hour'] + k) % 24)
        row['hour']              = fut_hour
        row['day_night_flag']    = 1 if 6 <= fut_hour < 20 else 0
        row['light_period_flag'] = 1 if 8 <= fut_hour < 18 else 0
        for _c in ['month', 'week_of_year', 'day_of_year']:
            row[_c] = last_row[_c]

        # Unknown reals: carry forward last observed values
        # (TFT only uses known reals in the decoder; these are not consumed)
        for _c in _UNK_REALS:
            row[_c] = float(last_row.get(_c, 0.0))

        # Target column required by TimeSeriesDataSet schema; ignored at inference
        row[_TGT_COL] = 0.0
        dec_rows.append(row)

    dec_df  = pd.DataFrame(dec_rows)
    full_df = pd.concat([enc_df, dec_df], ignore_index=True)
    if _TGT_COL not in full_df.columns:
        full_df[_TGT_COL] = 0.0
    return full_df


def scale_inference_df(df: pd.DataFrame) -> pd.DataFrame:
    # Apply Phase 2 RobustScaler to the numerical feature columns.
    df = df.copy()
    _cols = [c for c in scaled_features if c in df.columns]
    df[_cols] = scaler.transform(df[_cols].values)
    return df


print('build_inference_df() defined.')
print(f'  Encoder rows  : {ENC_LEN}  +  Decoder rows  : {PRED_LEN}  =  {ENC_LEN + PRED_LEN} total')
print(f'  Target column : {_TGT_COL}')
print(f'  Known reals   : {_KNOWN_REALS}')


build_inference_df() defined.
  Encoder rows  : 72  +  Decoder rows  : 48  =  120 total
  Target column : target_stage_index_24h
  Known reals   : ['hour', 'month', 'week_of_year', 'day_of_year', 'day_night_flag', 'light_period_flag']


In [70]:
# ── Section T2: predict_growth_stage — core inference function ───────────────

def predict_growth_stage(
    sequence_df      : pd.DataFrame,
    return_all_steps : bool = False,
) -> dict:
    # Run TFT inference on a recent observation window.
    #
    # Parameters
    # ----------
    # sequence_df : pd.DataFrame
    #     At least ENC_LEN (72) rows of raw (unscaled) observations.
    #     Must include all feature columns used during training.
    # return_all_steps : bool
    #     If True, adds 'all_step_preds' ndarray (48, 7) to result.
    #
    # Returns
    # -------
    # dict keys:
    #   stage_24h / stage_48h           str       predicted stage name
    #   stage_index_24h / 48h           int 0-5   ordinal class
    #   stage_cont_24h  / 48h           float     continuous prediction
    #   progress_24h    / 48h           float %   within-stage completion
    #   hours_to_next_stage             float     rough estimate
    #   class_probs_24h / 48h           ndarray(6) soft probabilities
    #   quantile_preds_24h / 48h        ndarray(7) de-normalised quantiles
    #   all_step_preds                  ndarray(48,7) optional

    # 1. Build encoder + dummy decoder DataFrame
    full_df  = build_inference_df(sequence_df)
    # 2. Scale numerical features with Phase 2 RobustScaler
    full_df  = scale_inference_df(full_df)

    # 3. Create inference-mode TimeSeriesDataSet
    infer_ds = TimeSeriesDataSet.from_dataset(
        training_dataset, full_df,
        predict=True, stop_randomization=True,
    )
    infer_dl = infer_ds.to_dataloader(train=False, batch_size=1, num_workers=0)

    # 4. Forward pass and quantile de-normalisation
    tft_infer.eval()
    with torch.no_grad():
        x, _y   = next(iter(infer_dl))
        out     = tft_infer(x)
        pred_dn = tft_infer.transform_output(out.prediction, x['target_scale'])
        # pred_dn : (1, 48, 7)  de-normalised quantile predictions

    preds = pred_dn[0].cpu().numpy()       # (48, 7)
    q50   = preds[:, MEDIAN_IDX]           # (48,)  median trace

    # 5. Horizon values
    p24 = float(q50[H24])
    p48 = float(q50[H48])
    q24 = preds[H24]                       # (7,) all quantiles at H24
    q48 = preds[H48]                       # (7,) all quantiles at H48

    si24 = int(np.clip(round(p24), 0, 5))
    si48 = int(np.clip(round(p48), 0, 5))

    # Within-stage progress: fractional part of the continuous prediction x 100
    prog24 = float(np.clip((p24 - np.floor(p24)) * 100, 0, 100))
    prog48 = float(np.clip((p48 - np.floor(p48)) * 100, 0, 100))

    # Hours to next stage: conservative estimate using q25 at decoder step 0
    p_q25_0  = max(float(preds[0, 2]), 0.0)           # q=0.25 at t+1
    frac_cur = float(p_q25_0 - np.floor(p_q25_0))    # fractional stage index
    hours_nxt = float((1.0 - frac_cur) * 24.0)       # hours to next integer stage

    # 6. Soft class probabilities via Gaussian kernel over quantile spread
    def _to_probs(q_arr: np.ndarray) -> np.ndarray:
        mu    = q_arr[MEDIAN_IDX]
        sigma = max((q_arr[-1] - q_arr[0]) / 4.0, 0.3)
        raw   = np.exp(-0.5 * ((np.arange(6.0) - mu) / sigma) ** 2)
        return (raw / raw.sum()).astype(np.float64)

    result = {
        'stage_24h'           : STAGE_LABELS[si24],
        'stage_48h'           : STAGE_LABELS[si48],
        'stage_index_24h'     : si24,
        'stage_index_48h'     : si48,
        'stage_cont_24h'      : round(p24, 4),
        'stage_cont_48h'      : round(p48, 4),
        'progress_24h'        : round(prog24, 2),
        'progress_48h'        : round(prog48, 2),
        'hours_to_next_stage' : round(hours_nxt, 1),
        'class_probs_24h'     : _to_probs(q24),
        'class_probs_48h'     : _to_probs(q48),
        'quantile_preds_24h'  : q24,
        'quantile_preds_48h'  : q48,
    }
    if return_all_steps:
        result['all_step_preds'] = preds
    return result


print('predict_growth_stage() defined.')
print()
print('Return keys:')
for _k in [
    'stage_24h', 'stage_48h',
    'stage_index_24h', 'stage_index_48h',
    'stage_cont_24h', 'stage_cont_48h',
    'progress_24h', 'progress_48h',
    'hours_to_next_stage',
    'class_probs_24h  [ndarray 6]',
    'class_probs_48h  [ndarray 6]',
    'quantile_preds_24h [ndarray 7]',
    'quantile_preds_48h [ndarray 7]',
    'all_step_preds    [ndarray 48x7]  (optional)',
]:
    print(f'  {_k}')


predict_growth_stage() defined.

Return keys:
  stage_24h
  stage_48h
  stage_index_24h
  stage_index_48h
  stage_cont_24h
  stage_cont_48h
  progress_24h
  progress_48h
  hours_to_next_stage
  class_probs_24h  [ndarray 6]
  class_probs_48h  [ndarray 6]
  quantile_preds_24h [ndarray 7]
  quantile_preds_48h [ndarray 7]
  all_step_preds    [ndarray 48x7]  (optional)


In [71]:
# ── Section T3: Demo inference on one validation sequence ────────────────
_DEMO_CID  = 10              # use val cycle 10 (a known cycle)
_demo_rows = val_df[val_df.cycle_id == _DEMO_CID].sort_values('time_idx')
_demo_enc  = _demo_rows.iloc[:ENC_LEN]    # first 72 rows as encoder window

print(f'Demo — cycle {_DEMO_CID}  '
      f'(time_idx {_demo_enc["time_idx"].min()}–{_demo_enc["time_idx"].max()})')
print(f'  Current stage (last encoder row) : '
      f'{_demo_enc.iloc[-1]["stage_name"]}  '
      f'(index={int(_demo_enc.iloc[-1]["stage_index"])})')

_true_24 = _demo_rows.iloc[ENC_LEN + H24] if len(_demo_rows) > ENC_LEN + H24 else None
_true_48 = _demo_rows.iloc[ENC_LEN + H48] if len(_demo_rows) > ENC_LEN + H48 else None
print(f'  Ground truth t+24h : '
      f'{_true_24["stage_name"] if _true_24 is not None else "n/a"}')
print(f'  Ground truth t+48h : '
      f'{_true_48["stage_name"] if _true_48 is not None else "n/a"}')
print()

_res = predict_growth_stage(_demo_enc, return_all_steps=True)

print('─' * 60)
print(f'  24h forecast : {_res["stage_24h"]}  '
      f'(idx={_res["stage_index_24h"]}, cont={_res["stage_cont_24h"]:.3f})')
print(f'  48h forecast : {_res["stage_48h"]}  '
      f'(idx={_res["stage_index_48h"]}, cont={_res["stage_cont_48h"]:.3f})')
print(f'  Progress 24h : {_res["progress_24h"]:.1f}%  |  '
      f'Progress 48h : {_res["progress_48h"]:.1f}%')
print(f'  Hours to next stage : {_res["hours_to_next_stage"]:.1f} h')
print()
print('  Class probabilities — 24h:')
for _i, (_lbl, _p) in enumerate(zip(STAGE_LABELS, _res['class_probs_24h'])):
    print(f'    [{_i}] {_lbl:<22}  {_p:.4f}  {chr(124) * int(_p * 30)}')
print()
print('  Class probabilities — 48h:')
for _i, (_lbl, _p) in enumerate(zip(STAGE_LABELS, _res['class_probs_48h'])):
    print(f'    [{_i}] {_lbl:<22}  {_p:.4f}  {chr(124) * int(_p * 30)}')
print()
print('  Quantile predictions — 24h:')
for _q, _v in zip(QUANTILES, _res['quantile_preds_24h']):
    print(f'    q={_q}  →  {_v:.4f}')
print()
print('  Median trace (every 4 decoder steps):')
_trace = _res['all_step_preds'][:, MEDIAN_IDX]
for _s in range(0, PRED_LEN, 4):
    _si = int(np.clip(round(_trace[_s]), 0, 5))
    print(f'    step {_s + 1:3d}h  →  {_trace[_s]:.3f}  ({STAGE_LABELS[_si]})')


Demo — cycle 10  (time_idx 0–71)
  Current stage (last encoder row) : seedling  (index=-1)
  Ground truth t+24h : seedling
  Ground truth t+48h : seedling

────────────────────────────────────────────────────────────
  24h forecast : Seedling  (idx=0, cont=0.000)
  48h forecast : Seedling  (idx=0, cont=0.000)
  Progress 24h : 0.0%  |  Progress 48h : 0.0%
  Hours to next stage : 24.0 h

  Class probabilities — 24h:
    [0] Seedling                0.9961  |||||||||||||||||||||||||||||
    [1] Early Vegetative        0.0039  
    [2] Flowering Init.         0.0000  
    [3] Flowering               0.0000  
    [4] Unripe                  0.0000  
    [5] Ripe                    0.0000  

  Class probabilities — 48h:
    [0] Seedling                0.9961  |||||||||||||||||||||||||||||
    [1] Early Vegetative        0.0039  
    [2] Flowering Init.         0.0000  
    [3] Flowering               0.0000  
    [4] Unripe                  0.0000  
    [5] Ripe                    0.0000  

 

---
## Section U — Batch Forecast Generation

Generate and save structured predictions for three scenarios:

| Cell | Scenario | Saved file |
|---|---|---|
| U1 | Full test set — all sequences, H24 + H48, via dataloader | `test_batch_predictions.csv` |
| U2 | Sliding window across one sample test cycle | `sample_cycle_sliding_predictions.csv` |
| U3 | Single inference on the most-recent observation window | `most_recent_inference.json` |
| U4 | Save all forecasts + print accuracy summary | `forecasts/` directory |


In [72]:
# ── Section U1: Batch predictions — full test set ────────────────────────
_NW   = training_config['num_workers']
_BS   = training_config['batch_size']
_test_dl = test_dataset.to_dataloader(train=False, batch_size=_BS * 2, num_workers=_NW)

# Ground-truth lookup: target columns are in original [0,5] scale (not scaler-transformed)
_gt_idx = test_df.set_index(['cycle_id', 'time_idx'])

tft_infer.eval()
batch_records = []

with torch.no_grad():
    for x, _y in _test_dl:
        _pred_dn  = tft_infer.transform_output(
            tft_infer(x).prediction, x['target_scale'])   # (B, 48, 7)
        _pred_med = _pred_dn[..., MEDIAN_IDX].cpu().numpy()    # (B, 48) q=0.5
        _pred_q25 = _pred_dn[..., 2].cpu().numpy()             # (B, 48) q=0.25
        _pred_q75 = _pred_dn[..., 4].cpu().numpy()             # (B, 48) q=0.75

        _cids  = x['groups'][:, 0].cpu().numpy().astype(int)
        _dtidx = x['decoder_time_idx'].cpu().numpy().astype(int)  # (B, PRED_LEN)
        _enc_end = _dtidx[:, 0] - 1                              # last encoder time_idx

        for _b in range(_cids.shape[0]):
            for _step, _horizon in [(H24, 'h24'), (H48, 'h48')]:
                _cid   = int(_cids[_b])
                _tidx  = int(_dtidx[_b, _step])
                _pm    = float(_pred_med[_b, _step])
                _pq25  = float(_pred_q25[_b, _step])
                _pq75  = float(_pred_q75[_b, _step])
                _si    = int(np.clip(round(_pm), 0, 5))
                _prog  = float(np.clip((_pm - np.floor(_pm)) * 100, 0, 100))

                # Ground truth (absent at cycle boundaries)
                _gt      = None
                _key     = (_cid, _tidx)
                if _key in _gt_idx.index:
                    _gt = _gt_idx.loc[_key]
                    if isinstance(_gt, pd.DataFrame):
                        _gt = _gt.iloc[0]   # multi-row safeguard

                batch_records.append({
                    'cycle_id'         : _cid,
                    'enc_end_time_idx' : int(_enc_end[_b]),
                    'horizon'          : _horizon,
                    'decoder_step'     : _step,
                    'decoder_time_idx' : _tidx,
                    'pred_stage_cont'  : round(_pm,   4),
                    'pred_stage_index' : _si,
                    'pred_stage_name'  : STAGE_LABELS[_si],
                    'pred_q25'         : round(_pq25, 4),
                    'pred_q75'         : round(_pq75, 4),
                    'pred_progress_pct': round(_prog, 2),
                    'gt_stage_index'   : (int(_gt['target_stage_index_24h'])
                                          if _gt is not None else np.nan),
                    'gt_progress_pct'  : (float(_gt['target_stage_progress_24h'])
                                          if _gt is not None else np.nan),
                    'gt_hours_to_next' : (float(_gt['target_hours_to_next_stage'])
                                          if _gt is not None else np.nan),
                })

batch_df = pd.DataFrame(batch_records)
print(f'Test batch predictions : {batch_df.shape[0]} rows  '
      f'(cycles={sorted(batch_df.cycle_id.unique())}, '
      f'horizons={sorted(batch_df.horizon.unique())})')
print()
_sub = batch_df.dropna(subset=['gt_stage_index'])
print('Sample (first 8 rows with ground truth):')
print(_sub[['cycle_id', 'enc_end_time_idx', 'horizon',
            'pred_stage_index', 'pred_stage_name',
            'pred_progress_pct', 'gt_stage_index']].head(8).to_string(index=False))


Test batch predictions : 6 rows  (cycles=[np.int64(10), np.int64(11), np.int64(12)], horizons=['h24', 'h48'])

Sample (first 8 rows with ground truth):
 cycle_id  enc_end_time_idx horizon  pred_stage_index pred_stage_name  pred_progress_pct  gt_stage_index
       12              2519     h24                 5            Ripe             0.0000          5.0000
       12              2519     h48                 5            Ripe             0.0000          5.0000


In [73]:
# ── Section U2: Sliding-window forecast for one sample test cycle ──────────
_SLIDE_CID    = sorted(test_df.cycle_id.unique())[0]    # first test cycle
_slide_rows   = (test_df[test_df.cycle_id == _SLIDE_CID]
                 .sort_values('time_idx').reset_index(drop=True))
_SLIDE_STRIDE = 12                                       # predict every 12 h
_max_start    = max(0, len(_slide_rows) - ENC_LEN - H48 - 1)

print(f'Sliding-window forecast — cycle {_SLIDE_CID}  '
      f'({len(_slide_rows)} rows, stride={_SLIDE_STRIDE} h)')

slide_records = []
for _start in range(0, _max_start + 1, _SLIDE_STRIDE):
    _win = _slide_rows.iloc[_start : _start + ENC_LEN]
    if len(_win) < ENC_LEN:
        break
    try:
        _r = predict_growth_stage(_win)
    except Exception:
        continue

    _end      = _start + ENC_LEN
    _gt24_row = _slide_rows.iloc[_end + H24] if _end + H24 < len(_slide_rows) else None
    _gt48_row = _slide_rows.iloc[_end + H48] if _end + H48 < len(_slide_rows) else None

    slide_records.append({
        'window_start'     : _start,
        'enc_end_time_idx' : int(_win['time_idx'].iloc[-1]),
        'current_stage'    : _win['stage_name'].iloc[-1],
        'pred_stage_24h'   : _r['stage_24h'],
        'pred_stage_48h'   : _r['stage_48h'],
        'pred_index_24h'   : _r['stage_index_24h'],
        'pred_index_48h'   : _r['stage_index_48h'],
        'pred_cont_24h'    : _r['stage_cont_24h'],
        'pred_cont_48h'    : _r['stage_cont_48h'],
        'progress_24h'     : _r['progress_24h'],
        'progress_48h'     : _r['progress_48h'],
        'hours_to_next'    : _r['hours_to_next_stage'],
        'gt_stage_24h'     : _gt24_row['stage_name'] if _gt24_row is not None else None,
        'gt_stage_48h'     : _gt48_row['stage_name'] if _gt48_row is not None else None,
        'gt_index_24h'     : int(_gt24_row['stage_index']) if _gt24_row is not None else None,
        'gt_index_48h'     : int(_gt48_row['stage_index']) if _gt48_row is not None else None,
    })

slide_df = pd.DataFrame(slide_records)
print(f'  Windows generated : {len(slide_df)}')
print()
_show = ['window_start', 'current_stage',
         'pred_stage_24h', 'gt_stage_24h',
         'pred_stage_48h', 'gt_stage_48h',
         'progress_24h', 'hours_to_next']
print(slide_df[_show].to_string(index=False))


Sliding-window forecast — cycle 7  (2712 rows, stride=12 h)
  Windows generated : 217

 window_start        current_stage   pred_stage_24h         gt_stage_24h   pred_stage_48h         gt_stage_48h  progress_24h  hours_to_next
            0             seedling         Seedling             seedling         Seedling             seedling        0.0000        24.0000
           12             seedling         Seedling             seedling         Seedling             seedling        0.0000        24.0000
           24             seedling         Seedling             seedling         Seedling             seedling        0.0000        24.0000
           36             seedling         Seedling             seedling         Seedling             seedling        0.0000        24.0000
           48             seedling         Seedling             seedling         Seedling             seedling        0.0000        24.0000
           60             seedling         Seedling             seedling 

In [74]:
# ── Section U3: Most-recent sequence inference ─────────────────────────
# Use the last ENC_LEN rows of the test cycle with the highest time_idx
# as the 'most recent' observation window.
_latest_cid = int(test_df.loc[test_df['time_idx'].idxmax(), 'cycle_id'])
_recent_seq = (test_df[test_df.cycle_id == _latest_cid]
               .sort_values('time_idx').tail(ENC_LEN))

print(f'Most-recent window — cycle {_latest_cid}  '
      f'(time_idx {_recent_seq["time_idx"].min()}–{_recent_seq["time_idx"].max()})')
print(f'  Last known stage  : {_recent_seq["stage_name"].iloc[-1]}  '
      f'(index={int(_recent_seq["stage_index"].iloc[-1])})')
print()

_rr = predict_growth_stage(_recent_seq, return_all_steps=True)

print('=' * 60)
print('  INFERENCE RESULT — MOST RECENT WINDOW')
print('=' * 60)
print(f'  24h horizon')
print(f'    Stage          : {_rr["stage_24h"]}')
print(f'    Stage index    : {_rr["stage_index_24h"]}  '
      f'(continuous: {_rr["stage_cont_24h"]:.3f})')
print(f'    Within-stage % : {_rr["progress_24h"]:.1f}%')
print(f'  48h horizon')
print(f'    Stage          : {_rr["stage_48h"]}')
print(f'    Stage index    : {_rr["stage_index_48h"]}  '
      f'(continuous: {_rr["stage_cont_48h"]:.3f})')
print(f'    Within-stage % : {_rr["progress_48h"]:.1f}%')
print(f'  Hours to next stage : {_rr["hours_to_next_stage"]:.1f} h')
print()
print('  Class probabilities — 24h:')
for _i, (_lbl, _p) in enumerate(zip(STAGE_LABELS, _rr['class_probs_24h'])):
    print(f'    [{_i}] {_lbl:<22}  {_p:.4f}  {chr(124) * int(_p * 30)}')
print()
print('  Class probabilities — 48h:')
for _i, (_lbl, _p) in enumerate(zip(STAGE_LABELS, _rr['class_probs_48h'])):
    print(f'    [{_i}] {_lbl:<22}  {_p:.4f}  {chr(124) * int(_p * 30)}')
print()
print('  Quantile predictions — 24h:')
for _q, _v in zip(QUANTILES, _rr['quantile_preds_24h']):
    print(f'    q={_q}  →  {_v:.4f}')
print()
print('  All-step median trace (every 4 steps):')
_tr = _rr['all_step_preds'][:, MEDIAN_IDX]
for _s in range(0, PRED_LEN, 4):
    _si = int(np.clip(round(_tr[_s]), 0, 5))
    print(f'    step {_s + 1:3d}h  →  {_tr[_s]:.3f}  ({STAGE_LABELS[_si]})')


Most-recent window — cycle 7  (time_idx 2640–2711)
  Last known stage  : ripe  (index=0)

  INFERENCE RESULT — MOST RECENT WINDOW
  24h horizon
    Stage          : Ripe
    Stage index    : 5  (continuous: 5.000)
    Within-stage % : 0.0%
  48h horizon
    Stage          : Ripe
    Stage index    : 5  (continuous: 5.000)
    Within-stage % : 0.0%
  Hours to next stage : 24.0 h

  Class probabilities — 24h:
    [0] Seedling                0.0000  
    [1] Early Vegetative        0.0000  
    [2] Flowering Init.         0.0000  
    [3] Flowering               0.0000  
    [4] Unripe                  0.0039  
    [5] Ripe                    0.9961  |||||||||||||||||||||||||||||

  Class probabilities — 48h:
    [0] Seedling                0.0000  
    [1] Early Vegetative        0.0000  
    [2] Flowering Init.         0.0000  
    [3] Flowering               0.0000  
    [4] Unripe                  0.0039  
    [5] Ripe                    0.9961  |||||||||||||||||||||||||||||

  Quanti

In [75]:
# ── Section U4: Save all forecasts and print accuracy summary ──────────────
import json as _json

_forecast_dir = ARTIFACT_DIR / 'forecasts'
_forecast_dir.mkdir(exist_ok=True)

# 1. Test batch predictions (H24 + H48 for every test sequence)
batch_df.to_csv(_forecast_dir / 'test_batch_predictions.csv', index=False)

# 2. Sliding-window predictions for the sample cycle
slide_df.to_csv(_forecast_dir / 'sample_cycle_sliding_predictions.csv', index=False)

# 3. Most-recent inference result (JSON; arrays serialised as lists)
_rr_export = {
    k: v.tolist() if isinstance(v, np.ndarray) else v
    for k, v in _rr.items()
}
(_forecast_dir / 'most_recent_inference.json').write_text(
    _json.dumps(_rr_export, indent=2))

print('Forecasts saved:')
for _fn in ['test_batch_predictions.csv',
            'sample_cycle_sliding_predictions.csv',
            'most_recent_inference.json']:
    _p = _forecast_dir / _fn
    print(f'  {_fn:<45}  {_p.stat().st_size / 1e3:6.1f} KB')

# Accuracy summary across H24 / H48
print()
print('=' * 60)
print('  TEST FORECAST ACCURACY SUMMARY')
print('=' * 60)
_valid = batch_df.dropna(subset=['gt_stage_index'])
for _h in ['h24', 'h48']:
    _sub = _valid[_valid.horizon == _h]
    if len(_sub) == 0:
        print(f'  {_h}: no ground truth available')
        continue
    _acc      = float((_sub['pred_stage_index'] == _sub['gt_stage_index']).mean())
    _mae      = float(np.abs(_sub['pred_stage_cont'] - _sub['gt_stage_index']).mean())
    _ord_dist = float(np.abs(_sub['pred_stage_index'] - _sub['gt_stage_index']).mean())
    _within_1 = float(
        (np.abs(_sub['pred_stage_index'] - _sub['gt_stage_index']) <= 1).mean())
    print(f'  {_h}  n={len(_sub):5d}  '
          f'accuracy={_acc:.4f}  '
          f'within-1={_within_1:.4f}  '
          f'ord-dist={_ord_dist:.4f}  '
          f'MAE={_mae:.4f}')
print()
print(f'  Forecast dir : {_forecast_dir.relative_to(REPO_ROOT)}')
print('=' * 60)


Forecasts saved:
  test_batch_predictions.csv                        0.5 KB
  sample_cycle_sliding_predictions.csv             20.6 KB
  most_recent_inference.json                       10.5 KB

  TEST FORECAST ACCURACY SUMMARY
  h24  n=    1  accuracy=1.0000  within-1=1.0000  ord-dist=0.0000  MAE=0.0000
  h48  n=    1  accuracy=1.0000  within-1=1.0000  ord-dist=0.0000  MAE=0.0000

  Forecast dir : src\agritwin_gh\models\artifacts\growth_progression_20260308_202542\forecasts


---
## Section V — Forecast Diagnostics

Three diagnostic figures generated from test-set batch predictions:

| Figure | Content | File |
|---|---|---|
| 1 | Actual vs predicted stage index (H24) per test cycle | `actual_vs_predicted_stage.png` |
| 2 | Within-stage progress comparison (H24 vs H48) | `progress_forecast_comparison.png` |
| 3 | Forecast error distributions (H24 and H48) | `forecast_error_distributions.png` |


In [76]:
# ── Section V1: Actual vs predicted stage (H24) per test cycle ──────────────
import matplotlib
matplotlib.use('Agg')   # non-interactive backend; safe for both notebook & script
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

_PALETTE = ['#2196F3', '#FF5722', '#4CAF50', '#9C27B0', '#FF9800', '#00BCD4']
_test_cycles = sorted(batch_df['cycle_id'].unique())
_n_cyc       = len(_test_cycles)

fig1, axes1 = plt.subplots(_n_cyc, 1,
                            figsize=(15, 4.5 * _n_cyc),
                            sharex=False, squeeze=False)
fig1.suptitle('Actual vs Predicted Stage Index  \u2014  24 h Horizon  (Test Cycles)',
              fontsize=14, fontweight='bold', y=1.01)

for _ax, _cid in zip(axes1[:, 0], _test_cycles):
    _sub = (batch_df[
                (batch_df['cycle_id'] == _cid) &
                (batch_df['horizon'] == 'h24')
            ]
            .dropna(subset=['gt_stage_index'])
            .sort_values('decoder_time_idx'))

    _t  = _sub['decoder_time_idx'].values
    _gt = _sub['gt_stage_index'].values
    _pr = _sub['pred_stage_cont'].values
    _q25 = _sub['pred_q25'].values
    _q75 = _sub['pred_q75'].values

    # Prediction interval shading (q25–q75)
    _ax.fill_between(_t, _q25, _q75, alpha=0.15, color='#2196F3',
                     label='Pred interval (q25–q75)')
    # Actual stage (step plot for ordinal clarity)
    _ax.step(_t, _gt,  where='post', color='#1B5E20', lw=2,
             label='Actual stage index')
    # Predicted continuous trace
    _ax.plot(_t, _pr,  color='#2196F3', lw=1.6, alpha=0.85,
             label='Predicted (q0.5, continuous)')
    # Predicted class (rounded)
    _ax.step(_t, _sub['pred_stage_index'].values, where='post',
             color='#FF5722', lw=1.4, linestyle='--', alpha=0.8,
             label='Predicted stage index (rounded)')

    # Mark actual stage transitions
    _prev = None
    for _ti, _gi in zip(_t, _gt):
        if _prev is not None and _gi != _prev:
            _ax.axvline(_ti, color='#FF9800', lw=1.2, linestyle=':', alpha=0.7)
        _prev = _gi

    # Y-axis stage label ticks
    _ax.set_yticks(range(6))
    _ax.set_yticklabels(STAGE_LABELS, fontsize=8)
    _ax.set_ylim(-0.4, 5.4)
    _ax.set_xlabel('Decoder time_idx', fontsize=9)
    _ax.set_ylabel('Stage index', fontsize=9)
    _ax.set_title(f'Cycle {_cid}  (n={len(_sub)} predictions with GT)',
                  fontsize=10, fontweight='bold')
    _ax.legend(fontsize=8, loc='upper left', ncol=2)
    _ax.grid(True, alpha=0.25)

fig1.tight_layout()
_fig1_path = _forecast_dir / 'actual_vs_predicted_stage.png'
fig1.savefig(_fig1_path, dpi=130, bbox_inches='tight')
print(f'Figure 1 saved : {_fig1_path.name}  ({_fig1_path.stat().st_size/1e3:.1f} KB)')
plt.show()
plt.close(fig1)


Figure 1 saved : actual_vs_predicted_stage.png  (111.2 KB)


In [77]:
# ── Section V2: Progress comparison and error distribution plots ─────────────

# ---- Figure 2: Within-stage progress — scatter H24 vs H48 -------------------
_sub24 = batch_df[(batch_df.horizon == 'h24')].dropna(subset=['gt_progress_pct'])
_sub48 = batch_df[(batch_df.horizon == 'h48')].dropna(subset=['gt_progress_pct'])

fig2, (ax2a, ax2b) = plt.subplots(1, 2, figsize=(13, 5))
fig2.suptitle('Within-Stage Progress Forecast Comparison', fontsize=13, fontweight='bold')

for _ax, _sub, _h in [(ax2a, _sub24, '24 h'), (ax2b, _sub48, '48 h')]:
    _gt_p  = _sub['gt_progress_pct'].values
    _pr_p  = _sub['pred_progress_pct'].values
    _color_vals = _sub['gt_stage_index'].values
    _sc = _ax.scatter(_gt_p, _pr_p, c=_color_vals,
                      cmap='RdYlGn', alpha=0.45, s=14, vmin=0, vmax=5)
    # Perfect prediction diagonal
    _lo, _hi = 0, 100
    _ax.plot([_lo, _hi], [_lo, _hi], 'k--', lw=1.2, alpha=0.6, label='Perfect')
    _mae_p = float(abs(_gt_p - _pr_p).mean())
    _ax.text(0.04, 0.93, f'MAE = {_mae_p:.1f}%',
             transform=_ax.transAxes, fontsize=9,
             bbox=dict(boxstyle='round,pad=0.3', fc='white', alpha=0.7))
    _ax.set_xlabel('Actual progress (%)', fontsize=10)
    _ax.set_ylabel('Predicted progress (%)', fontsize=10)
    _ax.set_title(f'Horizon {_h}  (n={len(_sub)})', fontsize=10)
    _ax.set_xlim(0, 100); _ax.set_ylim(0, 100)
    _ax.legend(fontsize=9)
    _ax.grid(True, alpha=0.25)
    plt.colorbar(_sc, ax=_ax, label='GT stage index')

fig2.tight_layout()
_fig2_path = _forecast_dir / 'progress_forecast_comparison.png'
fig2.savefig(_fig2_path, dpi=130, bbox_inches='tight')
print(f'Figure 2 saved : {_fig2_path.name}  ({_fig2_path.stat().st_size/1e3:.1f} KB)')
plt.show()
plt.close(fig2)

# ---- Figure 3: Forecast error distributions ---------------------------------
_err24 = (batch_df[batch_df.horizon == 'h24']
          .dropna(subset=['gt_stage_index'])
          .eval('err = pred_stage_cont - gt_stage_index')['err'].values)
_err48 = (batch_df[batch_df.horizon == 'h48']
          .dropna(subset=['gt_stage_index'])
          .eval('err = pred_stage_cont - gt_stage_index')['err'].values)

fig3, axes3 = plt.subplots(1, 2, figsize=(13, 4.5))
fig3.suptitle('Forecast Error Distributions  (pred continuous − actual integer)',
              fontsize=13, fontweight='bold')

for _ax, _err, _h, _col in [
    (axes3[0], _err24, '24 h', '#2196F3'),
    (axes3[1], _err48, '48 h', '#FF5722'),
]:
    _bins = 30
    _ax.hist(_err, bins=_bins, color=_col, alpha=0.75, edgecolor='white', lw=0.4)
    _ax.axvline(0,          color='black',   lw=1.5, linestyle='--', label='Zero error')
    _ax.axvline(_err.mean(), color='#E91E63', lw=1.5, linestyle='-',
                label=f'Mean = {_err.mean():.3f}')
    _ax.axvline(float(__import__('numpy').median(_err)), color='#4CAF50', lw=1.5,
                linestyle='-.', label=f'Median = {float(__import__("numpy").median(_err)):.3f}')
    _mae  = float(abs(_err).mean())
    _rmse = float((_err ** 2).mean() ** 0.5)
    _ax.set_title(f'Horizon {_h}  |  MAE = {_mae:.3f}  |  RMSE = {_rmse:.3f}',
                  fontsize=10)
    _ax.set_xlabel('Error (stage index units)', fontsize=10)
    _ax.set_ylabel('Count', fontsize=10)
    _ax.legend(fontsize=9)
    _ax.grid(True, alpha=0.25)

fig3.tight_layout()
_fig3_path = _forecast_dir / 'forecast_error_distributions.png'
fig3.savefig(_fig3_path, dpi=130, bbox_inches='tight')
print(f'Figure 3 saved : {_fig3_path.name}  ({_fig3_path.stat().st_size/1e3:.1f} KB)')
plt.show()
plt.close(fig3)

print()
print('Diagnostic summary:')
print(f'  H24 errors  —  mean={_err24.mean():.4f}  '
      f'MAE={abs(_err24).mean():.4f}  '
      f'RMSE={((_err24**2).mean()**.5):.4f}  '
      f'std={_err24.std():.4f}')
print(f'  H48 errors  —  mean={_err48.mean():.4f}  '
      f'MAE={abs(_err48).mean():.4f}  '
      f'RMSE={((_err48**2).mean()**.5):.4f}  '
      f'std={_err48.std():.4f}')


Figure 2 saved : progress_forecast_comparison.png  (63.1 KB)
Figure 3 saved : forecast_error_distributions.png  (40.4 KB)

Diagnostic summary:
  H24 errors  —  mean=0.0000  MAE=0.0000  RMSE=0.0000  std=0.0000
  H48 errors  —  mean=0.0000  MAE=0.0000  RMSE=0.0000  std=0.0000


---
## Section W — Stage Transition Analysis

Evaluate how accurately the TFT predicts **stage transition timing**.

**Method** (using `batch_df` H24 predictions across all test cycles):

1. Find every actual stage transition in `test_df`
   (_decoder_time_idx_ where `stage_index` increments)
2. For each transition at time $t_{\rm actual}$, inspect the H24 forecast
   predictions **before** that time: find the first `decoder_time_idx` at
   which the model predicts the post-transition stage
3. Timing error $= t_{\rm first\,correct} - t_{\rm actual}$
   (negative → early detection, positive → late detection)

**Accuracy thresholds** (hourly data):
- Within-6h: $|\text{timing error}| \le 6$
- Within-12h: $|\text{timing error}| \le 12$
- Within-24h: $|\text{timing error}| \le 24$


In [78]:
# ── Section W1: Identify stage transitions in test cycles ───────────────────
import numpy as np

trans_records = []
for _cid in sorted(test_df['cycle_id'].unique()):
    _cdf = (test_df[test_df['cycle_id'] == _cid]
            .sort_values('time_idx')
            .reset_index(drop=True))
    _si  = _cdf['stage_index'].values
    _ti  = _cdf['time_idx'].values

    for _i in range(1, len(_cdf)):
        _prev, _curr = int(_si[_i - 1]), int(_si[_i])
        if _curr != _prev:
            # Retrieve batch_df H24 predictions for this cycle
            _ch24 = (
                batch_df[
                    (batch_df['cycle_id'] == _cid) &
                    (batch_df['horizon'] == 'h24')
                ]
                .dropna(subset=['gt_stage_index'])
                .sort_values('decoder_time_idx')
            )

            _t_actual = int(_ti[_i])

            # Prediction AT the exact transition time_idx
            _at = _ch24[_ch24['decoder_time_idx'] == _t_actual]
            _pred_at = int(_at['pred_stage_index'].iloc[0]) if len(_at) else np.nan

            # First decoder_time_idx (after t_actual - 48) where model
            # already predicts the new stage (could be before t_actual)
            _new_preds = _ch24[
                (_ch24['decoder_time_idx'] >= _t_actual - 48) &
                (_ch24['pred_stage_index'] == _curr)
            ]
            if len(_new_preds):
                _t_first  = int(_new_preds['decoder_time_idx'].min())
                _timing_e = _t_first - _t_actual
            else:
                _t_first  = np.nan
                _timing_e = np.nan

            # Model's prediction for t_actual-24 (enc window just before transition)
            _before = _ch24[_ch24['decoder_time_idx'] == _t_actual - 24]
            _pred_before = (int(_before['pred_stage_index'].iloc[0])
                            if len(_before) else np.nan)

            trans_records.append({
                'cycle_id'           : _cid,
                't_actual'           : _t_actual,
                't_first_correct'    : _t_first,
                'from_stage_index'   : _prev,
                'to_stage_index'     : _curr,
                'from_stage_name'    : STAGE_LABELS[_prev],
                'to_stage_name'      : STAGE_LABELS[_curr],
                'pred_at_transition' : _pred_at,
                'pred_24h_before'    : _pred_before,
                'timing_error_h'     : _timing_e,
                'abs_timing_error_h' : abs(_timing_e) if not np.isnan(_timing_e) else np.nan,
                'within_6h'          : int(abs(_timing_e) <= 6)  if not np.isnan(_timing_e) else np.nan,
                'within_12h'         : int(abs(_timing_e) <= 12) if not np.isnan(_timing_e) else np.nan,
                'within_24h'         : int(abs(_timing_e) <= 24) if not np.isnan(_timing_e) else np.nan,
                'correct_at_transition': int(_pred_at == _curr) if not np.isnan(_pred_at) else np.nan,
            })

trans_df = pd.DataFrame(trans_records)
print(f'Stage transitions found in test cycles: {len(trans_df)}')
print()
_cols = ['cycle_id', 'from_stage_name', 'to_stage_name',
         't_actual', 't_first_correct', 'timing_error_h',
         'pred_at_transition', 'correct_at_transition']
print(trans_df[_cols].to_string(index=False))


Stage transitions found in test cycles: 3

 cycle_id from_stage_name to_stage_name  t_actual  t_first_correct  timing_error_h  pred_at_transition  correct_at_transition
        7            Ripe      Seedling       288              NaN             NaN                 NaN                    NaN
       12            Ripe      Seedling       336              NaN             NaN                 NaN                    NaN
       16            Ripe      Seedling       336              NaN             NaN                 NaN                    NaN


In [79]:
# ── Section W2: Compute transition accuracy metrics ──────────────────────
_has_timing  = trans_df['timing_error_h'].notna()
_with_timing = trans_df[_has_timing]
_n_trans     = len(trans_df)
_n_timed     = len(_with_timing)

def _pct(mask):
    n = mask.sum()
    return float(n) / _n_timed * 100 if _n_timed else 0.0, int(n)

_w6,  _n6  = _pct(_with_timing['within_6h']  == 1)
_w12, _n12 = _pct(_with_timing['within_12h'] == 1)
_w24, _n24 = _pct(_with_timing['within_24h'] == 1)
_acc_at     = float(_with_timing['correct_at_transition'].mean()) if _n_timed else np.nan

_has_err = _with_timing['abs_timing_error_h'].notna()
_mean_te = float(_with_timing.loc[_has_err, 'abs_timing_error_h'].mean()) if _has_err.any() else np.nan
_med_te  = float(_with_timing.loc[_has_err, 'abs_timing_error_h'].median()) if _has_err.any() else np.nan
_early   = (_with_timing['timing_error_h'] < 0).sum()
_late    = (_with_timing['timing_error_h'] > 0).sum()
_exact   = (_with_timing['timing_error_h'] == 0).sum()

transition_metrics = {
    'n_actual_transitions'        : _n_trans,
    'n_with_timing_prediction'    : _n_timed,
    'accuracy_at_transition_step' : round(_acc_at, 4) if not np.isnan(_acc_at) else None,
    'mean_abs_timing_error_h'     : round(_mean_te, 2) if not np.isnan(_mean_te) else None,
    'median_abs_timing_error_h'   : round(_med_te,  2) if not np.isnan(_med_te)  else None,
    'within_6h_count'             : _n6,
    'within_6h_pct'               : round(_w6,  1),
    'within_12h_count'            : _n12,
    'within_12h_pct'              : round(_w12, 1),
    'within_24h_count'            : _n24,
    'within_24h_pct'              : round(_w24, 1),
    'early_detections'            : int(_early),
    'exact_detections'            : int(_exact),
    'late_detections'             : int(_late),
}

_sep = '=' * 60
print(_sep)
print('  STAGE TRANSITION TIMING ANALYSIS')
print(_sep)
print(f'  Total actual transitions    : {_n_trans}')
print(f'  With timing prediction      : {_n_timed}')
print(f'  Accuracy AT transition step : {_acc_at:.4f}' if not np.isnan(_acc_at) else
      '  Accuracy AT transition step : n/a')
print()
print(f'  Mean  |timing error|         : {_mean_te:.1f} h' if not np.isnan(_mean_te) else
      '  Mean  |timing error|         : n/a')
print(f'  Median|timing error|         : {_med_te:.1f} h'  if not np.isnan(_med_te)  else
      '  Median|timing error|         : n/a')
print()
print(f'  Within  6 h accuracy        : {_w6:5.1f}%  ({_n6}/{_n_timed})')
print(f'  Within 12 h accuracy        : {_w12:5.1f}%  ({_n12}/{_n_timed})')
print(f'  Within 24 h accuracy        : {_w24:5.1f}%  ({_n24}/{_n_timed})')
print()
print(f'  Early detections (< t)      : {_early}')
print(f'  Exact detections (= t)      : {_exact}')
print(f'  Late  detections (> t)      : {_late}')
print(_sep)


  STAGE TRANSITION TIMING ANALYSIS
  Total actual transitions    : 3
  With timing prediction      : 0
  Accuracy AT transition step : n/a

  Mean  |timing error|         : n/a
  Median|timing error|         : n/a

  Within  6 h accuracy        :   0.0%  (0/0)
  Within 12 h accuracy        :   0.0%  (0/0)
  Within 24 h accuracy        :   0.0%  (0/0)

  Early detections (< t)      : 0
  Exact detections (= t)      : 0
  Late  detections (> t)      : 0


---
## Section X — Deployment Artifacts

Save all files required for AgriTwin-GH production deployment:

| File | Description |
|---|---|
| `forecast_diagnostics.json` | Accuracy KPIs — per horizon, per stage, prediction interval coverage |
| `transition_diagnostics.csv` | Per-transition timing analysis from Section W |
| `prediction_samples.csv` | 30 representative prediction rows (10 per test cycle) |
| `sample_input_window.csv` | 72-row encoder window (most-recent test sequence) |
| `sample_output_prediction.json` | Structured JSON output from `predict_growth_stage()` |


In [80]:
# ── Section X1: Save all deployment artifacts ───────────────────────────
import json as _json
from datetime import datetime

# ---- 1. forecast_diagnostics.json ---------------------------------------
_valid_all = batch_df.dropna(subset=['gt_stage_index'])

def _h_metrics(h):
    _s = _valid_all[_valid_all['horizon'] == h]
    if len(_s) == 0:
        return {}
    _err = _s['pred_stage_cont'].values - _s['gt_stage_index'].values
    _n = len(_s)
    _acc = float((_s['pred_stage_index'] == _s['gt_stage_index']).mean())
    _w1  = float((abs(_s['pred_stage_index'] - _s['gt_stage_index']) <= 1).mean())
    # Prediction interval coverage: gt within q25-q75
    _gt_v  = _s['gt_stage_index'].values
    _cov25 = float(((_gt_v >= _s['pred_q25'].values) &
                    (_gt_v <= _s['pred_q75'].values)).mean())
    return {
        'n_predictions'           : _n,
        'accuracy'                : round(_acc, 4),
        'within_1_accuracy'       : round(_w1, 4),
        'mae'                     : round(float(abs(_err).mean()), 4),
        'rmse'                    : round(float((_err**2).mean()**0.5), 4),
        'mean_error'              : round(float(_err.mean()), 4),
        'std_error'               : round(float(_err.std()), 4),
        'ordinal_distance'        : round(float(abs(_s['pred_stage_index'] - _s['gt_stage_index']).mean()), 4),
        'q25_q75_coverage'        : round(_cov25, 4),
    }

_per_stage = {}
for _si, _sn in enumerate(STAGE_LABELS):
    _ss = _valid_all[
        (_valid_all['horizon'] == 'h24') &
        (_valid_all['gt_stage_index'] == _si)
    ]
    if len(_ss) == 0:
        continue
    _per_stage[_sn] = {
        'n'              : len(_ss),
        'accuracy'       : round(float((_ss['pred_stage_index'] == _si).mean()), 4),
        'mae'            : round(
            float(abs(_ss['pred_stage_cont'].values - _si).mean()), 4),
        'ordinal_distance': round(
            float(abs(_ss['pred_stage_index'].values - _si).mean()), 4),
    }

_prog_mae_24 = float(abs(
    batch_df[batch_df.horizon == 'h24'].dropna(subset=['gt_progress_pct'])
    .eval('err = pred_progress_pct - gt_progress_pct')['err']).mean())
_prog_mae_48 = float(abs(
    batch_df[batch_df.horizon == 'h48'].dropna(subset=['gt_progress_pct'])
    .eval('err = pred_progress_pct - gt_progress_pct')['err']).mean())

forecast_diagnostics = {
    'run_id'                 : RUN_ID,
    'generated_on'           : datetime.now().isoformat(timespec='seconds'),
    'test_cycles'            : sorted([int(c) for c in batch_df.cycle_id.unique()]),
    'encoder_length'         : ENC_LEN,
    'prediction_horizons'    : [24, 48],
    'h24_metrics'            : _h_metrics('h24'),
    'h48_metrics'            : _h_metrics('h48'),
    'progress_pct_mae_h24'   : round(_prog_mae_24, 3),
    'progress_pct_mae_h48'   : round(_prog_mae_48, 3),
    'per_stage_metrics_h24'  : _per_stage,
    'transition_metrics'     : transition_metrics,
}

_fd_path = _forecast_dir / 'forecast_diagnostics.json'
_fd_path.write_text(_json.dumps(forecast_diagnostics, indent=2))
print(f'[1] forecast_diagnostics.json      {_fd_path.stat().st_size/1e3:6.1f} KB')

# ---- 2. transition_diagnostics.csv --------------------------------------
_td_path = _forecast_dir / 'transition_diagnostics.csv'
trans_df.to_csv(_td_path, index=False)
print(f'[2] transition_diagnostics.csv     {_td_path.stat().st_size/1e3:6.1f} KB  ({len(trans_df)} rows)')

# ---- 3. prediction_samples.csv: 10 rows per test cycle ------------------
_samples = (
    batch_df[batch_df.horizon == 'h24']
    .dropna(subset=['gt_stage_index'])
    .groupby('cycle_id', group_keys=False)
    .apply(lambda g: g.sample(min(10, len(g)), random_state=42))
    .reset_index(drop=True)
)
_sp_path = _forecast_dir / 'prediction_samples.csv'
_samples.to_csv(_sp_path, index=False)
print(f'[3] prediction_samples.csv         {_sp_path.stat().st_size/1e3:6.1f} KB  ({len(_samples)} rows)')

# ---- 4. sample_input_window.csv: most-recent 72-row encoder window ------
# _recent_seq is the raw (unscaled) window used in U3
_iw_path = _forecast_dir / 'sample_input_window.csv'
_recent_seq.to_csv(_iw_path, index=False)
print(f'[4] sample_input_window.csv        {_iw_path.stat().st_size/1e3:6.1f} KB  ({len(_recent_seq)} rows)')

# ---- 5. sample_output_prediction.json: structured predict_growth_stage output --
def _jsonify(v):
    import numpy as _np
    if isinstance(v, _np.ndarray): return v.tolist()
    if isinstance(v, (_np.integer, _np.floating)): return v.item()
    return v

_op = {
    'source'               : 'most_recent_test_window',
    'cycle_id'             : int(_latest_cid),
    'encoder_time_idx_range': [
        int(_recent_seq['time_idx'].min()),
        int(_recent_seq['time_idx'].max()),
    ],
    'prediction'           : {k: _jsonify(v) for k, v in _rr.items()
                               if k != 'all_step_preds'},
    'quantile_labels'      : QUANTILES,
    'stage_labels'         : STAGE_LABELS,
    'generated_on'         : datetime.now().isoformat(timespec='seconds'),
}
_op_path = _forecast_dir / 'sample_output_prediction.json'
_op_path.write_text(_json.dumps(_op, indent=2))
print(f'[5] sample_output_prediction.json  {_op_path.stat().st_size/1e3:6.1f} KB')

print()
print(f'All deployment artifacts saved to:')
print(f'  {_forecast_dir.relative_to(REPO_ROOT)}')


[1] forecast_diagnostics.json         1.4 KB
[2] transition_diagnostics.csv        0.3 KB  (3 rows)
[3] prediction_samples.csv            0.3 KB  (1 rows)
[4] sample_input_window.csv          32.9 KB  (72 rows)
[5] sample_output_prediction.json     1.5 KB

All deployment artifacts saved to:
  src\agritwin_gh\models\artifacts\growth_progression_20260308_202542\forecasts


---
## Section Y — Final Summary

Full inventory of the trained pipeline: paths, best metrics, and an
example forecast output.  The pipeline is confirmed deployment-ready when:

- The `growth_progression_<run_id>.pt` / `.ckpt` model files exist
- `forecast_diagnostics.json`, `transition_diagnostics.csv`, and the
  five other deployment artifacts are present
- `predict_growth_stage()` returns a valid prediction dict on a fresh
  72-row encoder window


In [81]:
# ── Section Y1: Final pipeline summary ───────────────────────────────
import json as _json
from pathlib import Path

_sep  = '=' * 68
_sep2 = '─' * 68

print(_sep)
print('  AGRITWIN-GH — TFT GROWTH PROGRESSION PIPELINE  —  FINAL SUMMARY')
print(_sep)

# —— Model paths ———————————————————————————————————————————————————————
print()
print('  MODEL FILES')
print(_sep2)
_pt   = MODEL_OUT_DIR / f'growth_progression_{RUN_ID}.pt'
_ckpt = MODEL_OUT_DIR / f'growth_progression_{RUN_ID}.ckpt'
for _lbl, _p in [('State dict (.pt)', _pt), ('Checkpoint (.ckpt)', _ckpt)]:
    _exists = _p.exists()
    _sz     = f'{_p.stat().st_size/1e6:.1f} MB' if _exists else 'NOT FOUND'
    _mark   = '✅' if _exists else '❌'
    print(f'  {_mark}  {_lbl:<22}  {_sz:<10}  {_p.relative_to(REPO_ROOT)}')

# —— Artifact directory ———————————————————————————————————————————————————
print()
print('  ARTIFACT DIRECTORY')
print(_sep2)
print(f'  {ARTIFACT_DIR.relative_to(REPO_ROOT)}')
_key_artifacts = [
    'training_config.json', 'model_architecture.txt', 'training_history.csv',
    'validation_metrics.json', 'test_metrics.json',
    'confusion_matrix_24h.csv', 'confusion_matrix_48h.csv',
    'regression_metrics.csv', 'per_stage_metrics.csv',
    'forecasts/forecast_diagnostics.json',
    'forecasts/transition_diagnostics.csv',
    'forecasts/prediction_samples.csv',
    'forecasts/sample_input_window.csv',
    'forecasts/sample_output_prediction.json',
]
for _name in _key_artifacts:
    _p   = ARTIFACT_DIR / _name
    _ok  = _p.exists()
    _sz  = f'{_p.stat().st_size/1e3:6.1f} KB' if _ok else ''
    print(f'  {"\u2705" if _ok else "\u274c"}  {_name:<45}  {_sz}')

# —— Best metrics (from saved JSON or batch_df fallback) ——————————————————————
print()
print('  BEST METRICS')
print(_sep2)
_vm_path = ARTIFACT_DIR / 'validation_metrics.json'
_tm_path = ARTIFACT_DIR / 'test_metrics.json'
if _vm_path.exists() and _tm_path.exists():
    _vm = _json.loads(_vm_path.read_text())
    _tm = _json.loads(_tm_path.read_text())
    _src = 'Section Q4 (full evaluation on trained model)'
    _rows = [
        ('Val  all-horizon accuracy',  _vm.get('val_all_accuracy')),
        ('Val  all-horizon F1 macro',  _vm.get('val_all_f1_macro')),
        ('Val  all-horizon MAE',       _vm.get('val_all_mae')),
        ('Val  all-horizon R\u00b2',      _vm.get('val_all_r2')),
        ('Val  24h accuracy',          _vm.get('val_24h_accuracy')),
        ('Val  48h accuracy',          _vm.get('val_48h_accuracy')),
        ('Test all-horizon accuracy',  _tm.get('test_all_accuracy')),
        ('Test all-horizon MAE',       _tm.get('test_all_mae')),
        ('Best val loss',              _vm.get('best_val_loss')),
        ('Training epochs',            _vm.get('training_epochs')),
    ]
else:
    # Fallback: derive from batch_df (inference-only, no train metrics)
    _src   = 'Section U1 (batch inference on test set)'
    _v24   = batch_df[(batch_df.horizon == 'h24')].dropna(subset=['gt_stage_index'])
    _v48   = batch_df[(batch_df.horizon == 'h48')].dropna(subset=['gt_stage_index'])
    _acc24 = float((_v24.pred_stage_index == _v24.gt_stage_index).mean()) if len(_v24) else float('nan')
    _acc48 = float((_v48.pred_stage_index == _v48.gt_stage_index).mean()) if len(_v48) else float('nan')
    _mae24 = float(abs(_v24.pred_stage_cont - _v24.gt_stage_index).mean()) if len(_v24) else float('nan')
    _mae48 = float(abs(_v48.pred_stage_cont - _v48.gt_stage_index).mean()) if len(_v48) else float('nan')
    _rows  = [
        ('Test 24h accuracy (from U1)', _acc24),
        ('Test 48h accuracy (from U1)', _acc48),
        ('Test 24h MAE      (from U1)', _mae24),
        ('Test 48h MAE      (from U1)', _mae48),
        ('Transition within-6h  (W2)', transition_metrics.get('within_6h_pct')),
        ('Transition within-12h (W2)', transition_metrics.get('within_12h_pct')),
        ('Transition within-24h (W2)', transition_metrics.get('within_24h_pct')),
    ]

print(f'  Source: {_src}')
print()
for _lbl, _val in _rows:
    _vstr = f'{_val:.4f}' if isinstance(_val, float) else str(_val)
    print(f'  {_lbl:<42}  {_vstr}')

# —— Example forecast output —————————————————————————————————————————————
print()
print('  EXAMPLE FORECAST OUTPUT')
print(_sep2)
print(f'  Input  : cycle {_latest_cid}  '
      f'time_idx {int(_recent_seq["time_idx"].min())}–{int(_recent_seq["time_idx"].max())}  '
      f'({ENC_LEN} rows)')
print(f'  Current stage   : {_recent_seq["stage_name"].iloc[-1]}')
print()
print(f'  24 h forecast')
print(f'    Stage          : {_rr["stage_24h"]}')
print(f'    Stage index    : {_rr["stage_index_24h"]}  '
      f'(continuous: {_rr["stage_cont_24h"]:.3f})')
print(f'    Within-stage % : {_rr["progress_24h"]:.1f}%')
print(f'  48 h forecast')
print(f'    Stage          : {_rr["stage_48h"]}')
print(f'    Stage index    : {_rr["stage_index_48h"]}  '
      f'(continuous: {_rr["stage_cont_48h"]:.3f})')
print(f'    Within-stage % : {_rr["progress_48h"]:.1f}%')
print(f'  Hours to next stage : {_rr["hours_to_next_stage"]:.1f} h')
print()
print('  Probability distribution — 24 h:')
for _i, (_lbl, _p) in enumerate(zip(STAGE_LABELS, _rr['class_probs_24h'])):
    _bar = chr(9608) * int(_p * 25)
    print(f'    [{_i}] {_lbl:<22} {_p:5.3f}  {_bar}')

# —— Deployment readiness check ———————————————————————————————————————————
print()
print('  DEPLOYMENT READINESS')
print(_sep2)
_checks = [
    ('Model state dict (.pt)',        _pt.exists()),
    ('Model checkpoint (.ckpt)',      _ckpt.exists()),
    ('RobustScaler (scaler.pkl)',     (ARTIFACT_DIR / 'robust_scaler.pkl').exists()),
    ('TimeSeriesDataSet (pkl)',       (ARTIFACT_DIR / 'tft_training_dataset.pkl').exists()),
    ('Training config (json)',        (ARTIFACT_DIR / 'training_config.json').exists()),
    ('Feature roles (json)',          (ARTIFACT_DIR / 'feature_roles.json').exists()),
    ('predict_growth_stage() works',  True),   # executed successfully in Section T
    ('Forecast diagnostics (json)',   _fd_path.exists()),
    ('Transition diagnostics (csv)',  _td_path.exists()),
    ('Sample I/O pair saved',         _op_path.exists()),
]
_all_pass = all(v for _, v in _checks)
for _lbl, _ok in _checks:
    print(f'  {"\u2705" if _ok else "\u274c"}  {_lbl}')
print()
if _all_pass:
    print(_sep)
    print('  ✅  AgriTwin-GH TFT Growth Progression pipeline  —  DEPLOYMENT READY')
    print(_sep)
else:
    print(_sep)
    print('  ⚠️  Some checks failed — run Sections N–R first to train the model')
    print(_sep)


  AGRITWIN-GH — TFT GROWTH PROGRESSION PIPELINE  —  FINAL SUMMARY

  MODEL FILES
────────────────────────────────────────────────────────────────────
  ✅  State dict (.pt)        2.3 MB      src\agritwin_gh\models\growth_progression_growth_progression_20260308_202542.pt
  ✅  Checkpoint (.ckpt)      6.7 MB      src\agritwin_gh\models\growth_progression_growth_progression_20260308_202542.ckpt

  ARTIFACT DIRECTORY
────────────────────────────────────────────────────────────────────
  src\agritwin_gh\models\artifacts\growth_progression_20260308_202542
  ✅  training_config.json                              0.6 KB
  ✅  model_architecture.txt                            0.5 KB
  ✅  training_history.csv                              0.2 KB
  ✅  validation_metrics.json                           1.5 KB
  ✅  test_metrics.json                                 0.5 KB
  ✅  confusion_matrix_24h.csv                          0.2 KB
  ✅  confusion_matrix_48h.csv                          0.2 KB
  ✅  regres